In [28]:
import pandas as pd
import plotly.express as px
import plotly.io as pio

pio.renderers.default = "notebook_connected"   # change to "browser" if needed

FILE_PATH = "SCB_India.xlsx"

In [29]:
def read_series(sheet_name, date_col, value_col):
    df = pd.read_excel(FILE_PATH, sheet_name=sheet_name)
    
    df[date_col] = pd.to_datetime(df[date_col], errors="coerce")
    df[value_col] = pd.to_numeric(df[value_col], errors="coerce")
    
    df = (
        df.dropna(subset=[date_col, value_col])
          .sort_values(date_col)
          .drop_duplicates(subset=[date_col], keep="last")
          .reset_index(drop=True)
    )
    
    return df

gross_npa   = read_series("Gross_NPA", "Quarter", "Gross NPA")
cdr         = read_series("CDR", "Quarter", "CDR(%)")
medium      = read_series("Medium", "Month", "Credit_gross_medium")
small_micro = read_series("Small_micro", "Month", "Gross_SM_Credit")
repo        = read_series("Repo", "Month", "Repo_rate")
crr         = read_series("CRR", "Month", "CRR")
bank_rate   = read_series("Bank_rate", "Month", "Bank_rate")
pcr         = read_series("PCR", "Quarter", "Provision_coverage")
walr        = read_series("WALR", "Quarter", "WALR")
nnpa        = read_series("NNPA", "Year", "NNPA")
gnpa_ratio  = read_series("GNPA_Ratio", "Quarter", "GNPR")

## Plots

In [30]:
def plot_line(df, x_col, y_col, title, y_title, hover_format=".2f"):
    fig = px.line(
        df,
        x=x_col,
        y=y_col,
        title=title,
        markers=True
    )
    
    fig.update_traces(
        line=dict(width=2),
        marker=dict(size=4),
        hovertemplate=(
            "<b>%{x|%b %Y}</b><br>"
            + y_title + ": %{y:" + hover_format + "}<extra></extra>"
        )
    )
    
    fig.update_layout(
        template="plotly_white",
        width=1150,
        height=520,
        title=dict(x=0.02, font=dict(size=20)),
        margin=dict(l=70, r=40, t=80, b=65),
        hovermode="x unified",
        showlegend=False
    )
    
    fig.update_xaxes(
        title="Period",
        showgrid=True,
        gridcolor="rgba(180,180,180,0.25)"
    )
    
    fig.update_yaxes(
        title=y_title,
        showgrid=True,
        gridcolor="rgba(180,180,180,0.25)",
        zeroline=False
    )
    
    fig.show()

In [31]:
plot_line(
    gross_npa,
    "Quarter",
    "Gross NPA",
    "Scheduled Commercial Banks: Gross Non-Performing Assets",
    "Gross NPA (INR million)",
    ",.0f"
)

In [32]:
plot_line(
    gnpa_ratio,
    "Quarter",
    "GNPR",
    "Scheduled Commercial Banks: Gross NPA Ratio",
    "GNPA Ratio (%)",
    ".2f"
)

In [33]:
plot_line(
    cdr,
    "Quarter",
    "CDR(%)",
    "Scheduled Commercial Banks: Credit-Deposit Ratio",
    "Credit-Deposit Ratio (%)",
    ".2f"
)

In [34]:
plot_line(
    small_micro,
    "Month",
    "Gross_SM_Credit",
    "Scheduled Commercial Banks: Credit Outstanding to Small and Micro Enterprises",
    "Credit Outstanding (INR million)",
    ",.0f"
)

In [35]:
small_micro_growth = small_micro.copy()

small_micro_growth["SME_Credit_Growth_YoY"] = (
    small_micro_growth["Gross_SM_Credit"]
    .pct_change(12)
    .mul(100)
)

small_micro_growth = small_micro_growth.dropna(
    subset=["SME_Credit_Growth_YoY"]
)

plot_line(
    small_micro_growth,
    "Month",
    "SME_Credit_Growth_YoY",
    "Small and Micro Enterprise Credit: Year-on-Year Growth",
    "YoY Growth (%)",
    ".2f"
)

In [36]:
plot_line(
    medium,
    "Month",
    "Credit_gross_medium",
    "Scheduled Commercial Banks: Credit Outstanding to Medium Enterprises",
    "Credit Outstanding (INR million)",
    ",.0f"
)

In [37]:
medium_growth = medium.copy()

medium_growth["Medium_Credit_Growth_YoY"] = (
    medium_growth["Credit_gross_medium"]
    .pct_change(12)
    .mul(100)
)

medium_growth = medium_growth.dropna(
    subset=["Medium_Credit_Growth_YoY"]
)

plot_line(
    medium_growth,
    "Month",
    "Medium_Credit_Growth_YoY",
    "Medium Enterprise Credit: Year-on-Year Growth",
    "YoY Growth (%)",
    ".2f"
)

In [38]:
plot_line(
    repo,
    "Month",
    "Repo_rate",
    "Reserve Bank of India: Repo Rate",
    "Repo Rate (%)",
    ".2f"
)

In [39]:
plot_line(
    crr,
    "Month",
    "CRR",
    "Reserve Bank of India: Cash Reserve Ratio",
    "CRR (%)",
    ".2f"
)

In [40]:
plot_line(
    bank_rate,
    "Month",
    "Bank_rate",
    "Reserve Bank of India: Bank Rate",
    "Bank Rate (%)",
    ".2f"
)

In [41]:
plot_line(
    pcr,
    "Quarter",
    "Provision_coverage",
    "Deposit Takers: Provision Coverage Ratio",
    "Provision Coverage Ratio (%)",
    ".2f"
)

In [42]:
plot_line(
    walr,
    "Quarter",
    "WALR",
    "Scheduled Commercial Banks: Weighted Average Lending Rate",
    "WALR (%)",
    ".2f"
)

In [43]:
plot_line(
    nnpa,
    "Year",
    "NNPA",
    "Scheduled Commercial Banks: Net Non-Performing Assets",
    "NNPA (INR million)",
    ",.0f"
)

# Correlations

In [44]:
# ── Monthly → quarter-end conversion ─────────────────────────────────────────

def monthly_to_quarter_end(df, date_col, value_col):
    temp = df.copy()
    temp[date_col] = pd.to_datetime(temp[date_col])
    
    temp = (
        temp.set_index(date_col)[value_col]
        .resample("QE")
        .last()
        .reset_index()
    )
    
    return temp

sm_credit_q = monthly_to_quarter_end(
    small_micro,
    "Month",
    "Gross_SM_Credit"
)

repo_q = monthly_to_quarter_end(
    repo,
    "Month",
    "Repo_rate"
)

crr_q = monthly_to_quarter_end(
    crr,
    "Month",
    "CRR"
)

bank_rate_q = monthly_to_quarter_end(
    bank_rate,
    "Month",
    "Bank_rate"
)

In [45]:
# ── Quarter-on-quarter annualised comparison: YoY quarterly credit growth ────

sm_credit_q["MSE_Credit_Growth_YoY"] = (
    sm_credit_q["Gross_SM_Credit"]
    .pct_change(4)
    .mul(100)
)

sm_credit_q = sm_credit_q.dropna(
    subset=["MSE_Credit_Growth_YoY"]
)

sm_credit_q.head()

,Month,Gross_SM_Credit,MSE_Credit_Growth_YoY
4,2006-06-30,904550.0,23.511661
5,2006-09-30,949340.0,21.980804
6,2006-12-31,1005990.0,23.016252
7,2007-03-31,1169080.0,28.442101
8,2007-06-30,2107980.0,133.041844


In [46]:
# ── Make all dates quarter-end dates ─────────────────────────────────────────

def to_quarter_end(df, date_col):
    temp = df.copy()
    temp[date_col] = pd.to_datetime(temp[date_col])
    temp["Quarter_End"] = temp[date_col].dt.to_period("Q").dt.end_time.dt.normalize()
    return temp

cdr_q        = to_quarter_end(cdr, "Quarter")
gnpa_q       = to_quarter_end(gnpa_ratio, "Quarter")
gross_npa_q  = to_quarter_end(gross_npa, "Quarter")
pcr_q        = to_quarter_end(pcr, "Quarter")
walr_q       = to_quarter_end(walr, "Quarter")

sm_credit_q["Quarter_End"] = (
    sm_credit_q["Month"]
    .dt.to_period("Q")
    .dt.end_time
    .dt.normalize()
)

repo_q["Quarter_End"] = (
    repo_q["Month"]
    .dt.to_period("Q")
    .dt.end_time
    .dt.normalize()
)

crr_q["Quarter_End"] = (
    crr_q["Month"]
    .dt.to_period("Q")
    .dt.end_time
    .dt.normalize()
)

bank_rate_q["Quarter_End"] = (
    bank_rate_q["Month"]
    .dt.to_period("Q")
    .dt.end_time
    .dt.normalize()
)

In [47]:
# ── Quarterly banking + SME credit dataset ───────────────────────────────────

quarterly_df = (
    sm_credit_q[
        ["Quarter_End", "Gross_SM_Credit", "MSE_Credit_Growth_YoY"]
    ]
    .merge(
        cdr_q[["Quarter_End", "CDR(%)"]],
        on="Quarter_End",
        how="inner"
    )
    .merge(
        gnpa_q[["Quarter_End", "GNPR"]],
        on="Quarter_End",
        how="inner"
    )
    .merge(
        pcr_q[["Quarter_End", "Provision_coverage"]],
        on="Quarter_End",
        how="left"
    )
    .merge(
        walr_q[["Quarter_End", "WALR"]],
        on="Quarter_End",
        how="inner"
    )
    .merge(
        repo_q[["Quarter_End", "Repo_rate"]],
        on="Quarter_End",
        how="inner"
    )
    .merge(
        crr_q[["Quarter_End", "CRR"]],
        on="Quarter_End",
        how="inner"
    )
    .merge(
        bank_rate_q[["Quarter_End", "Bank_rate"]],
        on="Quarter_End",
        how="inner"
    )
    .sort_values("Quarter_End")
    .reset_index(drop=True)
)

quarterly_df.head()

,Quarter_End,Gross_SM_Credit,MSE_Credit_Growth_YoY,CDR(%),GNPR,Provision_coverage,WALR,Repo_rate,CRR,Bank_rate
0,2013-03-31,5622960.0,12.769316,77.93,3.422945,46.088216,11.46,7.50,4.0,8.50
1,2013-06-30,6051297.6,22.630200,76.42,4.004487,47.716288,11.40,7.25,4.0,8.25
2,2013-09-30,6041250.0,21.847790,78.35,4.218281,47.301251,11.96,7.50,4.0,9.50
3,2013-12-31,6546330.0,22.894464,76.81,4.397677,47.353143,11.70,7.75,4.0,8.75
4,2014-03-31,7078130.0,25.879074,77.79,4.115068,54.563439,11.57,8.00,4.0,9.00


In [48]:
quarterly_df = quarterly_df.rename(
    columns={
        "CDR(%)": "Credit_Deposit_Ratio",
        "GNPR": "GNPA_Ratio",
        "Provision_coverage": "PCR",
        "WALR": "WALR",
        "Repo_rate": "Repo_Rate",
        "CRR": "Cash_Reserve_Ratio",
        "Bank_rate": "Bank_Rate"
    }
)

quarterly_df.tail()

,Quarter_End,Gross_SM_Credit,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,PCR,WALR,Repo_Rate,Cash_Reserve_Ratio,Bank_Rate
47,2024-12-31,2.145252e+07,12.066691,80.42,2.512434,77.892479,9.25,6.50,4.00,6.75
48,2025-03-31,2.239409e+07,13.434789,80.79,2.311152,76.595614,9.35,6.25,4.00,6.50
49,2025-06-30,2.460012e+07,21.754197,78.91,2.324439,76.461608,8.62,5.50,4.00,5.75
50,2025-09-30,2.520121e+07,22.462731,80.28,2.152929,75.917115,8.39,5.50,3.75,5.75
51,2025-12-31,2.755699e+07,28.455728,81.75,1.991088,73.203094,8.28,5.25,3.00,5.50


In [49]:
coverage_check = pd.DataFrame({
    "First Available": quarterly_df.dropna().min(numeric_only=False),
    "Last Available": quarterly_df.dropna().max(numeric_only=False),
    "Missing Values": quarterly_df.isna().sum(),
    "Non-Missing Observations": quarterly_df.notna().sum()
})

coverage_check

,First Available,Last Available,Missing Values,Non-Missing Observations
Quarter_End,2013-03-31 00:00:00,2025-12-31 00:00:00,0,52
Gross_SM_Credit,5622960.0,27556990.0,0,52
MSE_Credit_Growth_YoY,-1.070759,28.455728,0,52
Credit_Deposit_Ratio,69.54,81.75,0,52
GNPA_Ratio,1.991088,11.460877,0,52
PCR,40.87502,77.892479,0,52
WALR,7.63,11.96,0,52
Repo_Rate,4.0,8.0,0,52
Cash_Reserve_Ratio,3.0,4.5,0,52
Bank_Rate,4.25,9.5,0,52


In [50]:
# Keep only the variables intended for the baseline correlation analysis.

corr_vars = [
    "MSE_Credit_Growth_YoY",
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "PCR",
    "WALR",
    "Repo_Rate",
    "Cash_Reserve_Ratio"
]

corr_df = quarterly_df[
    ["Quarter_End"] + corr_vars
].dropna()

print("Correlation sample starts:", corr_df["Quarter_End"].min().date())
print("Correlation sample ends  :", corr_df["Quarter_End"].max().date())
print("Number of quarterly observations:", len(corr_df))

corr_df.head()

Correlation sample starts: 2013-03-31
Correlation sample ends  : 2025-12-31
Number of quarterly observations: 52


,Quarter_End,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,PCR,WALR,Repo_Rate,Cash_Reserve_Ratio
0,2013-03-31,12.769316,77.93,3.422945,46.088216,11.46,7.50,4.0
1,2013-06-30,22.630200,76.42,4.004487,47.716288,11.40,7.25,4.0
2,2013-09-30,21.847790,78.35,4.218281,47.301251,11.96,7.50,4.0
3,2013-12-31,22.894464,76.81,4.397677,47.353143,11.70,7.75,4.0
4,2014-03-31,25.879074,77.79,4.115068,54.563439,11.57,8.00,4.0


In [51]:
pearson_corr = corr_df[corr_vars].corr(method="pearson").round(3)

pearson_corr

,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,PCR,WALR,Repo_Rate,Cash_Reserve_Ratio
MSE_Credit_Growth_YoY,1.000,0.511,-0.695,0.405,0.141,0.267,0.164
Credit_Deposit_Ratio,0.511,1.000,-0.545,0.247,0.319,0.487,0.179
GNPA_Ratio,-0.695,-0.545,1.000,-0.546,-0.110,-0.273,-0.234
PCR,0.405,0.247,-0.546,1.000,-0.634,-0.407,0.100
WALR,0.141,0.319,-0.110,-0.634,1.000,0.912,0.231
Repo_Rate,0.267,0.487,-0.273,-0.407,0.912,1.000,0.480
Cash_Reserve_Ratio,0.164,0.179,-0.234,0.100,0.231,0.480,1.000


In [52]:
spearman_corr = corr_df[corr_vars].corr(method="spearman").round(3)

spearman_corr

,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,PCR,WALR,Repo_Rate,Cash_Reserve_Ratio
MSE_Credit_Growth_YoY,1.000,0.493,-0.727,0.492,0.022,0.277,0.243
Credit_Deposit_Ratio,0.493,1.000,-0.600,0.306,0.313,0.475,0.163
GNPA_Ratio,-0.727,-0.600,1.000,-0.609,0.049,-0.309,-0.280
PCR,0.492,0.306,-0.609,1.000,-0.610,-0.283,0.253
WALR,0.022,0.313,0.049,-0.610,1.000,0.874,0.071
Repo_Rate,0.277,0.475,-0.309,-0.283,0.874,1.000,0.339
Cash_Reserve_Ratio,0.243,0.163,-0.280,0.253,0.071,0.339,1.000


In [53]:
fig = px.imshow(
    pearson_corr,
    text_auto=".2f",
    aspect="auto",
    color_continuous_scale="RdBu_r",
    zmin=-1,
    zmax=1,
    title="Pearson Correlation Matrix: SME Credit Growth and Banking Variables"
)

fig.update_layout(
    template="plotly_white",
    width=1050,
    height=800,
    title=dict(x=0.02, font=dict(size=20)),
    margin=dict(l=120, r=50, t=90, b=120),
    coloraxis_colorbar=dict(title="Correlation")
)

fig.update_xaxes(tickangle=-35)
fig.show()

In [54]:
from scipy.stats import pearsonr
import numpy as np

pearson_r_matrix = pd.DataFrame(
    np.nan,
    index=corr_vars,
    columns=corr_vars
)

pearson_p_matrix = pd.DataFrame(
    np.nan,
    index=corr_vars,
    columns=corr_vars
)

for var_1 in corr_vars:
    for var_2 in corr_vars:
        
        temp = corr_df[[var_1, var_2]].dropna()
        
        if var_1 == var_2:
            pearson_r_matrix.loc[var_1, var_2] = 1.0
            pearson_p_matrix.loc[var_1, var_2] = 0.0
        
        else:
            r, p = pearsonr(temp[var_1], temp[var_2])
            
            pearson_r_matrix.loc[var_1, var_2] = r
            pearson_p_matrix.loc[var_1, var_2] = p

pearson_r_matrix = pearson_r_matrix.round(3)
pearson_p_matrix = pearson_p_matrix.round(4)

print("Pearson correlation coefficients:")
display(pearson_r_matrix)

print("Pearson correlation p-values:")
display(pearson_p_matrix)

Pearson correlation coefficients:


,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,PCR,WALR,Repo_Rate,Cash_Reserve_Ratio
MSE_Credit_Growth_YoY,1.000,0.511,-0.695,0.405,0.141,0.267,0.164
Credit_Deposit_Ratio,0.511,1.000,-0.545,0.247,0.319,0.487,0.179
GNPA_Ratio,-0.695,-0.545,1.000,-0.546,-0.110,-0.273,-0.234
PCR,0.405,0.247,-0.546,1.000,-0.634,-0.407,0.100
WALR,0.141,0.319,-0.110,-0.634,1.000,0.912,0.231
Repo_Rate,0.267,0.487,-0.273,-0.407,0.912,1.000,0.480
Cash_Reserve_Ratio,0.164,0.179,-0.234,0.100,0.231,0.480,1.000


Pearson correlation p-values:


,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,PCR,WALR,Repo_Rate,Cash_Reserve_Ratio
MSE_Credit_Growth_YoY,0.0000,0.0001,0.0000,0.0029,0.3175,0.0556,0.2461
Credit_Deposit_Ratio,0.0001,0.0000,0.0000,0.0781,0.0211,0.0003,0.2051
GNPA_Ratio,0.0000,0.0000,0.0000,0.0000,0.4392,0.0505,0.0946
PCR,0.0029,0.0781,0.0000,0.0000,0.0000,0.0027,0.4805
WALR,0.3175,0.0211,0.4392,0.0000,0.0000,0.0000,0.1001
Repo_Rate,0.0556,0.0003,0.0505,0.0027,0.0000,0.0000,0.0003
Cash_Reserve_Ratio,0.2461,0.2051,0.0946,0.4805,0.1001,0.0003,0.0000


In [55]:
def significance_stars(p):
    if p < 0.01:
        return "***"
    elif p < 0.05:
        return "**"
    elif p < 0.10:
        return "*"
    else:
        return ""

pearson_r_with_stars = pearson_r_matrix.copy().astype(object)

for row in pearson_r_matrix.index:
    for col in pearson_r_matrix.columns:
        
        r = pearson_r_matrix.loc[row, col]
        p = pearson_p_matrix.loc[row, col]
        
        if row == col:
            pearson_r_with_stars.loc[row, col] = "1.000"
        else:
            pearson_r_with_stars.loc[row, col] = (
                f"{r:.3f}{significance_stars(p)}"
            )

pearson_r_with_stars

,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,PCR,WALR,Repo_Rate,Cash_Reserve_Ratio
MSE_Credit_Growth_YoY,1.000,0.511***,-0.695***,0.405***,0.141,0.267*,0.164
Credit_Deposit_Ratio,0.511***,1.000,-0.545***,0.247*,0.319**,0.487***,0.179
GNPA_Ratio,-0.695***,-0.545***,1.000,-0.546***,-0.110,-0.273*,-0.234*
PCR,0.405***,0.247*,-0.546***,1.000,-0.634***,-0.407***,0.100
WALR,0.141,0.319**,-0.110,-0.634***,1.000,0.912***,0.231
Repo_Rate,0.267*,0.487***,-0.273*,-0.407***,0.912***,1.000,0.480***
Cash_Reserve_Ratio,0.164,0.179,-0.234*,0.100,0.231,0.480***,1.000


In [56]:
credit_growth_corr = (
    pearson_corr["MSE_Credit_Growth_YoY"]
    .drop("MSE_Credit_Growth_YoY")
    .sort_values()
    .reset_index()
)

credit_growth_corr.columns = [
    "Banking Variable",
    "Pearson Correlation with MSE Credit Growth"
]

credit_growth_corr

,Banking Variable,Pearson Correlation with MSE Credit Growth
0,GNPA_Ratio,-0.695
1,WALR,0.141
2,Cash_Reserve_Ratio,0.164
3,Repo_Rate,0.267
4,PCR,0.405
5,Credit_Deposit_Ratio,0.511


In [57]:
fig = px.bar(
    credit_growth_corr,
    x="Pearson Correlation with MSE Credit Growth",
    y="Banking Variable",
    orientation="h",
    text="Pearson Correlation with MSE Credit Growth",
    title="Correlation of Banking Variables with MSE Credit Growth"
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside"
)

fig.update_layout(
    template="plotly_white",
    width=1050,
    height=550,
    title=dict(x=0.02, font=dict(size=20)),
    margin=dict(l=230, r=80, t=90, b=70),
    xaxis_title="Pearson Correlation",
    yaxis_title="",
    showlegend=False
)

fig.update_xaxes(
    range=[
        min(-0.5, credit_growth_corr["Pearson Correlation with MSE Credit Growth"].min() - 0.1),
        max(0.5, credit_growth_corr["Pearson Correlation with MSE Credit Growth"].max() + 0.1)
    ],
    zeroline=True,
    zerolinewidth=2
)

fig.show()

## MCL

In [58]:
banking_only_vars = [
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "PCR",
    "WALR",
    "Repo_Rate",
    "Cash_Reserve_Ratio"
]

banking_corr = corr_df[banking_only_vars].corr().round(3)

banking_corr

,Credit_Deposit_Ratio,GNPA_Ratio,PCR,WALR,Repo_Rate,Cash_Reserve_Ratio
Credit_Deposit_Ratio,1.000,-0.545,0.247,0.319,0.487,0.179
GNPA_Ratio,-0.545,1.000,-0.546,-0.110,-0.273,-0.234
PCR,0.247,-0.546,1.000,-0.634,-0.407,0.100
WALR,0.319,-0.110,-0.634,1.000,0.912,0.231
Repo_Rate,0.487,-0.273,-0.407,0.912,1.000,0.480
Cash_Reserve_Ratio,0.179,-0.234,0.100,0.231,0.480,1.000


In [59]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm

# Keep only complete observations for the selected explanatory variables
vif_df = corr_df[banking_only_vars].dropna().copy()

# Add constant for VIF calculation
X_vif = sm.add_constant(vif_df)

vif_results = pd.DataFrame({
    "Variable": banking_only_vars,
    "VIF": [
        variance_inflation_factor(X_vif.values, i + 1)
        for i in range(len(banking_only_vars))
    ]
})

vif_results["Tolerance"] = 1 / vif_results["VIF"]

vif_results["VIF Interpretation"] = pd.cut(
    vif_results["VIF"],
    bins=[0, 3, 5, 10, float("inf")],
    labels=[
        "Low multicollinearity",
        "Moderate multicollinearity",
        "High multicollinearity",
        "Severe multicollinearity"
    ],
    include_lowest=True
)

vif_results = vif_results.sort_values(
    "VIF",
    ascending=False
).reset_index(drop=True)

vif_results

,Variable,VIF,Tolerance,VIF Interpretation
0,Repo_Rate,16.899647,0.059173,Severe multicollinearity
1,WALR,16.882693,0.059232,Severe multicollinearity
2,PCR,5.883130,0.169978,High multicollinearity
3,GNPA_Ratio,2.866505,0.348857,Low multicollinearity
4,Cash_Reserve_Ratio,2.258167,0.442837,Low multicollinearity
5,Credit_Deposit_Ratio,2.228661,0.448700,Low multicollinearity


In [60]:
# ── VIF after removing Repo Rate ─────────────────────────────────────────────

baseline_vars = [
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "PCR",
    "WALR",
    "Cash_Reserve_Ratio"
]

baseline_vif_df = corr_df[baseline_vars].dropna().copy()
X_baseline_vif = sm.add_constant(baseline_vif_df)

baseline_vif_results = pd.DataFrame({
    "Variable": baseline_vars,
    "VIF": [
        variance_inflation_factor(X_baseline_vif.values, i + 1)
        for i in range(len(baseline_vars))
    ]
})

baseline_vif_results["Tolerance"] = 1 / baseline_vif_results["VIF"]

baseline_vif_results["Interpretation"] = pd.cut(
    baseline_vif_results["VIF"],
    bins=[0, 3, 5, 10, float("inf")],
    labels=[
        "Low multicollinearity",
        "Moderate multicollinearity",
        "High multicollinearity",
        "Severe multicollinearity"
    ],
    include_lowest=True
)

baseline_vif_results.sort_values("VIF", ascending=False).reset_index(drop=True)

,Variable,VIF,Tolerance,Interpretation
0,PCR,5.806515,0.172220,High multicollinearity
1,WALR,4.736252,0.211137,Moderate multicollinearity
2,GNPA_Ratio,2.866504,0.348857,Low multicollinearity
3,Credit_Deposit_Ratio,1.824374,0.548133,Low multicollinearity
4,Cash_Reserve_Ratio,1.204925,0.829927,Low multicollinearity


In [61]:
# ── Core banking-stress model without PCR ────────────────────────────────────

core_vars = [
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "WALR",
    "Cash_Reserve_Ratio"
]

core_vif_df = corr_df[core_vars].dropna().copy()
X_core_vif = sm.add_constant(core_vif_df)

core_vif_results = pd.DataFrame({
    "Variable": core_vars,
    "VIF": [
        variance_inflation_factor(X_core_vif.values, i + 1)
        for i in range(len(core_vars))
    ]
})

core_vif_results["Tolerance"] = 1 / core_vif_results["VIF"]

core_vif_results.sort_values("VIF", ascending=False).reset_index(drop=True)

,Variable,VIF,Tolerance
0,Credit_Deposit_Ratio,1.574212,0.635238
1,GNPA_Ratio,1.481744,0.674880
2,WALR,1.168818,0.855565
3,Cash_Reserve_Ratio,1.107890,0.902617


## Core model:

In [62]:
core_vars = [
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "WALR",
    "Cash_Reserve_Ratio"
]

In [63]:
core_model_df = quarterly_df[
    [
        "Quarter_End",
        "MSE_Credit_Growth_YoY",
        "Credit_Deposit_Ratio",
        "GNPA_Ratio",
        "WALR",
        "Cash_Reserve_Ratio"
    ]
].dropna().copy()

print("Sample start:", core_model_df["Quarter_End"].min().date())
print("Sample end  :", core_model_df["Quarter_End"].max().date())
print("Observations:", len(core_model_df))

core_model_df.head()

Sample start: 2013-03-31
Sample end  : 2025-12-31
Observations: 52


,Quarter_End,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,WALR,Cash_Reserve_Ratio
0,2013-03-31,12.769316,77.93,3.422945,11.46,4.0
1,2013-06-30,22.630200,76.42,4.004487,11.40,4.0
2,2013-09-30,21.847790,78.35,4.218281,11.96,4.0
3,2013-12-31,22.894464,76.81,4.397677,11.70,4.0
4,2014-03-31,25.879074,77.79,4.115068,11.57,4.0


In [64]:
import statsmodels.api as sm

y = core_model_df["MSE_Credit_Growth_YoY"]

X = core_model_df[core_vars]
X = sm.add_constant(X)

core_ols = sm.OLS(y, X).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(core_ols.summary())

                              OLS Regression Results                             
Dep. Variable:     MSE_Credit_Growth_YoY   R-squared:                       0.509
Model:                               OLS   Adj. R-squared:                  0.467
Method:                    Least Squares   F-statistic:                     7.912
Date:                   Tue, 23 Jun 2026   Prob (F-statistic):           5.82e-05
Time:                           12:55:45   Log-Likelihood:                -154.55
No. Observations:                     52   AIC:                             319.1
Df Residuals:                         47   BIC:                             328.9
Df Model:                              4                                         
Covariance Type:                     HAC                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
co

In [65]:
core_results = pd.DataFrame({
    "Variable": core_ols.params.index,
    "Coefficient": core_ols.params.values,
    "HAC Std. Error": core_ols.bse.values,
    "t-statistic": core_ols.tvalues.values,
    "p-value": core_ols.pvalues.values,
    "95% CI Lower": core_ols.conf_int().iloc[:, 0].values,
    "95% CI Upper": core_ols.conf_int().iloc[:, 1].values
})

core_results["Significance"] = core_results["p-value"].apply(
    lambda p: "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""
)

core_results = core_results.round(4)

core_results

,Variable,Coefficient,HAC Std. Error,t-statistic,p-value,95% CI Lower,95% CI Upper,Significance
0,const,-10.4149,26.4689,-0.3935,0.6940,-62.2930,41.4631,
1,Credit_Deposit_Ratio,0.4251,0.3534,1.2029,0.2290,-0.2675,1.1176,
2,GNPA_Ratio,-1.4250,0.3050,-4.6719,0.0000,-2.0229,-0.8272,***
3,WALR,0.1203,1.0307,0.1167,0.9071,-1.8998,2.1404,
4,Cash_Reserve_Ratio,-0.2236,1.9178,-0.1166,0.9072,-3.9825,3.5353,


In [66]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

X_vif = sm.add_constant(core_model_df[core_vars])

core_vif = pd.DataFrame({
    "Variable": core_vars,
    "VIF": [
        variance_inflation_factor(X_vif.values, i + 1)
        for i in range(len(core_vars))
    ]
})

core_vif["Tolerance"] = 1 / core_vif["VIF"]

core_vif.round(3)

,Variable,VIF,Tolerance
0,Credit_Deposit_Ratio,1.574,0.635
1,GNPA_Ratio,1.482,0.675
2,WALR,1.169,0.856
3,Cash_Reserve_Ratio,1.108,0.903


### A 1 percentage-point increase in the GNPA ratio is associated with about a 1.43 percentage-point decline in MSE credit growth, holding CDR, WALR, and CRR constant.

## Stationary checks:

In [67]:
from statsmodels.tsa.stattools import adfuller, kpss
import numpy as np

In [68]:
def stationarity_tests(series, variable_name):
    series = pd.Series(series).dropna().astype(float)

    # ADF: H0 = unit root / non-stationary
    adf_result = adfuller(
        series,
        regression="c",
        autolag="AIC"
    )

    # KPSS: H0 = stationary
    try:
        kpss_result = kpss(
            series,
            regression="c",
            nlags="auto"
        )

        kpss_stat = kpss_result[0]
        kpss_pvalue = kpss_result[1]
    except Exception:
        kpss_stat = np.nan
        kpss_pvalue = np.nan

    return {
        "Variable": variable_name,
        "Observations": len(series),

        "ADF Statistic": adf_result[0],
        "ADF p-value": adf_result[1],
        "ADF Lags Used": adf_result[2],

        "KPSS Statistic": kpss_stat,
        "KPSS p-value": kpss_pvalue,

        "ADF Decision": (
            "Stationary"
            if adf_result[1] < 0.05
            else "Non-stationary"
        ),

        "KPSS Decision": (
            "Stationary"
            if pd.notna(kpss_pvalue) and kpss_pvalue > 0.05
            else "Non-stationary"
        )
    }

In [69]:
stationarity_vars = [
    "MSE_Credit_Growth_YoY",
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "WALR",
    "Cash_Reserve_Ratio"
]

stationarity_results = []

for var in stationarity_vars:
    stationarity_results.append(
        stationarity_tests(
            core_model_df[var],
            var
        )
    )

stationarity_table = pd.DataFrame(stationarity_results)

stationarity_table[
    [
        "Variable",
        "Observations",
        "ADF Statistic",
        "ADF p-value",
        "ADF Lags Used",
        "ADF Decision",
        "KPSS Statistic",
        "KPSS p-value",
        "KPSS Decision"
    ]
].round(4)

C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\2920927558.py:13: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_result = kpss(
C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\2920927558.py:13: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_result = kpss(
C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\2920927558.py:13: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is smaller than the p-value returned.

  kpss_result = kpss(
C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\2920927558.py:13: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is gr

,Variable,Observations,ADF Statistic,ADF p-value,ADF Lags Used,ADF Decision,KPSS Statistic,KPSS p-value,KPSS Decision
0,MSE_Credit_Growth_YoY,52,-1.9680,0.3008,0,Non-stationary,0.2558,0.1000,Stationary
1,Credit_Deposit_Ratio,52,-1.3630,0.5999,0,Non-stationary,0.2075,0.1000,Stationary
2,GNPA_Ratio,52,-1.7048,0.4286,5,Non-stationary,0.3856,0.0834,Stationary
3,WALR,52,-1.9885,0.2916,9,Non-stationary,0.8282,0.0100,Non-stationary
4,Cash_Reserve_Ratio,52,-2.1263,0.2341,5,Non-stationary,0.1006,0.1000,Stationary


In [70]:
def combined_stationarity_conclusion(row):
    
    adf_stationary = row["ADF p-value"] < 0.05
    kpss_stationary = row["KPSS p-value"] > 0.05
    
    if adf_stationary and kpss_stationary:
        return "Stationary: use in levels"
    
    elif (not adf_stationary) and (not kpss_stationary):
        return "Non-stationary: transform required"
    
    else:
        return "Mixed result: inspect trend / test transformation"

stationarity_table["Final Conclusion"] = stationarity_table.apply(
    combined_stationarity_conclusion,
    axis=1
)

stationarity_table[
    [
        "Variable",
        "ADF p-value",
        "KPSS p-value",
        "Final Conclusion"
    ]
].round(4)

,Variable,ADF p-value,KPSS p-value,Final Conclusion
0,MSE_Credit_Growth_YoY,0.3008,0.1000,Mixed result: inspect trend / test transformation
1,Credit_Deposit_Ratio,0.5999,0.1000,Mixed result: inspect trend / test transformation
2,GNPA_Ratio,0.4286,0.0834,Mixed result: inspect trend / test transformation
3,WALR,0.2916,0.0100,Non-stationary: transform required
4,Cash_Reserve_Ratio,0.2341,0.1000,Mixed result: inspect trend / test transformation


In [71]:
# ── Create quarter-on-quarter changes where needed ───────────────────────────

ts_df = core_model_df.copy().sort_values("Quarter_End").reset_index(drop=True)

# Primary required transformation
ts_df["d_WALR"] = ts_df["WALR"].diff()

# Robustness transformations
ts_df["d_Credit_Deposit_Ratio"] = ts_df["Credit_Deposit_Ratio"].diff()
ts_df["d_GNPA_Ratio"] = ts_df["GNPA_Ratio"].diff()
ts_df["d_Cash_Reserve_Ratio"] = ts_df["Cash_Reserve_Ratio"].diff()

ts_df = ts_df.dropna().reset_index(drop=True)

ts_df[
    [
        "Quarter_End",
        "MSE_Credit_Growth_YoY",
        "Credit_Deposit_Ratio",
        "GNPA_Ratio",
        "d_WALR",
        "Cash_Reserve_Ratio"
    ]
].head()

,Quarter_End,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,d_WALR,Cash_Reserve_Ratio
0,2013-06-30,22.630200,76.42,4.004487,-0.06,4.0
1,2013-09-30,21.847790,78.35,4.218281,0.56,4.0
2,2013-12-31,22.894464,76.81,4.397677,-0.26,4.0
3,2014-03-31,25.879074,77.79,4.115068,-0.13,4.0
4,2014-06-30,19.548574,77.16,4.319126,0.03,4.0


In [72]:
transformed_stationarity_vars = [
    "MSE_Credit_Growth_YoY",
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "Cash_Reserve_Ratio",
    "d_WALR",
    "d_Credit_Deposit_Ratio",
    "d_GNPA_Ratio",
    "d_Cash_Reserve_Ratio"
]

transformed_results = []

for var in transformed_stationarity_vars:
    transformed_results.append(
        stationarity_tests(ts_df[var], var)
    )

transformed_stationarity_table = pd.DataFrame(transformed_results)

transformed_stationarity_table[
    [
        "Variable",
        "Observations",
        "ADF Statistic",
        "ADF p-value",
        "KPSS Statistic",
        "KPSS p-value",
        "ADF Decision",
        "KPSS Decision"
    ]
].round(4)

C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\2920927558.py:13: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_result = kpss(
C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\2920927558.py:13: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_result = kpss(
C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\2920927558.py:13: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is greater than the p-value returned.

  kpss_result = kpss(
C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\2920927558.py:13: InterpolationWarning: The test statistic is outside of the range of p-values available in the
look-up table. The actual p-value is gr

,Variable,Observations,ADF Statistic,ADF p-value,KPSS Statistic,KPSS p-value,ADF Decision,KPSS Decision
0,MSE_Credit_Growth_YoY,51,-1.4549,0.5557,0.2602,0.1000,Non-stationary,Stationary
1,Credit_Deposit_Ratio,51,-1.2522,0.6508,0.2170,0.1000,Non-stationary,Stationary
2,GNPA_Ratio,51,-1.7358,0.4127,0.4102,0.0728,Non-stationary,Stationary
3,Cash_Reserve_Ratio,51,-2.0994,0.2447,0.1028,0.1000,Non-stationary,Stationary
4,d_WALR,51,-2.7892,0.0598,0.1104,0.1000,Non-stationary,Stationary
5,d_Credit_Deposit_Ratio,51,-7.3602,0.0000,0.2169,0.1000,Stationary,Stationary
6,d_GNPA_Ratio,51,-1.4643,0.5511,0.5808,0.0244,Non-stationary,Non-stationary
7,d_Cash_Reserve_Ratio,51,-2.8529,0.0511,0.1631,0.1000,Non-stationary,Stationary


In [73]:
# ── Preferred hybrid stationary specification ────────────────────────────────

hybrid_vars = [
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "d_WALR",
    "Cash_Reserve_Ratio"
]

hybrid_model_df = ts_df[
    ["Quarter_End", "MSE_Credit_Growth_YoY"] + hybrid_vars
].dropna().copy()

y_hybrid = hybrid_model_df["MSE_Credit_Growth_YoY"]

X_hybrid = sm.add_constant(hybrid_model_df[hybrid_vars])

hybrid_ols = sm.OLS(y_hybrid, X_hybrid).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(hybrid_ols.summary())

                              OLS Regression Results                             
Dep. Variable:     MSE_Credit_Growth_YoY   R-squared:                       0.534
Model:                               OLS   Adj. R-squared:                  0.493
Method:                    Least Squares   F-statistic:                     8.902
Date:                   Tue, 23 Jun 2026   Prob (F-statistic):           2.07e-05
Time:                           12:55:45   Log-Likelihood:                -150.73
No. Observations:                     51   AIC:                             311.5
Df Residuals:                         46   BIC:                             321.1
Df Model:                              4                                         
Covariance Type:                     HAC                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
co

In [74]:
# ── Preferred hybrid stationary specification ────────────────────────────────

hybrid_vars = [
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "d_WALR",
]

hybrid_model_df = ts_df[
    ["Quarter_End", "MSE_Credit_Growth_YoY"] + hybrid_vars
].dropna().copy()

y_hybrid = hybrid_model_df["MSE_Credit_Growth_YoY"]

X_hybrid = sm.add_constant(hybrid_model_df[hybrid_vars])

hybrid_ols = sm.OLS(y_hybrid, X_hybrid).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(hybrid_ols.summary())

                              OLS Regression Results                             
Dep. Variable:     MSE_Credit_Growth_YoY   R-squared:                       0.530
Model:                               OLS   Adj. R-squared:                  0.500
Method:                    Least Squares   F-statistic:                     10.83
Date:                   Tue, 23 Jun 2026   Prob (F-statistic):           1.58e-05
Time:                           12:55:45   Log-Likelihood:                -150.93
No. Observations:                     51   AIC:                             309.9
Df Residuals:                         47   BIC:                             317.6
Df Model:                              3                                         
Covariance Type:                     HAC                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
co

## Granger Causality

In [75]:
from statsmodels.tsa.api import VAR
from scipy.stats import chi2
import numpy as np
import pandas as pd

In [76]:
causality_vars = [
    "MSE_Credit_Growth_YoY",
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "d_WALR",
    "Cash_Reserve_Ratio"
]

causality_df = ts_df[
    ["Quarter_End"] + causality_vars
].dropna().copy()

causality_df = causality_df.sort_values("Quarter_End").reset_index(drop=True)

print("Start:", causality_df["Quarter_End"].min().date())
print("End  :", causality_df["Quarter_End"].max().date())
print("Observations:", len(causality_df))

causality_df.head()

Start: 2013-06-30
End  : 2025-12-31
Observations: 51


,Quarter_End,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,d_WALR,Cash_Reserve_Ratio
0,2013-06-30,22.630200,76.42,4.004487,-0.06,4.0
1,2013-09-30,21.847790,78.35,4.218281,0.56,4.0
2,2013-12-31,22.894464,76.81,4.397677,-0.26,4.0
3,2014-03-31,25.879074,77.79,4.115068,-0.13,4.0
4,2014-06-30,19.548574,77.16,4.319126,0.03,4.0


In [77]:
var_data = causality_df[causality_vars].copy()

lag_selection = VAR(var_data).select_order(maxlags=4)

print(lag_selection.summary())

 VAR Order Selection (* highlights the minimums) 
      AIC         BIC         FPE         HQIC   
-------------------------------------------------
0       2.212       2.409       9.134       2.286
1     -3.747*     -2.566*    0.02376*     -3.302*
2      -3.559      -1.394     0.02976      -2.744
3      -3.440     -0.2905     0.03695      -2.255
4      -3.411      0.7218     0.04630      -1.856
-------------------------------------------------


In [78]:
selected_lag = lag_selection.selected_orders["bic"]

print("BIC-selected lag:", selected_lag)

# Toda-Yamamoto augmentation:
# d_max = 1 because some variables appear I(1) / mixed
d_max = 1

ty_lag = selected_lag + d_max

print("Toda-Yamamoto VAR lag =", ty_lag)

BIC-selected lag: 1
Toda-Yamamoto VAR lag = 2


In [79]:
ty_var_model = VAR(var_data).fit(ty_lag)

print(ty_var_model.summary())

  Summary of Regression Results   
Model:                         VAR
Method:                        OLS
Date:           Tue, 23, Jun, 2026
Time:                     12:55:45
--------------------------------------------------------------------
No. of Equations:         5.00000    BIC:                   -1.44694
Nobs:                     49.0000    HQIC:                  -2.76477
Log likelihood:          -205.165    FPE:                  0.0292605
AIC:                     -3.57041    Det(Omega_mle):       0.0106293
--------------------------------------------------------------------
Results for equation MSE_Credit_Growth_YoY
                              coefficient       std. error           t-stat            prob
-------------------------------------------------------------------------------------------
const                            2.139250        23.932076            0.089           0.929
L1.MSE_Credit_Growth_YoY         0.677395         0.184217            3.677           0.000


In [80]:
def causality_test(var_model, caused_variable, causing_variable):
    """
    Wald Granger-causality test from a fitted VAR model.
    H0: causing_variable does not Granger-cause caused_variable.
    """

    result = var_model.test_causality(
        caused=caused_variable,
        causing=causing_variable,
        kind="wald"
    )

    return {
        "Causing Variable": causing_variable,
        "Caused Variable": caused_variable,
        "Wald Statistic": float(result.test_statistic),
        "Degrees of Freedom": str(result.df),
        "P-value": float(result.pvalue),
        "Conclusion": result.conclusion
    }

In [81]:
predictor_vars = [
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "d_WALR",
    "Cash_Reserve_Ratio"
]

forward_results = []

for predictor in predictor_vars:
    forward_results.append(
        causality_test(
            var_model=ty_var_model,
            caused_variable="MSE_Credit_Growth_YoY",
            causing_variable=predictor
        )
    )

forward_causality_df = pd.DataFrame(forward_results)

forward_causality_df["Significance"] = forward_causality_df["P-value"].apply(
    lambda p: "*** 1%" if p < 0.01
    else "** 5%" if p < 0.05
    else "* 10%" if p < 0.10
    else "Not significant"
)

forward_causality_df = (
    forward_causality_df
    .sort_values("P-value")
    .reset_index(drop=True)
)

forward_causality_df.round(4)

,Causing Variable,Caused Variable,Wald Statistic,Degrees of Freedom,P-value,Conclusion,Significance
0,GNPA_Ratio,MSE_Credit_Growth_YoY,4.0140,2,0.1344,fail to reject,Not significant
1,Credit_Deposit_Ratio,MSE_Credit_Growth_YoY,1.3822,2,0.5010,fail to reject,Not significant
2,Cash_Reserve_Ratio,MSE_Credit_Growth_YoY,1.0395,2,0.5947,fail to reject,Not significant
3,d_WALR,MSE_Credit_Growth_YoY,1.0246,2,0.5991,fail to reject,Not significant


In [82]:
reverse_results = []

for predictor in predictor_vars:
    reverse_results.append(
        causality_test(
            var_model=ty_var_model,
            caused_variable=predictor,
            causing_variable="MSE_Credit_Growth_YoY"
        )
    )

reverse_causality_df = pd.DataFrame(reverse_results)

reverse_causality_df["Significance"] = reverse_causality_df["P-value"].apply(
    lambda p: "*** 1%" if p < 0.01
    else "** 5%" if p < 0.05
    else "* 10%" if p < 0.10
    else "Not significant"
)

reverse_causality_df = (
    reverse_causality_df
    .sort_values("P-value")
    .reset_index(drop=True)
)

reverse_causality_df.round(4)

,Causing Variable,Caused Variable,Wald Statistic,Degrees of Freedom,P-value,Conclusion,Significance
0,MSE_Credit_Growth_YoY,d_WALR,7.0970,2,0.0288,reject,** 5%
1,MSE_Credit_Growth_YoY,Cash_Reserve_Ratio,1.7698,2,0.4128,fail to reject,Not significant
2,MSE_Credit_Growth_YoY,GNPA_Ratio,1.5645,2,0.4574,fail to reject,Not significant
3,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,1.2639,2,0.5316,fail to reject,Not significant


In [83]:
causality_summary = forward_causality_df[
    [
        "Causing Variable",
        "Wald Statistic",
        "P-value",
        "Significance"
    ]
].rename(
    columns={
        "Causing Variable": "Variable",
        "Wald Statistic": "Wald: Variable → MSE Growth",
        "P-value": "P-value: Variable → MSE Growth",
        "Significance": "Decision: Variable → MSE Growth"
    }
)

reverse_summary = reverse_causality_df[
    [
        "Causing Variable",
        "Wald Statistic",
        "P-value",
        "Significance"
    ]
].rename(
    columns={
        "Causing Variable": "Variable",
        "Wald Statistic": "Wald: MSE Growth → Variable",
        "P-value": "P-value: MSE Growth → Variable",
        "Significance": "Decision: MSE Growth → Variable"
    }
)

causality_summary = causality_summary.merge(
    reverse_summary,
    on="Variable",
    how="left"
)

causality_summary.round(4)

,Variable,Wald: Variable → MSE Growth,P-value: Variable → MSE Growth,Decision: Variable → MSE Growth,Wald: MSE Growth → Variable,P-value: MSE Growth → Variable,Decision: MSE Growth → Variable
0,GNPA_Ratio,4.0140,0.1344,Not significant,NaN,NaN,NaN
1,Credit_Deposit_Ratio,1.3822,0.5010,Not significant,NaN,NaN,NaN
2,Cash_Reserve_Ratio,1.0395,0.5947,Not significant,NaN,NaN,NaN
3,d_WALR,1.0246,0.5991,Not significant,NaN,NaN,NaN


### The augmented-VAR Granger-causality tests did not find evidence that past changes in the credit-deposit ratio, gross NPA ratio, cash reserve ratio, or weighted average lending rate predict subsequent MSE credit growth. However, the contemporaneous HAC regression shows a strong negative association between the gross NPA ratio and MSE credit growth. This indicates that banking-sector asset-quality stress is closely associated with weaker MSE lending conditions, although the available quarterly sample does not establish a robust lagged predictive relationship.

## ARDL

In [84]:
# ── Distributed lag dataset ──────────────────────────────────────────────────

adl_base = ts_df[
    [
        "Quarter_End",
        "MSE_Credit_Growth_YoY",
        "Credit_Deposit_Ratio",
        "GNPA_Ratio",
        "d_WALR",
        "Cash_Reserve_Ratio"
    ]
].copy()

adl_base = adl_base.sort_values("Quarter_End").reset_index(drop=True)

dependent_var = "MSE_Credit_Growth_YoY"

adl_predictors = [
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "d_WALR",
    "Cash_Reserve_Ratio"
]

adl_base.head()

,Quarter_End,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,d_WALR,Cash_Reserve_Ratio
0,2013-06-30,22.630200,76.42,4.004487,-0.06,4.0
1,2013-09-30,21.847790,78.35,4.218281,0.56,4.0
2,2013-12-31,22.894464,76.81,4.397677,-0.26,4.0
3,2014-03-31,25.879074,77.79,4.115068,-0.13,4.0
4,2014-06-30,19.548574,77.16,4.319126,0.03,4.0


In [85]:
def create_adl_dataset(df, dependent, predictors, y_lags=1, x_lags=1):
    """
    Builds an autoregressive distributed lag dataset.

    y_lags = number of lags of MSE credit growth
    x_lags = number of lags for each banking variable
    """

    temp = df.copy()

    # Lagged dependent variable
    for lag in range(1, y_lags + 1):
        temp[f"{dependent}_L{lag}"] = temp[dependent].shift(lag)

    # Current and lagged explanatory variables
    for variable in predictors:
        for lag in range(0, x_lags + 1):
            if lag == 0:
                temp[f"{variable}_L0"] = temp[variable]
            else:
                temp[f"{variable}_L{lag}"] = temp[variable].shift(lag)

    temp = temp.dropna().reset_index(drop=True)

    y_col = dependent

    x_cols = (
        [f"{dependent}_L{lag}" for lag in range(1, y_lags + 1)]
        +
        [
            f"{variable}_L{lag}"
            for variable in predictors
            for lag in range(0, x_lags + 1)
        ]
    )

    return temp, y_col, x_cols

In [86]:
adl_selection_results = []

for y_lags in [0, 1, 2]:
    for x_lags in [0, 1, 2]:

        temp_df, y_col, x_cols = create_adl_dataset(
            df=adl_base,
            dependent=dependent_var,
            predictors=adl_predictors,
            y_lags=y_lags,
            x_lags=x_lags
        )

        if len(temp_df) <= len(x_cols) + 8:
            continue

        y_temp = temp_df[y_col]
        X_temp = sm.add_constant(temp_df[x_cols])

        model_temp = sm.OLS(y_temp, X_temp).fit(
            cov_type="HAC",
            cov_kwds={"maxlags": 4}
        )

        adl_selection_results.append({
            "Dependent Lags": y_lags,
            "Banking Variable Lags": x_lags,
            "Observations": int(model_temp.nobs),
            "Parameters": int(len(model_temp.params)),
            "AIC": model_temp.aic,
            "BIC": model_temp.bic,
            "Adjusted R²": model_temp.rsquared_adj
        })

adl_selection_table = (
    pd.DataFrame(adl_selection_results)
    .sort_values("BIC")
    .reset_index(drop=True)
)

adl_selection_table.round(3)

,Dependent Lags,Banking Variable Lags,Observations,Parameters,AIC,BIC,Adjusted R²
0,2,0,49,7,287.156,300.398,0.606
1,1,0,50,6,290.364,301.836,0.625
2,2,1,49,11,285.485,306.295,0.643
3,1,1,50,10,288.628,307.748,0.660
4,1,2,49,14,289.328,315.814,0.629
5,2,2,49,15,291.167,319.545,0.619
6,0,0,51,5,311.466,321.126,0.493
7,0,1,50,9,306.899,324.107,0.503
8,0,2,49,13,308.022,332.615,0.449


In [87]:
# Replace these only if Cell 54 strongly favours another compact specification.

selected_y_lags = 1
selected_x_lags = 1

In [88]:
adl_df, adl_y_col, adl_x_cols = create_adl_dataset(
    df=adl_base,
    dependent=dependent_var,
    predictors=adl_predictors,
    y_lags=selected_y_lags,
    x_lags=selected_x_lags
)

y_adl = adl_df[adl_y_col]
X_adl = sm.add_constant(adl_df[adl_x_cols])

adl_model = sm.OLS(y_adl, X_adl).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(adl_model.summary())

                              OLS Regression Results                             
Dep. Variable:     MSE_Credit_Growth_YoY   R-squared:                       0.723
Model:                               OLS   Adj. R-squared:                  0.660
Method:                    Least Squares   F-statistic:                     25.06
Date:                   Tue, 23 Jun 2026   Prob (F-statistic):           9.40e-14
Time:                           12:55:45   Log-Likelihood:                -134.31
No. Observations:                     50   AIC:                             288.6
Df Residuals:                         40   BIC:                             307.7
Df Model:                              9                                         
Covariance Type:                     HAC                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [89]:
adl_results = pd.DataFrame({
    "Variable": adl_model.params.index,
    "Coefficient": adl_model.params.values,
    "HAC Std. Error": adl_model.bse.values,
    "z-statistic": adl_model.tvalues.values,
    "P-value": adl_model.pvalues.values,
    "95% CI Lower": adl_model.conf_int().iloc[:, 0].values,
    "95% CI Upper": adl_model.conf_int().iloc[:, 1].values
})

adl_results["Significance"] = adl_results["P-value"].apply(
    lambda p: "***" if p < 0.01
    else "**" if p < 0.05
    else "*" if p < 0.10
    else ""
)

adl_results.round(4)

,Variable,Coefficient,HAC Std. Error,z-statistic,P-value,95% CI Lower,95% CI Upper,Significance
0,const,0.7101,19.1215,0.0371,0.9704,-36.7674,38.1876,
1,MSE_Credit_Growth_YoY_L1,0.6115,0.1016,6.0198,0.0000,0.4124,0.8106,***
2,Credit_Deposit_Ratio_L0,1.0800,0.3592,3.0066,0.0026,0.3760,1.7840,***
3,Credit_Deposit_Ratio_L1,-0.9876,0.4126,-2.3936,0.0167,-1.7962,-0.1789,**
4,GNPA_Ratio_L0,-2.5359,1.0518,-2.4109,0.0159,-4.5975,-0.4743,**
5,GNPA_Ratio_L1,2.0134,1.0781,1.8676,0.0618,-0.0995,4.1264,*
6,d_WALR_L0,-3.7705,2.3308,-1.6177,0.1057,-8.3388,0.7977,
7,d_WALR_L1,0.4115,3.0740,0.1339,0.8935,-5.6135,6.4365,
8,Cash_Reserve_Ratio_L0,1.4454,2.3108,0.6255,0.5316,-3.0836,5.9744,
9,Cash_Reserve_Ratio_L1,-1.3539,2.1688,-0.6243,0.5325,-5.6046,2.8968,


In [90]:
joint_lag_results = []

for variable in adl_predictors:

    terms_to_test = [
        f"{variable}_L{lag}"
        for lag in range(0, selected_x_lags + 1)
        if f"{variable}_L{lag}" in adl_model.params.index
    ]

    restriction = " = 0, ".join(terms_to_test) + " = 0"

    test = adl_model.wald_test(restriction)

    joint_lag_results.append({
        "Variable": variable,
        "Lags Tested": ", ".join(terms_to_test),
        "Wald Statistic": float(test.statistic),
        "P-value": float(test.pvalue)
    })

joint_lag_results_df = pd.DataFrame(joint_lag_results)

joint_lag_results_df["Significance"] = joint_lag_results_df["P-value"].apply(
    lambda p: "*** 1%" if p < 0.01
    else "** 5%" if p < 0.05
    else "* 10%" if p < 0.10
    else "Not significant"
)

joint_lag_results_df.round(4)

C:\Users\JAYPAL SINGH\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\base\model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\596433558.py:18: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "Wald Statistic": float(test.statistic),


,Variable,Lags Tested,Wald Statistic,P-value,Significance
0,Credit_Deposit_Ratio,"Credit_Deposit_Ratio_L0, Credit_Deposit_Ratio_L1",9.0813,0.0107,** 5%
1,GNPA_Ratio,"GNPA_Ratio_L0, GNPA_Ratio_L1",13.4921,0.0012,*** 1%
2,d_WALR,"d_WALR_L0, d_WALR_L1",2.6285,0.2687,Not significant
3,Cash_Reserve_Ratio,"Cash_Reserve_Ratio_L0, Cash_Reserve_Ratio_L1",0.6340,0.7283,Not significant


In [91]:
cumulative_effects = []

for variable in adl_predictors:

    lag_terms = [
        f"{variable}_L{lag}"
        for lag in range(0, selected_x_lags + 1)
        if f"{variable}_L{lag}" in adl_model.params.index
    ]

    cumulative_beta = adl_model.params[lag_terms].sum()

    cumulative_effects.append({
        "Variable": variable,
        "Included Terms": ", ".join(lag_terms),
        "Cumulative Effect": cumulative_beta
    })

cumulative_effects_df = pd.DataFrame(cumulative_effects)

cumulative_effects_df.round(4)

,Variable,Included Terms,Cumulative Effect
0,Credit_Deposit_Ratio,"Credit_Deposit_Ratio_L0, Credit_Deposit_Ratio_L1",0.0924
1,GNPA_Ratio,"GNPA_Ratio_L0, GNPA_Ratio_L1",-0.5225
2,d_WALR,"d_WALR_L0, d_WALR_L1",-3.3590
3,Cash_Reserve_Ratio,"Cash_Reserve_Ratio_L0, Cash_Reserve_Ratio_L1",0.0915


In [92]:
from statsmodels.stats.diagnostic import acorr_breusch_godfrey

bg_test = acorr_breusch_godfrey(
    adl_model,
    nlags=4
)

bg_results = pd.DataFrame({
    "Test": [
        "Breusch-Godfrey LM Statistic",
        "Breusch-Godfrey LM p-value",
        "F Statistic",
        "F p-value"
    ],
    "Value": bg_test
})

bg_results

,Test,Value
0,Breusch-Godfrey LM Statistic,8.122955
1,Breusch-Godfrey LM p-value,0.087177
2,F Statistic,1.745744
3,F p-value,0.161386


In [93]:
# ── Long-run multipliers for ADL(1,1) ────────────────────────────────────────

phi = adl_model.params["MSE_Credit_Growth_YoY_L1"]

long_run_results = []

for variable in [
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "d_WALR",
    "Cash_Reserve_Ratio"
]:
    beta_0 = adl_model.params[f"{variable}_L0"]
    beta_1 = adl_model.params[f"{variable}_L1"]
    
    short_run_sum = beta_0 + beta_1
    long_run_multiplier = short_run_sum / (1 - phi)
    
    long_run_results.append({
        "Variable": variable,
        "Current Effect": beta_0,
        "Lagged Effect": beta_1,
        "Cumulative Short-Run Effect": short_run_sum,
        "Long-Run Multiplier": long_run_multiplier
    })

long_run_results_df = pd.DataFrame(long_run_results)

long_run_results_df.round(4)

,Variable,Current Effect,Lagged Effect,Cumulative Short-Run Effect,Long-Run Multiplier
0,Credit_Deposit_Ratio,1.0800,-0.9876,0.0924,0.2379
1,GNPA_Ratio,-2.5359,2.0134,-0.5225,-1.3449
2,d_WALR,-3.7705,0.4115,-3.3590,-8.6464
3,Cash_Reserve_Ratio,1.4454,-1.3539,0.0915,0.2356


## Bank asset-quality deterioration has a statistically significant immediate negative association with MSE credit growth. Credit-deposit pressure has a short-lived effect that reverses in the following quarter, while lending-rate changes and CRR do not show statistically precise independent effects.

## Full Model

In [94]:
# ── Build full-model dataset including PCR ───────────────────────────────────

full_adl_base = quarterly_df[
    [
        "Quarter_End",
        "MSE_Credit_Growth_YoY",
        "Credit_Deposit_Ratio",
        "GNPA_Ratio",
        "PCR",
        "WALR",
        "Cash_Reserve_Ratio"
    ]
].copy()

full_adl_base = (
    full_adl_base
    .sort_values("Quarter_End")
    .reset_index(drop=True)
)

# WALR level was non-stationary, so use quarterly change
full_adl_base["d_WALR"] = full_adl_base["WALR"].diff()

# Remove WALR level after creating its difference
full_adl_base = full_adl_base.drop(columns="WALR")

# Keep only quarters where PCR and all other required variables are observed
full_adl_base = (
    full_adl_base
    .dropna()
    .reset_index(drop=True)
)

print("Sample start:", full_adl_base["Quarter_End"].min().date())
print("Sample end  :", full_adl_base["Quarter_End"].max().date())
print("Observations before lags:", len(full_adl_base))

full_adl_base.head()

Sample start: 2013-06-30
Sample end  : 2025-12-31
Observations before lags: 51


,Quarter_End,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,PCR,Cash_Reserve_Ratio,d_WALR
0,2013-06-30,22.630200,76.42,4.004487,47.716288,4.0,-0.06
1,2013-09-30,21.847790,78.35,4.218281,47.301251,4.0,0.56
2,2013-12-31,22.894464,76.81,4.397677,47.353143,4.0,-0.26
3,2014-03-31,25.879074,77.79,4.115068,54.563439,4.0,-0.13
4,2014-06-30,19.548574,77.16,4.319126,51.119134,4.0,0.03


In [95]:
# Check whether retained observations remain consecutive quarters
quarter_gaps = full_adl_base["Quarter_End"].diff().dt.days

print(
    full_adl_base.loc[
        quarter_gaps > 100,
        ["Quarter_End"]
    ]
)

Empty DataFrame
Columns: [Quarter_End]
Index: []


In [96]:
full_predictors = [
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "PCR",
    "d_WALR",
    "Cash_Reserve_Ratio"
]

full_adl_df, full_y_col, full_x_cols = create_adl_dataset(
    df=full_adl_base,
    dependent="MSE_Credit_Growth_YoY",
    predictors=full_predictors,
    y_lags=1,
    x_lags=1
)

y_full_adl = full_adl_df[full_y_col]
X_full_adl = sm.add_constant(full_adl_df[full_x_cols])

full_adl_model = sm.OLS(y_full_adl, X_full_adl).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(full_adl_model.summary())

                              OLS Regression Results                             
Dep. Variable:     MSE_Credit_Growth_YoY   R-squared:                       0.732
Model:                               OLS   Adj. R-squared:                  0.655
Method:                    Least Squares   F-statistic:                     21.16
Date:                   Tue, 23 Jun 2026   Prob (F-statistic):           6.52e-13
Time:                           12:55:46   Log-Likelihood:                -133.41
No. Observations:                     50   AIC:                             290.8
Df Residuals:                         38   BIC:                             313.8
Df Model:                             11                                         
Covariance Type:                     HAC                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [97]:
full_adl_results = pd.DataFrame({
    "Variable": full_adl_model.params.index,
    "Coefficient": full_adl_model.params.values,
    "HAC Std. Error": full_adl_model.bse.values,
    "z-statistic": full_adl_model.tvalues.values,
    "P-value": full_adl_model.pvalues.values,
    "95% CI Lower": full_adl_model.conf_int().iloc[:, 0].values,
    "95% CI Upper": full_adl_model.conf_int().iloc[:, 1].values
})

full_adl_results["Significance"] = full_adl_results["P-value"].apply(
    lambda p: "***" if p < 0.01
    else "**" if p < 0.05
    else "*" if p < 0.10
    else ""
)

full_adl_results.round(4)

,Variable,Coefficient,HAC Std. Error,z-statistic,P-value,95% CI Lower,95% CI Upper,Significance
0,const,1.3237,18.1996,0.0727,0.9420,-34.3469,36.9943,
1,MSE_Credit_Growth_YoY_L1,0.6219,0.0839,7.4142,0.0000,0.4575,0.7863,***
2,Credit_Deposit_Ratio_L0,1.0101,0.3701,2.7292,0.0063,0.2847,1.7354,***
3,Credit_Deposit_Ratio_L1,-0.9099,0.4173,-2.1805,0.0292,-1.7279,-0.0920,**
4,GNPA_Ratio_L0,-2.3672,1.3275,-1.7833,0.0745,-4.9690,0.2345,*
5,GNPA_Ratio_L1,1.8097,1.2688,1.4263,0.1538,-0.6771,4.2964,
6,PCR_L0,0.1121,0.0460,2.4372,0.0148,0.0220,0.2023,**
7,PCR_L1,-0.1186,0.0728,-1.6306,0.1030,-0.2612,0.0240,
8,d_WALR_L0,-3.3734,2.4288,-1.3889,0.1649,-8.1338,1.3870,
9,d_WALR_L1,0.7528,2.7678,0.2720,0.7856,-4.6721,6.1776,


In [98]:
pcr_joint_test = full_adl_model.wald_test(
    "PCR_L0 = 0, PCR_L1 = 0"
)

pcr_joint_results = pd.DataFrame({
    "Test": ["PCR current + lagged effect jointly equals zero"],
    "Wald Statistic": [float(pcr_joint_test.statistic)],
    "P-value": [float(pcr_joint_test.pvalue)]
})

pcr_joint_results["Decision"] = pcr_joint_results["P-value"].apply(
    lambda p: "Significant at 1%" if p < 0.01
    else "Significant at 5%" if p < 0.05
    else "Significant at 10%" if p < 0.10
    else "Not significant"
)

pcr_joint_results.round(4)

C:\Users\JAYPAL SINGH\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\base\model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\3507513400.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "Wald Statistic": [float(pcr_joint_test.statistic)],


,Test,Wald Statistic,P-value,Decision
0,PCR current + lagged effect jointly equals zero,6.5754,0.0373,Significant at 5%


In [99]:
full_joint_lag_results = []

for variable in full_predictors:
    
    restriction = f"{variable}_L0 = 0, {variable}_L1 = 0"
    
    test = full_adl_model.wald_test(restriction)
    
    full_joint_lag_results.append({
        "Variable": variable,
        "Wald Statistic": float(test.statistic),
        "P-value": float(test.pvalue)
    })

full_joint_lag_results_df = pd.DataFrame(full_joint_lag_results)

full_joint_lag_results_df["Significance"] = full_joint_lag_results_df["P-value"].apply(
    lambda p: "*** 1%" if p < 0.01
    else "** 5%" if p < 0.05
    else "* 10%" if p < 0.10
    else "Not significant"
)

full_joint_lag_results_df.round(4)

C:\Users\JAYPAL SINGH\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\base\model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\70480022.py:11: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "Wald Statistic": float(test.statistic),


,Variable,Wald Statistic,P-value,Significance
0,Credit_Deposit_Ratio,7.4570,0.0240,** 5%
1,GNPA_Ratio,7.3396,0.0255,** 5%
2,PCR,6.5754,0.0373,** 5%
3,d_WALR,1.9571,0.3759,Not significant
4,Cash_Reserve_Ratio,0.7877,0.6745,Not significant


In [100]:
full_vif_vars = [
    "Credit_Deposit_Ratio_L0",
    "Credit_Deposit_Ratio_L1",
    "GNPA_Ratio_L0",
    "GNPA_Ratio_L1",
    "PCR_L0",
    "PCR_L1",
    "d_WALR_L0",
    "d_WALR_L1",
    "Cash_Reserve_Ratio_L0",
    "Cash_Reserve_Ratio_L1"
]

X_full_vif = sm.add_constant(full_adl_df[full_vif_vars])

full_vif_results = pd.DataFrame({
    "Variable": full_vif_vars,
    "VIF": [
        variance_inflation_factor(X_full_vif.values, i + 1)
        for i in range(len(full_vif_vars))
    ]
})

full_vif_results["Tolerance"] = 1 / full_vif_results["VIF"]

full_vif_results.sort_values("VIF", ascending=False).reset_index(drop=True)

,Variable,VIF,Tolerance
0,GNPA_Ratio_L0,67.799271,0.014749
1,GNPA_Ratio_L1,58.904509,0.016977
2,Credit_Deposit_Ratio_L0,5.723370,0.174722
3,PCR_L1,5.635977,0.177432
4,PCR_L0,5.629563,0.177634
5,Credit_Deposit_Ratio_L1,4.931373,0.202783
6,Cash_Reserve_Ratio_L1,4.395800,0.227490
7,Cash_Reserve_Ratio_L0,4.159181,0.240432
8,d_WALR_L1,1.568846,0.637411
9,d_WALR_L0,1.484362,0.673690


In [101]:
full_bg_test = acorr_breusch_godfrey(
    full_adl_model,
    nlags=4
)

full_bg_results = pd.DataFrame({
    "Test": [
        "Breusch-Godfrey LM Statistic",
        "Breusch-Godfrey LM p-value",
        "F Statistic",
        "F p-value"
    ],
    "Value": full_bg_test
})

full_bg_results

,Test,Value
0,Breusch-Godfrey LM Statistic,8.918499
1,Breusch-Godfrey LM p-value,0.063169
2,F Statistic,1.845289
3,F p-value,0.142905


In [102]:
same_sample_core_df = full_adl_base[
    [
        "Quarter_End",
        "MSE_Credit_Growth_YoY",
        "Credit_Deposit_Ratio",
        "GNPA_Ratio",
        "d_WALR",
        "Cash_Reserve_Ratio"
    ]
].copy()

same_core_df, same_core_y, same_core_x = create_adl_dataset(
    df=same_sample_core_df,
    dependent="MSE_Credit_Growth_YoY",
    predictors=[
        "Credit_Deposit_Ratio",
        "GNPA_Ratio",
        "d_WALR",
        "Cash_Reserve_Ratio"
    ],
    y_lags=1,
    x_lags=1
)

same_core_model = sm.OLS(
    same_core_df[same_core_y],
    sm.add_constant(same_core_df[same_core_x])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

model_comparison = pd.DataFrame({
    "Model": [
        "Core ADL(1,1): no PCR",
        "Full ADL(1,1): includes PCR"
    ],
    "Observations": [
        int(same_core_model.nobs),
        int(full_adl_model.nobs)
    ],
    "Adjusted R-squared": [
        same_core_model.rsquared_adj,
        full_adl_model.rsquared_adj
    ],
    "AIC": [
        same_core_model.aic,
        full_adl_model.aic
    ],
    "BIC": [
        same_core_model.bic,
        full_adl_model.bic
    ]
})

model_comparison.round(3)

,Model,Observations,Adjusted R-squared,AIC,BIC
0,"Core ADL(1,1): no PCR",50,0.660,288.628,307.748
1,"Full ADL(1,1): includes PCR",50,0.655,290.828,313.773


## Pure PCR excluding the GNPA

In [103]:
# ── PCR replacement ADL(1,1): exclude GNPA ───────────────────────────────────

pcr_only_predictors = [
    "Credit_Deposit_Ratio",
    "PCR",
    "d_WALR",
    "Cash_Reserve_Ratio"
]

pcr_adl_df, pcr_y_col, pcr_x_cols = create_adl_dataset(
    df=full_adl_base,
    dependent="MSE_Credit_Growth_YoY",
    predictors=pcr_only_predictors,
    y_lags=1,
    x_lags=1
)

pcr_adl_model = sm.OLS(
    pcr_adl_df[pcr_y_col],
    sm.add_constant(pcr_adl_df[pcr_x_cols])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(pcr_adl_model.summary())

                              OLS Regression Results                             
Dep. Variable:     MSE_Credit_Growth_YoY   R-squared:                       0.715
Model:                               OLS   Adj. R-squared:                  0.651
Method:                    Least Squares   F-statistic:                     24.07
Date:                   Tue, 23 Jun 2026   Prob (F-statistic):           1.82e-13
Time:                           12:55:46   Log-Likelihood:                -135.02
No. Observations:                     50   AIC:                             290.0
Df Residuals:                         40   BIC:                             309.2
Df Model:                              9                                         
Covariance Type:                     HAC                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [104]:
model_comparison_final = pd.DataFrame({
    "Model": [
        "Core ADL(1,1): GNPA",
        "Full ADL(1,1): GNPA + PCR",
        "Alternative ADL(1,1): PCR"
    ],
    "Observations": [
        int(same_core_model.nobs),
        int(full_adl_model.nobs),
        int(pcr_adl_model.nobs)
    ],
    "Adjusted R-squared": [
        same_core_model.rsquared_adj,
        full_adl_model.rsquared_adj,
        pcr_adl_model.rsquared_adj
    ],
    "AIC": [
        same_core_model.aic,
        full_adl_model.aic,
        pcr_adl_model.aic
    ],
    "BIC": [
        same_core_model.bic,
        full_adl_model.bic,
        pcr_adl_model.bic
    ]
})

model_comparison_final.round(3)

,Model,Observations,Adjusted R-squared,AIC,BIC
0,"Core ADL(1,1): GNPA",50,0.660,288.628,307.748
1,"Full ADL(1,1): GNPA + PCR",50,0.655,290.828,313.773
2,"Alternative ADL(1,1): PCR",50,0.651,290.034,309.154


### The GNPA-based ADL(1,1) model is retained as the preferred specification because it provides the best fit and information criteria while maintaining a stable multicollinearity profile. PCR is retained as an alternative robustness indicator rather than being jointly included with GNPA, since the combined model generates severe collinearity between current and lagged asset-quality measures.

# Add final residual diagnostics for the preferred core ADL model

In [105]:
from statsmodels.stats.diagnostic import (
    het_breuschpagan,
    het_white,
    linear_reset,
    acorr_ljungbox
)
from statsmodels.stats.stattools import jarque_bera
from statsmodels.stats.outliers_influence import OLSInfluence
import scipy.stats as stats
import numpy as np
import plotly.express as px
import plotly.graph_objects as go

In [106]:
# ── Define consistent names for the preferred Core ADL(1,1) model ───────────

core_model = adl_model
core_adl_df = adl_df
core_y_col = adl_y_col
core_x_cols = adl_x_cols

print("Core model and dataset aliases are ready.")
print("Observations:", int(core_model.nobs))
print("Regressors:")
print(core_x_cols)

Core model and dataset aliases are ready.
Observations: 50
Regressors:
['MSE_Credit_Growth_YoY_L1', 'Credit_Deposit_Ratio_L0', 'Credit_Deposit_Ratio_L1', 'GNPA_Ratio_L0', 'GNPA_Ratio_L1', 'd_WALR_L0', 'd_WALR_L1', 'Cash_Reserve_Ratio_L0', 'Cash_Reserve_Ratio_L1']


In [107]:
# Breusch-Pagan and White tests for the preferred core ADL model

bp_test = het_breuschpagan(
    core_model.resid,
    core_model.model.exog
)

white_test = het_white(
    core_model.resid,
    core_model.model.exog
)

heteroskedasticity_results = pd.DataFrame({
    "Test": [
        "Breusch-Pagan LM Statistic",
        "Breusch-Pagan LM p-value",
        "Breusch-Pagan F Statistic",
        "Breusch-Pagan F p-value",
        "White LM Statistic",
        "White LM p-value",
        "White F Statistic",
        "White F p-value"
    ],
    "Value": [
        bp_test[0], bp_test[1], bp_test[2], bp_test[3],
        white_test[0], white_test[1], white_test[2], white_test[3]
    ]
})

heteroskedasticity_results

,Test,Value
0,Breusch-Pagan LM Statistic,19.989405
1,Breusch-Pagan LM p-value,0.017978
2,Breusch-Pagan F Statistic,2.960348
3,Breusch-Pagan F p-value,0.008592
4,White LM Statistic,50.000000
5,White LM p-value,0.433437
6,White F Statistic,NaN
7,White F p-value,NaN


In [108]:
jb_stat, jb_pvalue, skewness, kurtosis = jarque_bera(core_model.resid)

normality_results = pd.DataFrame({
    "Metric": [
        "Jarque-Bera Statistic",
        "Jarque-Bera p-value",
        "Residual Skewness",
        "Residual Kurtosis"
    ],
    "Value": [
        jb_stat,
        jb_pvalue,
        skewness,
        kurtosis
    ]
})

normality_results

,Metric,Value
0,Jarque-Bera Statistic,0.721109
1,Jarque-Bera p-value,0.697289
2,Residual Skewness,-0.017537
3,Residual Kurtosis,2.412716


In [109]:
reset_test = linear_reset(
    core_model,
    power=2,
    use_f=True
)

reset_results = pd.DataFrame({
    "Metric": [
        "RESET F Statistic",
        "RESET p-value"
    ],
    "Value": [
        float(reset_test.statistic),
        float(reset_test.pvalue)
    ]
})

reset_results

,Metric,Value
0,RESET F Statistic,0.439460
1,RESET p-value,0.511282


In [110]:
ljung_box_results = acorr_ljungbox(
    core_model.resid,
    lags=[1, 2, 4, 6],
    return_df=True
)

ljung_box_results

,lb_stat,lb_pvalue
1,0.103804,0.747312
2,0.371614,0.830434
4,6.049380,0.195491
6,6.995329,0.321279


In [111]:
influence = OLSInfluence(core_model)

influence_df = pd.DataFrame({
    "Quarter_End": core_adl_df["Quarter_End"].values,
    "Residual": core_model.resid,
    "Standardized Residual": influence.resid_studentized_internal,
    "Leverage": influence.hat_matrix_diag,
    "Cooks_Distance": influence.cooks_distance[0]
})

cook_threshold = 4 / len(influence_df)

influential_points = (
    influence_df[
        influence_df["Cooks_Distance"] > cook_threshold
    ]
    .sort_values("Cooks_Distance", ascending=False)
)

print("Cook's Distance threshold:", round(cook_threshold, 4))

influential_points

Cook's Distance threshold: 0.08


,Quarter_End,Residual,Standardized Residual,Leverage,Cooks_Distance
30,2021-03-31,-8.088480,-2.393512,0.275635,0.217996
49,2025-12-31,6.137126,1.898160,0.336931,0.183083
37,2022-12-31,-7.191305,-2.072749,0.236487,0.133071
29,2020-12-31,-5.997029,-1.782180,0.281770,0.124604
47,2025-06-30,5.473491,1.647678,0.300031,0.116368
28,2020-09-30,6.125291,1.769529,0.239966,0.098863
35,2022-06-30,6.375932,1.797598,0.202011,0.081802
33,2021-12-31,5.488228,1.588223,0.242581,0.080788


In [112]:
fig = px.bar(
    influence_df,
    x="Quarter_End",
    y="Cooks_Distance",
    title="Core ADL(1,1): Cook's Distance by Quarter"
)

fig.add_hline(
    y=cook_threshold,
    line_dash="dash",
    annotation_text="4 / N threshold",
    annotation_position="top left"
)

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=500,
    xaxis_title="Quarter",
    yaxis_title="Cook's Distance",
    showlegend=False
)

fig.show()

In [113]:
residual_df = pd.DataFrame({
    "Quarter_End": core_adl_df["Quarter_End"].values,
    "Residual": core_model.resid
})

fig = px.line(
    residual_df,
    x="Quarter_End",
    y="Residual",
    markers=True,
    title="Core ADL(1,1): Residuals Over Time"
)

fig.add_hline(y=0, line_dash="dash")

fig.update_layout(
    template="plotly_white",
    width=1100,
    height=500,
    xaxis_title="Quarter",
    yaxis_title="Residual",
    showlegend=False
)

fig.show()

In [114]:
fig = px.histogram(
    residual_df,
    x="Residual",
    nbins=14,
    marginal="rug",
    title="Core ADL(1,1): Residual Distribution"
)

fig.update_layout(
    template="plotly_white",
    width=900,
    height=500,
    xaxis_title="Residual",
    yaxis_title="Frequency",
    showlegend=False
)

fig.show()

## COVID robustness model:
### Use a COVID-policy-period dummy from 2020 Q2 to 2021 Q4. This captures the unusual phase of loan moratoria, emergency liquidity support, credit guarantees, and disrupted SME credit conditions.

In [115]:
# ── Broad COVID / policy-distortion dummy: 2020Q1 to 2022Q4 ──────────────────

covid_adl_df = core_adl_df.copy()

covid_adl_df["COVID_Dummy"] = (
    (
        covid_adl_df["Quarter_End"] >= pd.Timestamp("2020-03-31")
    )
    &
    (
        covid_adl_df["Quarter_End"] <= pd.Timestamp("2022-12-31")
    )
).astype(int)

print(
    covid_adl_df["COVID_Dummy"]
    .value_counts()
    .sort_index()
)

covid_adl_df[
    ["Quarter_End", "COVID_Dummy"]
].tail(25)

COVID_Dummy
0    38
1    12
Name: count, dtype: int64


,Quarter_End,COVID_Dummy
25,2019-12-31,0
26,2020-03-31,1
27,2020-06-30,1
28,2020-09-30,1
29,2020-12-31,1
30,2021-03-31,1
31,2021-06-30,1
32,2021-09-30,1
33,2021-12-31,1
34,2022-03-31,1


In [116]:
covid_x_cols = core_x_cols + ["COVID_Dummy"]

y_covid = covid_adl_df[core_y_col]

X_covid = sm.add_constant(
    covid_adl_df[covid_x_cols]
)

core_covid_model = sm.OLS(
    y_covid,
    X_covid
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(core_covid_model.summary())

                              OLS Regression Results                             
Dep. Variable:     MSE_Credit_Growth_YoY   R-squared:                       0.725
Model:                               OLS   Adj. R-squared:                  0.655
Method:                    Least Squares   F-statistic:                     26.26
Date:                   Tue, 23 Jun 2026   Prob (F-statistic):           2.73e-14
Time:                           12:55:46   Log-Likelihood:                -134.10
No. Observations:                     50   AIC:                             290.2
Df Residuals:                         39   BIC:                             311.2
Df Model:                             10                                         
Covariance Type:                     HAC                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------

In [117]:
covid_results = pd.DataFrame({
    "Variable": core_covid_model.params.index,
    "Coefficient": core_covid_model.params.values,
    "HAC Std. Error": core_covid_model.bse.values,
    "z-statistic": core_covid_model.tvalues.values,
    "P-value": core_covid_model.pvalues.values,
    "95% CI Lower": core_covid_model.conf_int().iloc[:, 0].values,
    "95% CI Upper": core_covid_model.conf_int().iloc[:, 1].values
})

covid_results["Significance"] = covid_results["P-value"].apply(
    lambda p: "***" if p < 0.01
    else "**" if p < 0.05
    else "*" if p < 0.10
    else ""
)

covid_results.round(4)

,Variable,Coefficient,HAC Std. Error,z-statistic,P-value,95% CI Lower,95% CI Upper,Significance
0,const,15.3135,25.9116,0.5910,0.5545,-35.4722,66.0992,
1,MSE_Credit_Growth_YoY_L1,0.6152,0.1004,6.1286,0.0000,0.4184,0.8119,***
2,Credit_Deposit_Ratio_L0,0.9901,0.4243,2.3336,0.0196,0.1585,1.8217,**
3,Credit_Deposit_Ratio_L1,-1.0427,0.3956,-2.6358,0.0084,-1.8180,-0.2674,***
4,GNPA_Ratio_L0,-2.9337,1.0986,-2.6704,0.0076,-5.0869,-0.7805,***
5,GNPA_Ratio_L1,2.3414,1.1125,2.1046,0.0353,0.1609,4.5218,**
6,d_WALR_L0,-3.4387,2.2788,-1.5090,0.1313,-7.9051,1.0276,
7,d_WALR_L1,0.6120,3.1021,0.1973,0.8436,-5.4679,6.6920,
8,Cash_Reserve_Ratio_L0,1.0427,2.5136,0.4148,0.6783,-3.8839,5.9693,
9,Cash_Reserve_Ratio_L1,-1.6677,2.2509,-0.7409,0.4587,-6.0795,2.7440,


In [118]:
key_terms = [
    "MSE_Credit_Growth_YoY_L1",
    "Credit_Deposit_Ratio_L0",
    "Credit_Deposit_Ratio_L1",
    "GNPA_Ratio_L0",
    "GNPA_Ratio_L1",
    "d_WALR_L0",
    "d_WALR_L1",
    "Cash_Reserve_Ratio_L0",
    "Cash_Reserve_Ratio_L1",
    "COVID_Dummy"
]

comparison_rows = []

for term in key_terms:
    comparison_rows.append({
        "Variable": term,
        "Baseline Coefficient": core_model.params.get(term, np.nan),
        "Baseline p-value": core_model.pvalues.get(term, np.nan),
        "COVID Model Coefficient": core_covid_model.params.get(term, np.nan),
        "COVID Model p-value": core_covid_model.pvalues.get(term, np.nan)
    })

covid_stability_table = pd.DataFrame(comparison_rows)

covid_stability_table.round(4)

,Variable,Baseline Coefficient,Baseline p-value,COVID Model Coefficient,COVID Model p-value
0,MSE_Credit_Growth_YoY_L1,0.6115,0.0000,0.6152,0.0000
1,Credit_Deposit_Ratio_L0,1.0800,0.0026,0.9901,0.0196
2,Credit_Deposit_Ratio_L1,-0.9876,0.0167,-1.0427,0.0084
3,GNPA_Ratio_L0,-2.5359,0.0159,-2.9337,0.0076
4,GNPA_Ratio_L1,2.0134,0.0618,2.3414,0.0353
5,d_WALR_L0,-3.7705,0.1057,-3.4387,0.1313
6,d_WALR_L1,0.4115,0.8935,0.6120,0.8436
7,Cash_Reserve_Ratio_L0,1.4454,0.5316,1.0427,0.6783
8,Cash_Reserve_Ratio_L1,-1.3539,0.5325,-1.6677,0.4587
9,COVID_Dummy,NaN,NaN,-1.3968,0.4728


In [119]:
core_vs_covid = pd.DataFrame({
    "Model": [
        "Core ADL(1,1)",
        "Core ADL(1,1) + COVID-policy dummy (2020Q1–2022Q4)"
    ],
    "Observations": [
        int(core_model.nobs),
        int(core_covid_model.nobs)
    ],
    "Adjusted R-squared": [
        core_model.rsquared_adj,
        core_covid_model.rsquared_adj
    ],
    "AIC": [
        core_model.aic,
        core_covid_model.aic
    ],
    "BIC": [
        core_model.bic,
        core_covid_model.bic
    ]
})

core_vs_covid.round(3)

,Model,Observations,Adjusted R-squared,AIC,BIC
0,"Core ADL(1,1)",50,0.660,288.628,307.748
1,"Core ADL(1,1) + COVID-policy dummy (2020Q1–202...",50,0.655,290.193,311.225


### The inclusion of a broad COVID-policy dummy covering 2020Q1–2022Q4 does not improve model fit and is statistically insignificant. The signs, magnitudes, and significance of the key dynamic coefficients remain materially unchanged, particularly for the contemporaneous negative effect of GNPA on MSE credit growth. Therefore, the main asset-quality and funding-pressure findings are robust to the extraordinary COVID and policy-support period.

# Structural Break / Regime analysis

In [120]:
# ── Regime-analysis dataset ──────────────────────────────────────────────────

# Convert monthly Repo and Bank Rate series to quarter-end values
repo_regime_q = monthly_to_quarter_end(
    repo,
    "Month",
    "Repo_rate"
)

bank_rate_regime_q = monthly_to_quarter_end(
    bank_rate,
    "Month",
    "Bank_rate"
)

repo_regime_q["Quarter_End"] = (
    repo_regime_q["Month"]
    .dt.to_period("Q")
    .dt.end_time
    .dt.normalize()
)

bank_rate_regime_q["Quarter_End"] = (
    bank_rate_regime_q["Month"]
    .dt.to_period("Q")
    .dt.end_time
    .dt.normalize()
)

regime_df = (
    quarterly_df[
        [
            "Quarter_End",
            "MSE_Credit_Growth_YoY",
            "Credit_Deposit_Ratio",
            "GNPA_Ratio",
            "WALR"
        ]
    ]
    .merge(
        repo_regime_q[["Quarter_End", "Repo_rate"]],
        on="Quarter_End",
        how="inner"
    )
    .merge(
        bank_rate_regime_q[["Quarter_End", "Bank_rate"]],
        on="Quarter_End",
        how="inner"
    )
    .sort_values("Quarter_End")
    .reset_index(drop=True)
)

regime_df.head()

,Quarter_End,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,WALR,Repo_rate,Bank_rate
0,2013-03-31,12.769316,77.93,3.422945,11.46,7.50,8.50
1,2013-06-30,22.630200,76.42,4.004487,11.40,7.25,8.25
2,2013-09-30,21.847790,78.35,4.218281,11.96,7.50,9.50
3,2013-12-31,22.894464,76.81,4.397677,11.70,7.75,8.75
4,2014-03-31,25.879074,77.79,4.115068,11.57,8.00,9.00


In [121]:
# ── Regime labels ────────────────────────────────────────────────────────────

def assign_regime(date):
    
    if date <= pd.Timestamp("2019-12-31"):
        return "Pre-COVID / NPA Cleanup"
    
    elif date <= pd.Timestamp("2022-12-31"):
        return "COVID Support Phase"
    
    elif date <= pd.Timestamp("2024-12-31"):
        return "Post-COVID Tightening"
    
    else:
        return "Recent Policy Cycle"

regime_df["Regime"] = regime_df["Quarter_End"].apply(assign_regime)

regime_df["Regime"] = pd.Categorical(
    regime_df["Regime"],
    categories=[
        "Pre-COVID / NPA Cleanup",
        "COVID Support Phase",
        "Post-COVID Tightening",
        "Recent Policy Cycle"
    ],
    ordered=True
)

regime_df[
    ["Quarter_End", "Regime"]
].tail(15)

,Quarter_End,Regime
37,2022-06-30,COVID Support Phase
38,2022-09-30,COVID Support Phase
39,2022-12-31,COVID Support Phase
40,2023-03-31,Post-COVID Tightening
41,2023-06-30,Post-COVID Tightening
42,2023-09-30,Post-COVID Tightening
43,2023-12-31,Post-COVID Tightening
44,2024-03-31,Post-COVID Tightening
45,2024-06-30,Post-COVID Tightening
46,2024-09-30,Post-COVID Tightening


In [122]:
# ── Lending-rate transmission variables ──────────────────────────────────────

regime_df["WALR_Repo_Spread"] = regime_df["WALR"] - regime_df["Repo_rate"]

regime_df["WALR_BankRate_Spread"] = (
    regime_df["WALR"] - regime_df["Bank_rate"]
)

regime_df["d_Repo"] = regime_df["Repo_rate"].diff()

regime_df["d_Bank_Rate"] = regime_df["Bank_rate"].diff()

regime_df["d_WALR_Regime"] = regime_df["WALR"].diff()

regime_df.head()

,Quarter_End,MSE_Credit_Growth_YoY,Credit_Deposit_Ratio,GNPA_Ratio,WALR,Repo_rate,Bank_rate,Regime,WALR_Repo_Spread,WALR_BankRate_Spread,d_Repo,d_Bank_Rate,d_WALR_Regime
0,2013-03-31,12.769316,77.93,3.422945,11.46,7.50,8.50,Pre-COVID / NPA Cleanup,3.96,2.96,NaN,NaN,NaN
1,2013-06-30,22.630200,76.42,4.004487,11.40,7.25,8.25,Pre-COVID / NPA Cleanup,4.15,3.15,-0.25,-0.25,-0.06
2,2013-09-30,21.847790,78.35,4.218281,11.96,7.50,9.50,Pre-COVID / NPA Cleanup,4.46,2.46,0.25,1.25,0.56
3,2013-12-31,22.894464,76.81,4.397677,11.70,7.75,8.75,Pre-COVID / NPA Cleanup,3.95,2.95,0.25,-0.75,-0.26
4,2014-03-31,25.879074,77.79,4.115068,11.57,8.00,9.00,Pre-COVID / NPA Cleanup,3.57,2.57,0.25,0.25,-0.13


In [123]:
regime_summary = (
    regime_df
    .groupby("Regime", observed=True)
    .agg(
        Start=("Quarter_End", "min"),
        End=("Quarter_End", "max"),
        Quarters=("Quarter_End", "count"),
        Avg_MSE_Credit_Growth=("MSE_Credit_Growth_YoY", "mean"),
        Avg_GNPA=("GNPA_Ratio", "mean"),
        Avg_CDR=("Credit_Deposit_Ratio", "mean"),
        Avg_Repo=("Repo_rate", "mean"),
        Avg_Bank_Rate=("Bank_rate", "mean"),
        Avg_WALR=("WALR", "mean"),
        Avg_WALR_Repo_Spread=("WALR_Repo_Spread", "mean")
    )
    .reset_index()
)

regime_summary.round(2)

,Regime,Start,End,Quarters,Avg_MSE_Credit_Growth,Avg_GNPA,Avg_CDR,Avg_Repo,Avg_Bank_Rate,Avg_WALR,Avg_WALR_Repo_Spread
0,Pre-COVID / NPA Cleanup,2013-03-31,2019-12-31,28,11.61,7.53,75.87,6.73,7.40,10.43,3.69
1,COVID Support Phase,2020-03-31,2022-12-31,12,12.15,6.65,72.84,4.45,4.70,8.13,3.67
2,Post-COVID Tightening,2023-03-31,2024-12-31,8,16.50,3.07,78.57,6.50,6.75,9.32,2.82
3,Recent Policy Cycle,2025-03-31,2025-12-31,4,21.53,2.19,80.43,5.62,5.88,8.66,3.03


In [126]:
import plotly.graph_objects as go

regime_bounds = (
    regime_df
    .groupby("Regime", observed=True)["Quarter_End"]
    .agg(["min", "max"])
    .reset_index()
)

def add_regime_shading(fig):
    
    for _, row in regime_bounds.iterrows():
        
        fig.add_vrect(
            x0=row["min"],
            x1=row["max"],
            fillcolor="lightgrey",
            opacity=0.12,
            line_width=0,
            annotation_text=str(row["Regime"]),
            annotation_position="top left"
        )
    
    return fig

In [127]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=regime_df["Quarter_End"],
        y=regime_df["GNPA_Ratio"],
        mode="lines+markers",
        name="GNPA Ratio (%)",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=regime_df["Quarter_End"],
        y=regime_df["MSE_Credit_Growth_YoY"],
        mode="lines+markers",
        name="MSE Credit Growth YoY (%)",
        yaxis="y2"
    )
)

fig = add_regime_shading(fig)

fig.update_layout(
    template="plotly_white",
    width=1150,
    height=560,
    title="Asset Quality and MSE Credit Growth Across Banking Regimes",
    xaxis=dict(title="Quarter"),
    yaxis=dict(
        title="GNPA Ratio (%)"
    ),
    yaxis2=dict(
        title="MSE Credit Growth YoY (%)",
        overlaying="y",
        side="right"
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="left",
        x=0
    )
)

fig.show()

In [128]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=regime_df["Quarter_End"],
        y=regime_df["Credit_Deposit_Ratio"],
        mode="lines+markers",
        name="Credit-Deposit Ratio (%)",
        yaxis="y1"
    )
)

fig.add_trace(
    go.Scatter(
        x=regime_df["Quarter_End"],
        y=regime_df["MSE_Credit_Growth_YoY"],
        mode="lines+markers",
        name="MSE Credit Growth YoY (%)",
        yaxis="y2"
    )
)

fig = add_regime_shading(fig)

fig.update_layout(
    template="plotly_white",
    width=1150,
    height=560,
    title="Funding Pressure and MSE Credit Growth Across Banking Regimes",
    xaxis=dict(title="Quarter"),
    yaxis=dict(
        title="Credit-Deposit Ratio (%)"
    ),
    yaxis2=dict(
        title="MSE Credit Growth YoY (%)",
        overlaying="y",
        side="right"
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="left",
        x=0
    )
)

fig.show()

In [129]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=regime_df["Quarter_End"],
        y=regime_df["Repo_rate"],
        mode="lines+markers",
        name="Repo Rate"
    )
)

fig.add_trace(
    go.Scatter(
        x=regime_df["Quarter_End"],
        y=regime_df["Bank_rate"],
        mode="lines+markers",
        name="Bank Rate"
    )
)

fig.add_trace(
    go.Scatter(
        x=regime_df["Quarter_End"],
        y=regime_df["WALR"],
        mode="lines+markers",
        name="WALR"
    )
)

fig = add_regime_shading(fig)

fig.update_layout(
    template="plotly_white",
    width=1150,
    height=560,
    title="Policy-Rate and Lending-Rate Transmission Across Regimes",
    xaxis_title="Quarter",
    yaxis_title="Interest Rate (%)",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="left",
        x=0
    )
)

fig.show()

In [130]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=regime_df["Quarter_End"],
        y=regime_df["WALR_Repo_Spread"],
        mode="lines+markers",
        name="WALR − Repo Spread"
    )
)

fig.add_trace(
    go.Scatter(
        x=regime_df["Quarter_End"],
        y=regime_df["WALR_BankRate_Spread"],
        mode="lines+markers",
        name="WALR − Bank Rate Spread"
    )
)

fig.add_hline(
    y=0,
    line_dash="dash"
)

fig = add_regime_shading(fig)

fig.update_layout(
    template="plotly_white",
    width=1150,
    height=560,
    title="Lending-Rate Spreads Across Policy Regimes",
    xaxis_title="Quarter",
    yaxis_title="Spread (percentage points)",
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="left",
        x=0
    )
)

fig.show()

In [131]:
# ── Two-regime framework ─────────────────────────────────────────────────────

def assign_regime(date):
    if date <= pd.Timestamp("2019-12-31"):
        return "Pre-COVID / NPA Cleanup"
    return "COVID and Post-COVID Cycle"

regime_df["Regime"] = regime_df["Quarter_End"].apply(assign_regime)

regime_df["Regime"] = pd.Categorical(
    regime_df["Regime"],
    categories=[
        "Pre-COVID / NPA Cleanup",
        "COVID and Post-COVID Cycle"
    ],
    ordered=True
)

regime_df["Regime"].value_counts(sort=False)

Regime
Pre-COVID / NPA Cleanup       28
COVID and Post-COVID Cycle    24
Name: count, dtype: int64

In [132]:
# ── Monetary transmission dataset ────────────────────────────────────────────

transmission_df = regime_df[
    [
        "Quarter_End",
        "Regime",
        "Repo_rate",
        "Bank_rate",
        "WALR"
    ]
].copy()

transmission_df = transmission_df.sort_values("Quarter_End").reset_index(drop=True)

# Quarterly changes in percentage points
transmission_df["d_WALR"] = transmission_df["WALR"].diff()
transmission_df["d_Repo"] = transmission_df["Repo_rate"].diff()
transmission_df["d_Bank_Rate"] = transmission_df["Bank_rate"].diff()

# One-quarter lags
transmission_df["d_Repo_L1"] = transmission_df["d_Repo"].shift(1)
transmission_df["d_Bank_Rate_L1"] = transmission_df["d_Bank_Rate"].shift(1)

# Regime dummy: 1 from 2020Q1 onward
transmission_df["Post_COVID"] = (
    transmission_df["Quarter_End"] >= pd.Timestamp("2020-03-31")
).astype(int)

transmission_df = transmission_df.dropna().reset_index(drop=True)

print("Sample start:", transmission_df["Quarter_End"].min().date())
print("Sample end  :", transmission_df["Quarter_End"].max().date())
print("Observations:", len(transmission_df))

transmission_df.head()

Sample start: 2013-09-30
Sample end  : 2025-12-31
Observations: 50


,Quarter_End,Regime,Repo_rate,Bank_rate,WALR,d_WALR,d_Repo,d_Bank_Rate,d_Repo_L1,d_Bank_Rate_L1,Post_COVID
0,2013-09-30,Pre-COVID / NPA Cleanup,7.50,9.50,11.96,0.56,0.25,1.25,-0.25,-0.25,0
1,2013-12-31,Pre-COVID / NPA Cleanup,7.75,8.75,11.70,-0.26,0.25,-0.75,0.25,1.25,0
2,2014-03-31,Pre-COVID / NPA Cleanup,8.00,9.00,11.57,-0.13,0.25,0.25,0.25,-0.75,0
3,2014-06-30,Pre-COVID / NPA Cleanup,8.00,9.00,11.60,0.03,0.00,0.00,0.25,0.25,0
4,2014-09-30,Pre-COVID / NPA Cleanup,8.00,9.00,11.52,-0.08,0.00,0.00,0.00,0.00,0


In [134]:
fig = go.Figure()

fig.add_trace(
    go.Bar(
        x=transmission_df["Quarter_End"],
        y=transmission_df["d_Repo"],
        name="Change in Repo Rate"
    )
)

fig.add_trace(
    go.Scatter(
        x=transmission_df["Quarter_End"],
        y=transmission_df["d_WALR"],
        mode="lines+markers",
        name="Change in WALR",
        yaxis="y2"
    )
)

fig.add_shape(
    type="line",
    x0="2020-03-31",
    x1="2020-03-31",
    y0=0,
    y1=1,
    xref="x",
    yref="paper",
    line=dict(dash="dash", width=1)
)

fig.add_annotation(
    x="2020-03-31",
    y=1,
    xref="x",
    yref="paper",
    text="Post-COVID regime begins",
    showarrow=False,
    xanchor="left",
    yanchor="bottom"
)

fig.update_layout(
    template="plotly_white",
    width=1150,
    height=540,
    title="Quarterly Change in Repo Rate and WALR",
    xaxis_title="Quarter",
    yaxis=dict(title="Change in Repo Rate (percentage points)"),
    yaxis2=dict(
        title="Change in WALR (percentage points)",
        overlaying="y",
        side="right"
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="left",
        x=0
    )
)

fig.show()

In [135]:
# ── Repo-rate pass-through with regime interaction ───────────────────────────

repo_pass_df = transmission_df.copy()

repo_pass_df["d_Repo_x_PostCOVID"] = (
    repo_pass_df["d_Repo"] * repo_pass_df["Post_COVID"]
)

repo_pass_df["d_Repo_L1_x_PostCOVID"] = (
    repo_pass_df["d_Repo_L1"] * repo_pass_df["Post_COVID"]
)

repo_pass_vars = [
    "d_Repo",
    "d_Repo_L1",
    "Post_COVID",
    "d_Repo_x_PostCOVID",
    "d_Repo_L1_x_PostCOVID"
]

y_repo = repo_pass_df["d_WALR"]
X_repo = sm.add_constant(repo_pass_df[repo_pass_vars])

repo_pass_model = sm.OLS(
    y_repo,
    X_repo
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(repo_pass_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_WALR   R-squared:                       0.608
Model:                            OLS   Adj. R-squared:                  0.564
Method:                 Least Squares   F-statistic:                     84.52
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           2.01e-21
Time:                        13:06:08   Log-Likelihood:                 22.398
No. Observations:                  50   AIC:                            -32.80
Df Residuals:                      44   BIC:                            -21.32
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                            coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------
const                    -0.05

In [136]:
repo_pass_results = pd.DataFrame({
    "Variable": repo_pass_model.params.index,
    "Coefficient": repo_pass_model.params.values,
    "HAC Std. Error": repo_pass_model.bse.values,
    "z-statistic": repo_pass_model.tvalues.values,
    "P-value": repo_pass_model.pvalues.values,
    "95% CI Lower": repo_pass_model.conf_int().iloc[:, 0].values,
    "95% CI Upper": repo_pass_model.conf_int().iloc[:, 1].values
})

repo_pass_results["Significance"] = repo_pass_results["P-value"].apply(
    lambda p: "***" if p < 0.01
    else "**" if p < 0.05
    else "*" if p < 0.10
    else ""
)

repo_pass_results.round(4)

,Variable,Coefficient,HAC Std. Error,z-statistic,P-value,95% CI Lower,95% CI Upper,Significance
0,const,-0.0557,0.0377,-1.4766,0.1398,-0.1296,0.0182,
1,d_Repo,0.4097,0.1659,2.4703,0.0135,0.0847,0.7348,**
2,d_Repo_L1,-0.0657,0.2206,-0.2981,0.7656,-0.4981,0.3666,
3,Post_COVID,0.0130,0.0429,0.3027,0.7621,-0.0711,0.0971,
4,d_Repo_x_PostCOVID,0.1143,0.1807,0.6325,0.5271,-0.2398,0.4684,
5,d_Repo_L1_x_PostCOVID,0.3018,0.2244,1.3450,0.1786,-0.1380,0.7416,


In [137]:
# ── Cumulative Repo-to-WALR pass-through ─────────────────────────────────────

pre_covid_pass_through = (
    repo_pass_model.params["d_Repo"]
    + repo_pass_model.params["d_Repo_L1"]
)

post_covid_pass_through = (
    repo_pass_model.params["d_Repo"]
    + repo_pass_model.params["d_Repo_L1"]
    + repo_pass_model.params["d_Repo_x_PostCOVID"]
    + repo_pass_model.params["d_Repo_L1_x_PostCOVID"]
)

repo_pass_through_summary = pd.DataFrame({
    "Regime": [
        "Pre-COVID / NPA Cleanup",
        "COVID and Post-COVID Cycle"
    ],
    "Cumulative Repo-to-WALR Pass-Through": [
        pre_covid_pass_through,
        post_covid_pass_through
    ]
})

repo_pass_through_summary.round(4)

,Regime,Cumulative Repo-to-WALR Pass-Through
0,Pre-COVID / NPA Cleanup,0.3440
1,COVID and Post-COVID Cycle,0.7601


In [138]:
# ── Test whether interaction terms jointly equal zero ─────────────────────────

repo_regime_test = repo_pass_model.wald_test(
    "d_Repo_x_PostCOVID = 0, d_Repo_L1_x_PostCOVID = 0"
)

repo_regime_test_results = pd.DataFrame({
    "Test": ["Repo-to-WALR pass-through changed after COVID"],
    "Wald Statistic": [float(repo_regime_test.statistic)],
    "P-value": [float(repo_regime_test.pvalue)]
})

repo_regime_test_results["Decision"] = repo_regime_test_results["P-value"].apply(
    lambda p: "Changed significantly at 1%" if p < 0.01
    else "Changed significantly at 5%" if p < 0.05
    else "Changed significantly at 10%" if p < 0.10
    else "No statistically clear regime change"
)

repo_regime_test_results.round(4)

C:\Users\JAYPAL SINGH\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\base\model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\2088908059.py:9: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "Wald Statistic": [float(repo_regime_test.statistic)],


,Test,Wald Statistic,P-value,Decision
0,Repo-to-WALR pass-through changed after COVID,8.4998,0.0143,Changed significantly at 5%


In [139]:
# ── Bank Rate pass-through: separate model ───────────────────────────────────

bank_pass_df = transmission_df.copy()

bank_pass_df["d_Bank_Rate_x_PostCOVID"] = (
    bank_pass_df["d_Bank_Rate"] * bank_pass_df["Post_COVID"]
)

bank_pass_df["d_Bank_Rate_L1_x_PostCOVID"] = (
    bank_pass_df["d_Bank_Rate_L1"] * bank_pass_df["Post_COVID"]
)

bank_pass_vars = [
    "d_Bank_Rate",
    "d_Bank_Rate_L1",
    "Post_COVID",
    "d_Bank_Rate_x_PostCOVID",
    "d_Bank_Rate_L1_x_PostCOVID"
]

y_bank = bank_pass_df["d_WALR"]
X_bank = sm.add_constant(bank_pass_df[bank_pass_vars])

bank_pass_model = sm.OLS(
    y_bank,
    X_bank
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(bank_pass_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_WALR   R-squared:                       0.714
Model:                            OLS   Adj. R-squared:                  0.681
Method:                 Least Squares   F-statistic:                     87.37
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           1.04e-21
Time:                        13:06:44   Log-Likelihood:                 30.275
No. Observations:                  50   AIC:                            -48.55
Df Residuals:                      44   BIC:                            -37.08
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                                 coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------------
const               

In [140]:
transmission_model_comparison = pd.DataFrame({
    "Model": [
        "Repo Rate → WALR",
        "Bank Rate → WALR"
    ],
    "Observations": [
        int(repo_pass_model.nobs),
        int(bank_pass_model.nobs)
    ],
    "Adjusted R-squared": [
        repo_pass_model.rsquared_adj,
        bank_pass_model.rsquared_adj
    ],
    "AIC": [
        repo_pass_model.aic,
        bank_pass_model.aic
    ],
    "BIC": [
        repo_pass_model.bic,
        bank_pass_model.bic
    ]
})

transmission_model_comparison.round(3)

,Model,Observations,Adjusted R-squared,AIC,BIC
0,Repo Rate → WALR,50,0.564,-32.796,-21.324
1,Bank Rate → WALR,50,0.681,-48.550,-37.078


##  So, within your quarterly data, changes in the Bank Rate track changes in effective lending rates more closely than changes in the Repo Rate. That does not mean Bank Rate is necessarily the operational policy instrument. It means that, empirically, Bank Rate changes provide the stronger reduced-form explanation of WALR movements in your sample.

In [141]:
bank_regime_test = bank_pass_model.wald_test(
    "d_Bank_Rate_x_PostCOVID = 0, d_Bank_Rate_L1_x_PostCOVID = 0"
)

bank_regime_test_results = pd.DataFrame({
    "Test": ["Bank Rate-to-WALR pass-through changed after COVID"],
    "Wald Statistic": [float(bank_regime_test.statistic)],
    "P-value": [float(bank_regime_test.pvalue)]
})

bank_regime_test_results["Decision"] = bank_regime_test_results["P-value"].apply(
    lambda p: "Changed significantly at 1%" if p < 0.01
    else "Changed significantly at 5%" if p < 0.05
    else "Changed significantly at 10%" if p < 0.10
    else "No statistically clear regime change"
)

bank_regime_test_results.round(4)

C:\Users\JAYPAL SINGH\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\statsmodels\base\model.py:1912: FutureWarning: The behavior of wald_test will change after 0.14 to returning scalar test statistic values. To get the future behavior now, set scalar to True. To silence this message while retaining the legacy behavior, set scalar to False.
  warnings.warn(
C:\Users\JAYPAL SINGH\AppData\Local\Temp\ipykernel_22904\519032727.py:7: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  "Wald Statistic": [float(bank_regime_test.statistic)],


,Test,Wald Statistic,P-value,Decision
0,Bank Rate-to-WALR pass-through changed after C...,11.2136,0.0037,Changed significantly at 1%


In [143]:
# ── Cumulative Bank Rate-to-WALR pass-through by regime ──────────────────────

pre_covid_bank_pass = (
    bank_pass_model.params["d_Bank_Rate"]
    + bank_pass_model.params["d_Bank_Rate_L1"]
)

post_covid_bank_pass = (
    bank_pass_model.params["d_Bank_Rate"]
    + bank_pass_model.params["d_Bank_Rate_L1"]
    + bank_pass_model.params["d_Bank_Rate_x_PostCOVID"]
    + bank_pass_model.params["d_Bank_Rate_L1_x_PostCOVID"]
)

bank_pass_through_summary = pd.DataFrame({
    "Regime": [
        "Pre-COVID / NPA Cleanup",
        "COVID and Post-COVID Cycle"
    ],
    "Cumulative Bank Rate-to-WALR Pass-Through": [
        pre_covid_bank_pass,
        post_covid_bank_pass
    ]
})

bank_pass_through_summary.round(4)

,Regime,Cumulative Bank Rate-to-WALR Pass-Through
0,Pre-COVID / NPA Cleanup,0.4914
1,COVID and Post-COVID Cycle,0.7601


#### The relationship between RBI policy settings and banks’ effective lending rates strengthened materially after 2020. This means that monetary-policy tightening or easing was transmitted more forcefully to borrowing costs faced by firms during the COVID and post-COVID policy cycle than during the pre-COVID NPA-cleanup period.

#### In the post-2020 regime, both the Repo-rate and Bank-rate specifications imply substantially stronger pass-through to WALR, estimated at around 0.76 over the current and subsequent quarter. The stronger Bank Rate model fit suggests that Bank Rate movements more closely track effective lending-rate adjustment in this sample, while Repo Rate remains the primary monetary-policy reference rate.

In [144]:
# Run once if needed
# !pip install yfinance requests

import pandas as pd
import numpy as np
import requests
import yfinance as yf

In [145]:
START_DATE = "2013-01-01"
END_DATE = "2026-06-30"

india_vix_daily = yf.download(
    "^INDIAVIX",
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    progress=False
)

# Handle yfinance MultiIndex output safely
if isinstance(india_vix_daily.columns, pd.MultiIndex):
    india_vix_daily.columns = india_vix_daily.columns.get_level_values(0)

india_vix_monthly = (
    india_vix_daily["Close"]
    .resample("ME")
    .mean()
    .rename("India_VIX_Avg")
    .reset_index()
)

india_vix_monthly.head()

,index,India_VIX_Avg
0,2013-01-31,13.870435
1,2013-02-28,15.503000
2,2013-03-31,14.998947
3,2013-04-30,15.635500
4,2013-05-31,17.149546


In [148]:
import pandas as pd
import requests

DBNOMICS_BASE = "https://api.db.nomics.world/v22"

def dbnomics_search(query, limit=50):
    """
    Search DBnomics catalogue for matching series.
    """

    url = f"{DBNOMICS_BASE}/series"

    response = requests.get(
        url,
        params={
            "q": query,
            "limit": limit
        },
        timeout=30
    )

    response.raise_for_status()
    payload = response.json()

    docs = payload.get("series", {}).get("docs", [])

    rows = []

    for doc in docs:
        rows.append({
            "Provider": doc.get("provider_code"),
            "Dataset": doc.get("dataset_code"),
            "Series Code": doc.get("series_code"),
            "Series Name": doc.get("series_name"),
            "Frequency": doc.get("frequency"),
            "Dimensions": doc.get("dimensions_codes_order")
        })

    return pd.DataFrame(rows)

In [151]:
!pip install dbnomics

Defaulting to user installation because normal site-packages is not writeable


In [156]:
import dbnomics
import pandas as pd

In [157]:
df = dbnomics.fetch_series(
    provider_code="OECD",
    dataset_code="DSD_KEI@DF_KEI",
    series_code="IND.M.PRINTO01.IXOB.YS"
)

Finished call to 'dbnomics._fetch_response' after 1.17(s), this was the 1st time calling it.
Finished call to 'dbnomics._fetch_response' after 3.15(s), this was the 2nd time calling it.
Finished call to 'dbnomics._fetch_response' after 4.22(s), this was the 3rd time calling it.


FetchError: Could not fetch data from URL 'https://api.db.nomics.world/v22/series/OECD/DSD_KEI@DF_KEI/IND.M.PRINTO01.IXOB.YS?observations=1&offset=0'

In [159]:
import requests
import pandas as pd

BASE = "https://api.db.nomics.world/v22"

In [160]:
url = f"{BASE}/datasets/IMF"

r = requests.get(url, timeout=30)
r.raise_for_status()

imf_datasets = r.json()
imf_datasets.keys()

dict_keys(['_meta', 'datasets'])

In [161]:
datasets_df = pd.DataFrame(imf_datasets["datasets"]["docs"])

datasets_df[
    ["code", "name"]
].head(50)

,code,name
0,AFRREO,Sub-Saharan Africa Regional Economic Outlook (...
1,APDREO,Asia and Pacific Regional Economic Outlook (AP...
2,BOP,Balance of Payments (BOP)
3,BOPAGG,"Balance of Payments (BOP), World and Regional ..."
4,CDIS,Coordinated Direct Investment Survey (CDIS)
5,COFER,Currency Composition of Official Foreign Excha...
6,CPI,Consumer Price Index (CPI)
7,CPIS,Coordinated Portfolio Investment Survey (CPIS)
8,DOT,Direction of Trade Statistics (DOTS)
9,FAS,Financial Access Survey (FAS)


In [162]:
IFS_URL = f"{BASE}/series/IMF/IFS"

r = requests.get(
    IFS_URL,
    params={
        "limit": 5,
        "facets": "true"
    },
    timeout=30
)

print(r.status_code)
print(r.text[:1000])

200
{"_meta":{"args":{"align_periods":false,"dataset_code":"IFS","dimensions":{},"facets":true,"format":"json","limit":5,"metadata":true,"observations":false,"offset":0,"provider_code":"IMF","q":"","series_code":null},"version":"22.1.17"},"dataset":{"attributes_labels":{"BASE_YEAR":"Base Year","OBS_STATUS":"Observation Status (incl. Confidentiality)","TIME_FORMAT":"Time format","UNIT_MULT":"Scale"},"attributes_values_labels":{"TIME_FORMAT":{"P1D":"Daily","P1M":"Monthly","P1Y":"Annual","P3M":"Quarterly","P6M":"Bi-annual","P7D":"Weekly"},"UNIT_MULT":{"0":"Units","1":"Tens","10":"Ten Billions","11":"Hundred Billions","12":"Trillions","13":"Ten Trillions","14":"Hundred Trillions","15":"Quadrillions","2":"Hundreds","3":"Thousands","4":"Ten Thousands","5":"Hundred Thousands","6":"Millions","7":"Ten Millions","8":"Hundred Millions","9":"Billions","N1":"Tenths","N10":"Ten Billionths","N11":"Hundred Billionths","N12":"Trillionths","N13":"Ten Trillionths","N14":"Hundred Trillionths","N15":"Quadr

In [163]:
SEARCH_URL = f"{BASE}/search"

In [164]:
def dbnomics_search(query, limit=50):
    r = requests.get(
        f"{BASE}/search",
        params={
            "q": query,
            "limit": limit
        },
        timeout=30
    )

    print("Status:", r.status_code)

    if r.status_code != 200:
        print(r.text[:1000])
        r.raise_for_status()

    payload = r.json()

    docs = payload.get("results", {}).get("docs", [])

    rows = []

    for doc in docs:
        rows.append({
            "Provider": doc.get("provider_code"),
            "Dataset": doc.get("dataset_code"),
            "Series Code": doc.get("series_code"),
            "Series Name": doc.get("name"),
            "Frequency": doc.get("frequency")
        })

    return pd.DataFrame(rows)

In [165]:
cpi_candidates = dbnomics_search(
    "India consumer price index"
)

cpi_candidates.head(30)

Status: 200


,Provider,Dataset,Series Code,Series Name,Frequency
0,OECD,None,None,Main Economic Indicators Publication,None
1,OECD,None,None,"Consumer price indices (CPIs, HICPs), COICOP 1999",None
2,IMF,None,None,Consumer Price Index (CPI),None
3,ILO,None,None,"National consumer price index (CPI) by COICOP,...",None
4,ILO,None,None,National consumer price index (CPI) by COICOP ...,None
5,OECD,None,None,"Consumer price indices (CPIs), exchange, API o...",None
6,ILO,None,None,"National consumer price index (CPI) by COICOP,...",None
7,OECD,None,None,TEST - Main Economic Indicators Publication,None
8,IMF,None,None,Principal Global Indicators (PGI),None
9,IMF,None,None,International Financial Statistics (IFS),None


In [166]:
iip_candidates = dbnomics_search(
    "India industrial production"
)

iip_candidates.head(30)

Status: 200


,Provider,Dataset,Series Code,Series Name,Frequency
0,FAO,None,None,Forestry: Forestry Production and Trade,None
1,UNIDO,None,None,Monthly Index of Industrial Production (IIP) a...,None
2,IMF,None,None,International Financial Statistics (IFS),None
3,NBB,None,None,Foreign trade - Belgium - Community concept,None
4,NBB,None,None,Foreign trade - Belgium - National concept,None
5,NBB,None,None,Foreign trade - Flemish region - National concept,None
6,FAO,None,None,Climate Change: Agrifood systems emissions: Em...,None
7,OECD,None,None,Key Short-Term Economic Indicators,None
8,IMF,None,None,Principal Global Indicators (PGI),None
9,OECD,None,None,OECD Data Live dataset,None


In [167]:
IFS_URL = f"{BASE}/series/IMF/IFS"

r = requests.get(
    IFS_URL,
    params={
        "limit": 1,
        "facets": "true"
    },
    timeout=30
)

print("Status:", r.status_code)
payload_ifs = r.json()

payload_ifs.keys()

Status: 200


dict_keys(['_meta', 'dataset', 'errors', 'provider', 'series', 'series_dimensions_facets'])

In [168]:
dataset_meta = payload_ifs.get("dataset", {})
dataset_meta.keys()

dict_keys(['attributes_labels', 'attributes_values_labels', 'code', 'description', 'dimensions_codes_order', 'dimensions_labels', 'dimensions_values_labels', 'dir_hash', 'indexed_at', 'name', 'nb_series', 'notes', 'provider_code', 'provider_name', 'updated_at'])

In [169]:
dataset_meta.get("dimensions_codes_order")

['FREQ', 'REF_AREA', 'INDICATOR']

In [170]:
dataset_meta.get("dimensions_values_labels")

{'FREQ': {'A': 'Annual',
  'B': 'Bi-annual',
  'D': 'Daily',
  'M': 'Monthly',
  'Q': 'Quarterly',
  'W': 'Weekly'},
 'INDICATOR': {'10RA__XDC': 'Monetary, Monetary Authorities/Central Bank, Total Assets (Non-Standardized Presentation), Domestic Currency',
  '11____EUR': 'Monetary, Monetary Authorities, Foreign Assets (Non-Standardized Presentation), Euros',
  '11____USD': 'Monetary, Monetary Authorities, Foreign Assets (Non-Standardized Presentation), US Dollars',
  '11____XDC': 'Monetary, Monetary Authorities, Foreign Assets (Non-Standardized Presentation), Domestic Currency',
  '12AG__XDC': 'Monetary, Monetary Authorities, Claims on Central or General Government, Accumulated Interest Arrears (Non-Standardized Presentation), Domestic Currency',
  '12AN__XDC': 'Monetary, Monetary Authorities, Claims on Central Government, Net (Non-Standardized Presentation), Domestic Currency',
  '12AX__XDC': 'Monetary, Monetary Authorities, Claims on Central or General Government, Provincial Governme

In [171]:
dimensions = dataset_meta.get("dimensions_values_labels", {})

for dim_name, values in dimensions.items():
    print("\nDIMENSION:", dim_name)
    print(list(values.items())[:20])


DIMENSION: FREQ
[('A', 'Annual'), ('B', 'Bi-annual'), ('D', 'Daily'), ('M', 'Monthly'), ('Q', 'Quarterly'), ('W', 'Weekly')]

DIMENSION: INDICATOR
[('10RA__XDC', 'Monetary, Monetary Authorities/Central Bank, Total Assets (Non-Standardized Presentation), Domestic Currency'), ('11____EUR', 'Monetary, Monetary Authorities, Foreign Assets (Non-Standardized Presentation), Euros'), ('11____USD', 'Monetary, Monetary Authorities, Foreign Assets (Non-Standardized Presentation), US Dollars'), ('11____XDC', 'Monetary, Monetary Authorities, Foreign Assets (Non-Standardized Presentation), Domestic Currency'), ('12AG__XDC', 'Monetary, Monetary Authorities, Claims on Central or General Government, Accumulated Interest Arrears (Non-Standardized Presentation), Domestic Currency'), ('12AN__XDC', 'Monetary, Monetary Authorities, Claims on Central Government, Net (Non-Standardized Presentation), Domestic Currency'), ('12AX__XDC', 'Monetary, Monetary Authorities, Claims on Central or General Government, P

In [172]:
for dim_name, values in dimensions.items():
    india_matches = {
        code: label
        for code, label in values.items()
        if "india" in str(label).lower()
    }
    
    if india_matches:
        print("\nIndia found in:", dim_name)
        print(india_matches)


India found in: REF_AREA
{'IN': 'India'}


In [173]:
for dim_name, values in dimensions.items():
    cpi_matches = {
        code: label
        for code, label in values.items()
        if "consumer price" in str(label).lower()
        or "cpi" in str(label).lower()
    }
    
    if cpi_matches:
        print("\nCPI indicators found in:", dim_name)
        for code, label in list(cpi_matches.items())[:30]:
            print(code, "→", label)


CPI indicators found in: INDICATOR
EREER_IX → Exchange Rates, Real Effective Exchange Rate based on Consumer Price Index, Index
EREER_PC_CP_A_PT → Exchange Rates, Real Effective Exchange Rate based on Consumer Price Index, Percentage change, Corresponding period previous year, Percent
PCPIHA_IX → Prices, Consumer Price Index, Harmonized, Index
PCPI_IX → Prices, Consumer Price Index, All items, Index
PCPI_PC_CP_A_PT → Prices, Consumer Price Index, All items, Percentage change, Corresponding period previous year, Percent
PCPI_PC_PP_PT → Prices, Consumer Price Index, All items, Percentage change, Previous period, Percent


In [174]:
for dim_name, values in dimensions.items():
    iip_matches = {
        code: label
        for code, label in values.items()
        if "industrial production" in str(label).lower()
        or "production index" in str(label).lower()
    }
    
    if iip_matches:
        print("\nIndustrial-production indicators found in:", dim_name)
        for code, label in list(iip_matches.items())[:30]:
            print(code, "→", label)


Industrial-production indicators found in: INDICATOR
AIPCO_IX → Economic Activity, Industrial Production, Construction, Index
AIPCO_PC_CP_A_PT → Economic Activity, Industrial Production, Construction, Percentage change, Corresponding period previous year, Percent
AIPCO_PC_PP_PT → Economic Activity, Industrial Production, Construction, Percentage change, Previous period, Percent
AIPEE_IX → Economic Activity, Industrial Production, Energy, Electricity Production, Index
AIPEE_PC_CP_A_PT → Economic Activity, Industrial Production, Energy, Electricity Production, Percentage change, Corresponding period previous year, Percent
AIPEE_PC_PP_PT → Economic Activity, Industrial Production, Energy, Electricity Production, Percentage change, Previous period, Percent
AIPMA_IX → Economic Activity, Industrial Production, Manufacturing, Index
AIPMA_PC_CP_A_PT → Economic Activity, Industrial Production, Manufacturing, Percentage change, Corresponding period previous year, Percent
AIPMA_PC_PP_PT → Econom

In [175]:
id="get_india_code"
# Find the India code in every dimension

for dim_name, values in dimensions.items():
    india_matches = {
        code: label
        for code, label in values.items()
        if "india" in str(label).lower()
    }

    if india_matches:
        print("\nDIMENSION:", dim_name)
        print(india_matches)


DIMENSION: REF_AREA
{'IN': 'India'}


In [177]:
# ── Build India CPI and IIP DBnomics series codes automatically ──────────────

# `dataset_meta` and `dimensions` should already exist from the IFS metadata step.
# If not, run the IMF/IFS metadata cell once before this.

dimension_order = dataset_meta["dimensions_codes_order"]

# Find the dimension containing India
india_dim = None
india_code = None

for dim_name, values in dimensions.items():
    matches = {
        code: label
        for code, label in values.items()
        if str(label).strip().lower() == "india"
        or "india" in str(label).lower()
    }
    
    if matches:
        india_dim = dim_name
        india_code = list(matches.keys())[0]
        print(f"India found in dimension: {india_dim}")
        print(f"India code: {india_code} → {matches[india_code]}")
        break

if india_code is None:
    raise ValueError("India country code was not found. Print `dimensions.keys()` and inspect manually.")

# Build a DBnomics code while respecting actual IMF IFS dimension order
def build_ifs_series_code(indicator_code):
    code_parts = []
    
    for dim in dimension_order:
        if dim == "FREQ":
            code_parts.append("M")                   # Monthly
        elif dim == india_dim:
            code_parts.append(india_code)            # India
        elif dim == "INDICATOR":
            code_parts.append(indicator_code)        # CPI or IIP
        else:
            raise ValueError(
                f"Unexpected IFS dimension '{dim}'. "
                "Please print dimension_order and inspect it."
            )
    
    return ".".join(code_parts)

CPI_SERIES_CODE = build_ifs_series_code("PCPI_PC_CP_A_PT")
IIP_SERIES_CODE = build_ifs_series_code("AIP_PC_CP_A_PT")

print("\nDimension order:", dimension_order)
print("CPI series code:", CPI_SERIES_CODE)
print("IIP series code:", IIP_SERIES_CODE)

India found in dimension: REF_AREA
India code: IN → India

Dimension order: ['FREQ', 'REF_AREA', 'INDICATOR']
CPI series code: M.IN.PCPI_PC_CP_A_PT
IIP series code: M.IN.AIP_PC_CP_A_PT


In [178]:
id="download_cpi_iip"
def fetch_ifs_series(series_code, value_name):
    url = f"{BASE}/series/IMF/IFS/{series_code}"

    response = requests.get(
        url,
        params={"observations": 1},
        timeout=30
    )
    response.raise_for_status()

    doc = response.json()["series"]["docs"][0]

    df = pd.DataFrame({
        "Date": pd.to_datetime(doc["period"]),
        value_name: pd.to_numeric(doc["value"], errors="coerce")
    })

    return (
        df.dropna()
          .sort_values("Date")
          .reset_index(drop=True)
    )

cpi_monthly = fetch_ifs_series(
    CPI_SERIES_CODE,
    "CPI_Inflation_YoY"
)

iip_monthly = fetch_ifs_series(
    IIP_SERIES_CODE,
    "IIP_Growth_YoY"
)

cpi_monthly.head(), iip_monthly.head()

(        Date  CPI_Inflation_YoY
 0 1958-01-01           3.712876
 1 1958-02-01           2.722769
 2 1958-03-01           2.722769
 3 1958-04-01           1.699017
 4 1958-05-01           2.891590,
         Date  IIP_Growth_YoY
 0 1972-01-01        7.396734
 1 1972-02-01        9.154229
 2 1972-03-01        5.786618
 3 1972-04-01        3.339882
 4 1972-05-01        6.986028)

In [179]:
# ── Restrict macro series to banking-study window ────────────────────────────

START_DATE = "2013-01-01"

cpi_monthly = (
    cpi_monthly
    .query("Date >= @START_DATE")
    .sort_values("Date")
    .reset_index(drop=True)
)

iip_monthly = (
    iip_monthly
    .query("Date >= @START_DATE")
    .sort_values("Date")
    .reset_index(drop=True)
)

print("CPI period:", cpi_monthly["Date"].min().date(), "to", cpi_monthly["Date"].max().date())
print("IIP period:", iip_monthly["Date"].min().date(), "to", iip_monthly["Date"].max().date())

CPI period: 2013-01-01 to 2025-06-01
IIP period: 2013-01-01 to 2024-10-01


In [181]:
# ── Standardise India VIX monthly dataframe ──────────────────────────────────

india_vix_monthly = india_vix_monthly.copy()

# Move date index into a normal column when needed
if "Date" not in india_vix_monthly.columns:
    india_vix_monthly = india_vix_monthly.reset_index()

# Identify and standardise the date column
possible_date_cols = ["Date", "date", "Datetime", "index"]

date_col = next(
    (col for col in possible_date_cols if col in india_vix_monthly.columns),
    None
)

if date_col is None:
    raise ValueError(
        f"Could not identify the India VIX date column. Found: {india_vix_monthly.columns.tolist()}"
    )

india_vix_monthly = india_vix_monthly.rename(columns={date_col: "Date"})

# Ensure expected VIX column exists
if "India_VIX_Avg" not in india_vix_monthly.columns:
    numeric_cols = india_vix_monthly.select_dtypes(include="number").columns.tolist()

    if len(numeric_cols) != 1:
        raise ValueError(
            f"Could not uniquely identify India VIX values. Numeric columns found: {numeric_cols}"
        )

    india_vix_monthly = india_vix_monthly.rename(
        columns={numeric_cols[0]: "India_VIX_Avg"}
    )

india_vix_monthly["Date"] = pd.to_datetime(india_vix_monthly["Date"])

india_vix_monthly = (
    india_vix_monthly[["Date", "India_VIX_Avg"]]
    .dropna()
    .sort_values("Date")
    .reset_index(drop=True)
)

india_vix_monthly.head()

,Date,India_VIX_Avg
0,2013-01-31,13.870435
1,2013-02-28,15.503000
2,2013-03-31,14.998947
3,2013-04-30,15.635500
4,2013-05-31,17.149546


In [182]:
macro_monthly = (
    cpi_monthly
    .merge(iip_monthly, on="Date", how="inner")
    .merge(india_vix_monthly, on="Date", how="left")
    .sort_values("Date")
    .reset_index(drop=True)
)

print("Macro panel:", macro_monthly["Date"].min().date(), "to", macro_monthly["Date"].max().date())
print("Observations:", len(macro_monthly))

macro_monthly.head(15)

Macro panel: 2013-01-01 to 2024-10-01
Observations: 142


,Date,CPI_Inflation_YoY,IIP_Growth_YoY,India_VIX_Avg
0,2013-01-01,10.351157,-4.149798,NaN
1,2013-02-01,10.508023,-9.010774,NaN
2,2013-03-01,9.646922,-8.096085,NaN
3,2013-04-01,8.970400,4.702194,NaN
4,2013-05-01,8.770290,-0.100705,NaN
5,2013-06-01,9.910464,-2.091255,NaN
6,2013-07-01,9.964967,5.466238,NaN
7,2013-08-01,9.992858,-3.948773,NaN
8,2013-09-01,10.277989,0.098328,NaN
9,2013-10-01,10.542557,4.022989,NaN


In [183]:
# ── Align all monthly series using Month_Period ───────────────────────────────

for df in [cpi_monthly, iip_monthly, india_vix_monthly]:
    df["Month"] = pd.to_datetime(df["Date"]).dt.to_period("M")

macro_monthly = (
    cpi_monthly[["Month", "CPI_Inflation_YoY"]]
    .merge(
        iip_monthly[["Month", "IIP_Growth_YoY"]],
        on="Month",
        how="inner"
    )
    .merge(
        india_vix_monthly[["Month", "India_VIX_Avg"]],
        on="Month",
        how="left"
    )
    .sort_values("Month")
    .reset_index(drop=True)
)

# Use a consistent month-end date for later quarterly conversion
macro_monthly["Date"] = macro_monthly["Month"].dt.to_timestamp("M")

macro_monthly = macro_monthly[
    ["Date", "CPI_Inflation_YoY", "IIP_Growth_YoY", "India_VIX_Avg"]
]

print(
    "Macro panel:",
    macro_monthly["Date"].min().date(),
    "to",
    macro_monthly["Date"].max().date()
)

print("\nMissing values:")
display(macro_monthly.isna().sum().to_frame("Missing"))

macro_monthly.head(15)

Macro panel: 2013-01-31 to 2024-10-31

Missing values:


,Missing
Date,0
CPI_Inflation_YoY,0
IIP_Growth_YoY,0
India_VIX_Avg,0


,Date,CPI_Inflation_YoY,IIP_Growth_YoY,India_VIX_Avg
0,2013-01-31,10.351157,-4.149798,13.870435
1,2013-02-28,10.508023,-9.010774,15.503000
2,2013-03-31,9.646922,-8.096085,14.998947
3,2013-04-30,8.970400,4.702194,15.635500
4,2013-05-31,8.770290,-0.100705,17.149546
5,2013-06-30,9.910464,-2.091255,18.672000
6,2013-07-31,9.964967,5.466238,18.353043
7,2013-08-31,9.992858,-3.948773,24.573000
8,2013-09-30,10.277989,0.098328,27.666000
9,2013-10-31,10.542557,4.022989,22.551429


In [184]:
# ── USD/INR and Brent from yfinance ─────────────────────────────────────────

START_DATE = "2013-01-01"
END_DATE = "2026-06-30"

yf_prices = yf.download(
    ["INR=X", "BZ=F"],
    start=START_DATE,
    end=END_DATE,
    auto_adjust=False,
    progress=False
)

# Keep only adjusted/close price layer safely
if isinstance(yf_prices.columns, pd.MultiIndex):
    close_prices = yf_prices["Close"].copy()
else:
    close_prices = yf_prices[["INR=X", "BZ=F"]].copy()

close_prices = close_prices.rename(
    columns={
        "INR=X": "USDINR",
        "BZ=F": "Brent_USD"
    }
)

# Monthly average levels
external_monthly = (
    close_prices
    .resample("ME")
    .mean()
    .reset_index()
    .rename(columns={"Date": "Date"})
)

# Convert to monthly percentage changes
external_monthly["USDINR_Depreciation_MoM"] = (
    external_monthly["USDINR"].pct_change() * 100
)

external_monthly["Brent_Change_MoM"] = (
    external_monthly["Brent_USD"].pct_change() * 100
)

external_monthly.head()

Ticker,index,Brent_USD,USDINR,USDINR_Depreciation_MoM,Brent_Change_MoM
0,2013-01-31,112.268096,54.319174,NaN,NaN
1,2013-02-28,116.082632,53.778500,-0.995365,3.397702
2,2013-03-31,109.520526,54.412333,1.178599,-5.652961
3,2013-04-30,103.566819,54.363091,-0.090498,-5.436156
4,2013-05-31,103.215455,54.898391,0.984677,-0.339263


In [186]:
# ── Standardise date columns before merging macro + external data ─────────────

def ensure_date_column(df, date_name="Date"):
    out = df.copy()

    # If Date is in the index, bring it back as a normal column
    if date_name not in out.columns:
        out = out.reset_index()

    # Find likely date column after reset_index()
    candidates = ["Date", "date", "Datetime", "index"]

    found = next((c for c in candidates if c in out.columns), None)

    if found is None:
        raise ValueError(
            f"Could not identify date column. Found columns: {out.columns.tolist()}"
        )

    out = out.rename(columns={found: date_name})
    out[date_name] = pd.to_datetime(out[date_name])

    return out


macro_monthly = ensure_date_column(macro_monthly, "Date")
external_monthly = ensure_date_column(external_monthly, "Date")

macro_monthly["Month"] = macro_monthly["Date"].dt.to_period("M")
external_monthly["Month"] = external_monthly["Date"].dt.to_period("M")

print("Macro columns:", macro_monthly.columns.tolist())
print("External columns:", external_monthly.columns.tolist())

Macro columns: ['Date', 'CPI_Inflation_YoY', 'IIP_Growth_YoY', 'India_VIX_Avg', 'Month']
External columns: ['level_0', 'Date', 'Brent_USD', 'USDINR', 'USDINR_Depreciation_MoM', 'Brent_Change_MoM', 'Month']


In [187]:
# ── Merge USD/INR and Brent with macro panel ─────────────────────────────────

macro_monthly = (
    macro_monthly
    .drop(columns=[
        "USDINR",
        "Brent_USD",
        "USDINR_Depreciation_MoM",
        "Brent_Change_MoM"
    ], errors="ignore")
    .merge(
        external_monthly[
            [
                "Month",
                "USDINR",
                "Brent_USD",
                "USDINR_Depreciation_MoM",
                "Brent_Change_MoM"
            ]
        ],
        on="Month",
        how="left"
    )
    .sort_values("Month")
    .reset_index(drop=True)
)

# Standard month-end date for all rows
macro_monthly["Date"] = macro_monthly["Month"].dt.to_timestamp("M")

display(macro_monthly.head(15))

print("\nMissing values:")
display(macro_monthly.isna().sum().to_frame("Missing values"))

,Date,CPI_Inflation_YoY,IIP_Growth_YoY,India_VIX_Avg,Month,USDINR,Brent_USD,USDINR_Depreciation_MoM,Brent_Change_MoM
0,2013-01-31,10.351157,-4.149798,13.870435,2013-01,54.319174,112.268096,NaN,NaN
1,2013-02-28,10.508023,-9.010774,15.503000,2013-02,53.778500,116.082632,-0.995365,3.397702
2,2013-03-31,9.646922,-8.096085,14.998947,2013-03,54.412333,109.520526,1.178599,-5.652961
3,2013-04-30,8.970400,4.702194,15.635500,2013-04,54.363091,103.566819,-0.090498,-5.436156
4,2013-05-31,8.770290,-0.100705,17.149546,2013-05,54.898391,103.215455,0.984677,-0.339263
5,2013-06-30,9.910464,-2.091255,18.672000,2013-06,58.096751,103.142632,5.825962,-0.070555
6,2013-07-31,9.964967,5.466238,18.353043,2013-07,59.729869,107.491905,2.811033,4.216756
7,2013-08-31,9.992858,-3.948773,24.573000,2013-08,62.752409,110.511905,5.060349,2.809514
8,2013-09-30,10.277989,0.098328,27.666000,2013-09,64.019333,111.320001,2.018925,0.731229
9,2013-10-31,10.542557,4.022989,22.551429,2013-10,61.570000,109.502273,-3.825927,-1.632885



Missing values:


,Missing values
Date,0
CPI_Inflation_YoY,0
IIP_Growth_YoY,0
India_VIX_Avg,0
Month,0
USDINR,0
Brent_USD,0
USDINR_Depreciation_MoM,1
Brent_Change_MoM,1


## Historical RBI pressure model: 2013 to Oct-2024 , Uses CPI + IIP + VIX + banking variables.


In [188]:
# ── Historical macro panel: 2013 to Oct-2024 ─────────────────────────────────

hist_macro_monthly = macro_monthly.copy()

# Ensure Date is a proper datetime column
if "Date" not in hist_macro_monthly.columns:
    hist_macro_monthly = hist_macro_monthly.reset_index()

date_col = next(
    (c for c in ["Date", "date", "Datetime", "index"] if c in hist_macro_monthly.columns),
    None
)

if date_col is None:
    raise ValueError(f"Date column not found. Columns: {hist_macro_monthly.columns.tolist()}")

hist_macro_monthly = hist_macro_monthly.rename(columns={date_col: "Date"})
hist_macro_monthly["Date"] = pd.to_datetime(hist_macro_monthly["Date"])

hist_macro_monthly = (
    hist_macro_monthly
    .loc[
        (hist_macro_monthly["Date"] >= "2013-01-01") &
        (hist_macro_monthly["Date"] <= "2024-10-31")
    ]
    .sort_values("Date")
    .reset_index(drop=True)
)

required_macro_cols = [
    "CPI_Inflation_YoY",
    "IIP_Growth_YoY",
    "India_VIX_Avg"
]

missing_cols = [c for c in required_macro_cols if c not in hist_macro_monthly.columns]

if missing_cols:
    raise ValueError(f"Missing macro columns: {missing_cols}")

print(
    "Historical monthly sample:",
    hist_macro_monthly["Date"].min().date(),
    "to",
    hist_macro_monthly["Date"].max().date()
)

hist_macro_monthly.head()

Historical monthly sample: 2013-01-31 to 2024-10-31


,Date,CPI_Inflation_YoY,IIP_Growth_YoY,India_VIX_Avg,Month,USDINR,Brent_USD,USDINR_Depreciation_MoM,Brent_Change_MoM
0,2013-01-31,10.351157,-4.149798,13.870435,2013-01,54.319174,112.268096,NaN,NaN
1,2013-02-28,10.508023,-9.010774,15.503000,2013-02,53.778500,116.082632,-0.995365,3.397702
2,2013-03-31,9.646922,-8.096085,14.998947,2013-03,54.412333,109.520526,1.178599,-5.652961
3,2013-04-30,8.970400,4.702194,15.635500,2013-04,54.363091,103.566819,-0.090498,-5.436156
4,2013-05-31,8.770290,-0.100705,17.149546,2013-05,54.898391,103.215455,0.984677,-0.339263


In [189]:
# ── Monthly macro variables → quarterly averages ─────────────────────────────

hist_macro_monthly["Quarter_End"] = (
    hist_macro_monthly["Date"]
    .dt.to_period("Q")
    .dt.end_time
    .dt.normalize()
)

hist_macro_q = (
    hist_macro_monthly
    .groupby("Quarter_End", as_index=False)
    .agg(
        CPI_Inflation_YoY=("CPI_Inflation_YoY", "mean"),
        IIP_Growth_YoY=("IIP_Growth_YoY", "mean"),
        India_VIX_Avg=("India_VIX_Avg", "mean")
    )
    .sort_values("Quarter_End")
    .reset_index(drop=True)
)

hist_macro_q.head()

,Quarter_End,CPI_Inflation_YoY,IIP_Growth_YoY,India_VIX_Avg
0,2013-03-31,10.168701,-7.085552,14.790794
1,2013-06-30,9.217051,0.836745,17.152349
2,2013-09-30,10.078605,0.538598,23.530681
3,2013-12-31,10.574060,4.379646,20.370393
4,2014-03-31,8.312830,5.405573,16.557519


In [190]:
# ── Merge macro conditions with banking conditions ───────────────────────────

banking_pressure_cols = [
    "Quarter_End",
    "Repo_Rate",
    "Bank_Rate",
    "Credit_Deposit_Ratio",
    "GNPA_Ratio",
    "Cash_Reserve_Ratio",
    "WALR",
    "MSE_Credit_Growth_YoY"
]

available_banking_cols = [
    col for col in banking_pressure_cols
    if col in quarterly_df.columns
]

missing_banking_cols = [
    col for col in banking_pressure_cols
    if col not in quarterly_df.columns
]

print("Available banking variables:", available_banking_cols)
print("Missing banking variables:", missing_banking_cols)

historical_pressure_df = (
    hist_macro_q
    .merge(
        quarterly_df[available_banking_cols],
        on="Quarter_End",
        how="left"
    )
    .sort_values("Quarter_End")
    .reset_index(drop=True)
)

historical_pressure_df.head()

Available banking variables: ['Quarter_End', 'Repo_Rate', 'Bank_Rate', 'Credit_Deposit_Ratio', 'GNPA_Ratio', 'Cash_Reserve_Ratio', 'WALR', 'MSE_Credit_Growth_YoY']
Missing banking variables: []


,Quarter_End,CPI_Inflation_YoY,IIP_Growth_YoY,India_VIX_Avg,Repo_Rate,Bank_Rate,Credit_Deposit_Ratio,GNPA_Ratio,Cash_Reserve_Ratio,WALR,MSE_Credit_Growth_YoY
0,2013-03-31,10.168701,-7.085552,14.790794,7.50,8.50,77.93,3.422945,4.0,11.46,12.769316
1,2013-06-30,9.217051,0.836745,17.152349,7.25,8.25,76.42,4.004487,4.0,11.40,22.630200
2,2013-09-30,10.078605,0.538598,23.530681,7.50,9.50,78.35,4.218281,4.0,11.96,21.847790
3,2013-12-31,10.574060,4.379646,20.370393,7.75,8.75,76.81,4.397677,4.0,11.70,22.894464
4,2014-03-31,8.312830,5.405573,16.557519,8.00,9.00,77.79,4.115068,4.0,11.57,25.879074


In [191]:
# ── Completeness check ───────────────────────────────────────────────────────

coverage_table = pd.DataFrame({
    "Variable": historical_pressure_df.columns,
    "Non-missing observations": historical_pressure_df.notna().sum().values,
    "Missing observations": historical_pressure_df.isna().sum().values
})

coverage_table

,Variable,Non-missing observations,Missing observations
0,Quarter_End,48,0
1,CPI_Inflation_YoY,48,0
2,IIP_Growth_YoY,48,0
3,India_VIX_Avg,48,0
4,Repo_Rate,48,0
5,Bank_Rate,48,0
6,Credit_Deposit_Ratio,48,0
7,GNPA_Ratio,48,0
8,Cash_Reserve_Ratio,48,0
9,WALR,48,0


In [192]:
main_pressure_vars = [
    "CPI_Inflation_YoY",
    "IIP_Growth_YoY",
    "India_VIX_Avg",
    "Credit_Deposit_Ratio",
    "GNPA_Ratio"
]

In [193]:
# ── Standardise macro and banking pressure variables ─────────────────────────

from scipy.stats import zscore

pressure_df = historical_pressure_df.copy()

for variable in main_pressure_vars:
    pressure_df[f"z_{variable}"] = zscore(
        pressure_df[variable],
        nan_policy="omit"
    )

pressure_df[
    [
        "Quarter_End",
        "z_CPI_Inflation_YoY",
        "z_IIP_Growth_YoY",
        "z_India_VIX_Avg",
        "z_Credit_Deposit_Ratio",
        "z_GNPA_Ratio"
    ]
].head()

,Quarter_End,z_CPI_Inflation_YoY,z_IIP_Growth_YoY,z_India_VIX_Avg,z_Credit_Deposit_Ratio,z_GNPA_Ratio
0,2013-03-31,2.400923,-1.138105,-0.513360,0.894244,-1.172689
1,2013-06-30,1.904202,-0.339338,-0.008100,0.323540,-0.955763
2,2013-09-30,2.353897,-0.369399,1.356561,1.052983,-0.876015
3,2013-12-31,2.612503,0.017875,0.680409,0.470941,-0.809097
4,2014-03-31,1.432236,0.121315,-0.135365,0.841331,-0.914515


In [194]:
# ── RBI policy-pressure indexes ──────────────────────────────────────────────

# Higher = stronger inflation / market uncertainty pressure
pressure_df["Inflation_Uncertainty_Pressure"] = (
    pressure_df["z_CPI_Inflation_YoY"]
    + pressure_df["z_India_VIX_Avg"]
)

# Higher = weaker activity plus greater banking-sector stress
pressure_df["Growth_Banking_Stress"] = (
    -pressure_df["z_IIP_Growth_YoY"]
    + pressure_df["z_GNPA_Ratio"]
    + pressure_df["z_Credit_Deposit_Ratio"]
)

# Standardise the final indexes to improve comparability
pressure_df["Inflation_Uncertainty_Pressure_z"] = zscore(
    pressure_df["Inflation_Uncertainty_Pressure"],
    nan_policy="omit"
)

pressure_df["Growth_Banking_Stress_z"] = zscore(
    pressure_df["Growth_Banking_Stress"],
    nan_policy="omit"
)

pressure_df[
    [
        "Quarter_End",
        "Inflation_Uncertainty_Pressure_z",
        "Growth_Banking_Stress_z",
        "Repo_Rate",
        "Bank_Rate",
        "WALR",
        "MSE_Credit_Growth_YoY"
    ]
].head()

,Quarter_End,Inflation_Uncertainty_Pressure_z,Growth_Banking_Stress_z,Repo_Rate,Bank_Rate,WALR,MSE_Credit_Growth_YoY
0,2013-03-31,1.144304,0.544031,7.50,8.50,11.46,12.769316
1,2013-06-30,1.149481,-0.185350,7.25,8.25,11.40,22.630200
2,2013-09-30,2.249405,0.345765,7.50,9.50,11.96,21.847790
3,2013-12-31,1.996275,-0.225312,7.75,8.75,11.70,22.894464
4,2014-03-31,0.786207,-0.123087,8.00,9.00,11.57,25.879074


In [195]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=pressure_df["Quarter_End"],
        y=pressure_df["Inflation_Uncertainty_Pressure_z"],
        mode="lines+markers",
        name="Inflation–Uncertainty Pressure"
    )
)

fig.add_trace(
    go.Scatter(
        x=pressure_df["Quarter_End"],
        y=pressure_df["Growth_Banking_Stress_z"],
        mode="lines+markers",
        name="Growth–Banking Stress"
    )
)

fig.add_trace(
    go.Scatter(
        x=pressure_df["Quarter_End"],
        y=pressure_df["Repo_Rate"],
        mode="lines+markers",
        name="Repo Rate",
        yaxis="y2"
    )
)

fig.update_layout(
    template="plotly_white",
    width=1150,
    height=560,
    title="Historical RBI Pressure Indicators and Repo Rate",
    xaxis_title="Quarter",
    yaxis=dict(title="Standardised pressure index"),
    yaxis2=dict(
        title="Repo Rate (%)",
        overlaying="y",
        side="right"
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="left",
        x=0
    )
)

fig.show()

In [196]:
# ── Inspect what drives each pressure index ──────────────────────────────────

pressure_components = pressure_df[
    [
        "Quarter_End",
        "z_CPI_Inflation_YoY",
        "z_India_VIX_Avg",
        "z_IIP_Growth_YoY",
        "z_GNPA_Ratio",
        "z_Credit_Deposit_Ratio",
        "Inflation_Uncertainty_Pressure_z",
        "Growth_Banking_Stress_z"
    ]
].copy()

pressure_components.head(12)

,Quarter_End,z_CPI_Inflation_YoY,z_India_VIX_Avg,z_IIP_Growth_YoY,z_GNPA_Ratio,z_Credit_Deposit_Ratio,Inflation_Uncertainty_Pressure_z,Growth_Banking_Stress_z
0,2013-03-31,2.400923,-0.513360,-1.138105,-1.172689,0.894244,1.144304,0.544031
1,2013-06-30,1.904202,-0.008100,-0.339338,-0.955763,0.323540,1.149481,-0.185350
2,2013-09-30,2.353897,1.356561,-0.369399,-0.876015,1.052983,2.249405,0.345765
3,2013-12-31,2.612503,0.680409,0.017875,-0.809097,0.470941,1.996275,-0.225312
4,2014-03-31,1.432236,-0.135365,0.121315,-0.914515,0.841331,0.786207,-0.123087
5,2014-06-30,1.195610,1.471678,-0.224350,-0.838398,0.603223,1.617000,-0.006851
6,2014-09-30,0.580786,-0.636705,-0.447331,-0.756226,0.153463,-0.033900,-0.098364
7,2014-12-31,-0.790414,-0.702796,-0.593416,-0.665641,0.270628,-0.905235,0.125558
8,2015-03-31,-0.154720,0.082161,-0.634798,-0.724660,0.391571,-0.043988,0.190935
9,2015-06-30,-0.249524,-0.026416,-0.624944,-0.593182,0.096771,-0.167284,0.081341


In [197]:
# ── Component correlation check ──────────────────────────────────────────────

component_corr = pressure_components.drop(columns="Quarter_End").corr().round(3)

component_corr

,z_CPI_Inflation_YoY,z_India_VIX_Avg,z_IIP_Growth_YoY,z_GNPA_Ratio,z_Credit_Deposit_Ratio,Inflation_Uncertainty_Pressure_z,Growth_Banking_Stress_z
z_CPI_Inflation_YoY,1.000,0.360,-0.039,-0.541,0.168,0.825,-0.211
z_India_VIX_Avg,0.360,1.000,-0.148,0.023,-0.148,0.825,0.014
z_IIP_Growth_YoY,-0.039,-0.148,1.000,0.046,-0.237,-0.113,-0.754
z_GNPA_Ratio,-0.541,0.023,0.046,1.000,-0.442,-0.314,0.324
z_Credit_Deposit_Ratio,0.168,-0.148,-0.237,-0.442,1.000,0.012,0.502
Inflation_Uncertainty_Pressure_z,0.825,0.825,-0.113,-0.314,0.012,1.000,-0.120
Growth_Banking_Stress_z,-0.211,0.014,-0.754,0.324,0.502,-0.120,1.000


In [198]:
# ── Create quarterly Repo-rate changes ───────────────────────────────────────

policy_model_df = pressure_df[
    [
        "Quarter_End",
        "Repo_Rate",
        "Bank_Rate",
        "Inflation_Uncertainty_Pressure_z",
        "Growth_Banking_Stress_z",
        "WALR",
        "MSE_Credit_Growth_YoY"
    ]
].copy()

policy_model_df = policy_model_df.sort_values("Quarter_End").reset_index(drop=True)

policy_model_df["d_Repo"] = policy_model_df["Repo_Rate"].diff()
policy_model_df["d_Bank_Rate"] = policy_model_df["Bank_Rate"].diff()

policy_model_df["Inflation_Uncertainty_L1"] = (
    policy_model_df["Inflation_Uncertainty_Pressure_z"].shift(1)
)

policy_model_df["Growth_Banking_Stress_L1"] = (
    policy_model_df["Growth_Banking_Stress_z"].shift(1)
)

policy_model_df = policy_model_df.dropna().reset_index(drop=True)

policy_model_df.head()

,Quarter_End,Repo_Rate,Bank_Rate,Inflation_Uncertainty_Pressure_z,Growth_Banking_Stress_z,WALR,MSE_Credit_Growth_YoY,d_Repo,d_Bank_Rate,Inflation_Uncertainty_L1,Growth_Banking_Stress_L1
0,2013-06-30,7.25,8.25,1.149481,-0.185350,11.40,22.630200,-0.25,-0.25,1.144304,0.544031
1,2013-09-30,7.50,9.50,2.249405,0.345765,11.96,21.847790,0.25,1.25,1.149481,-0.185350
2,2013-12-31,7.75,8.75,1.996275,-0.225312,11.70,22.894464,0.25,-0.75,2.249405,0.345765
3,2014-03-31,8.00,9.00,0.786207,-0.123087,11.57,25.879074,0.25,0.25,1.996275,-0.225312
4,2014-06-30,8.00,9.00,1.617000,-0.006851,11.60,19.548574,0.00,0.00,0.786207,-0.123087


In [199]:
# ── Historical Repo policy reaction function ─────────────────────────────────

repo_reaction_vars = [
    "Inflation_Uncertainty_L1",
    "Growth_Banking_Stress_L1"
]

y_repo_reaction = policy_model_df["d_Repo"]
X_repo_reaction = sm.add_constant(policy_model_df[repo_reaction_vars])

repo_reaction_model = sm.OLS(
    y_repo_reaction,
    X_repo_reaction
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(repo_reaction_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.168
Model:                            OLS   Adj. R-squared:                  0.130
Method:                 Least Squares   F-statistic:                     2.225
Date:                Tue, 23 Jun 2026   Prob (F-statistic):              0.120
Time:                        15:34:14   Log-Likelihood:                -6.2822
No. Observations:                  47   AIC:                             18.56
Df Residuals:                      44   BIC:                             24.11
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                   

In [200]:
# ── Test whether Repo-rate changes are serially persistent ───────────────────

policy_model_df["d_Repo_L1"] = policy_model_df["d_Repo"].shift(1)

repo_reaction_dynamic_df = policy_model_df.dropna().copy()

repo_reaction_dynamic_vars = [
    "d_Repo_L1",
    "Inflation_Uncertainty_L1",
    "Growth_Banking_Stress_L1"
]

y_repo_dynamic = repo_reaction_dynamic_df["d_Repo"]
X_repo_dynamic = sm.add_constant(
    repo_reaction_dynamic_df[repo_reaction_dynamic_vars]
)

repo_reaction_dynamic_model = sm.OLS(
    y_repo_dynamic,
    X_repo_dynamic
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(repo_reaction_dynamic_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.457
Model:                            OLS   Adj. R-squared:                  0.418
Method:                 Least Squares   F-statistic:                     16.52
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           3.07e-07
Time:                        15:37:17   Log-Likelihood:                 3.4688
No. Observations:                  46   AIC:                             1.062
Df Residuals:                      42   BIC:                             8.377
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                   

## The dynamic policy-reaction model shows that RBI rate decisions were strongly persistent across quarters. Even after accounting for this policy inertia, inflation and market uncertainty remained a statistically significant predictor of subsequent Repo-rate changes. In contrast, weaker industrial activity and banking-sector stress were associated with an easing-oriented policy response, but this relationship was not estimated precisely over the available sample.

## RBI Policy-Pressure Indicators

We use two separate quarterly indicators because RBI can face opposite pressures.

**Inflation–Uncertainty Pressure**  
= Standardised CPI Inflation + Standardised India VIX

Higher inflation and market uncertainty reduce RBI’s room to cut interest rates and support a cautious policy stance.

**Growth–Banking Stress**  
= Negative of Standardised IIP Growth + Standardised GNPA Ratio + Standardised Credit–Deposit Ratio

Lower industrial growth, higher bad loans, and tighter bank funding conditions increase pressure for policy support and easier credit conditions.

The two indicators remain separate because inflation pressure discourages easing, while growth and banking stress encourage it.

In [201]:
# ── Individual-variable RBI reaction function ────────────────────────────────

individual_policy_df = pressure_df[
    [
        "Quarter_End",
        "Repo_Rate",
        "CPI_Inflation_YoY",
        "IIP_Growth_YoY",
        "India_VIX_Avg",
        "GNPA_Ratio",
        "Credit_Deposit_Ratio"
    ]
].copy()

individual_policy_df = (
    individual_policy_df
    .sort_values("Quarter_End")
    .reset_index(drop=True)
)

# Quarterly Repo-rate change
individual_policy_df["d_Repo"] = individual_policy_df["Repo_Rate"].diff()
individual_policy_df["d_Repo_L1"] = individual_policy_df["d_Repo"].shift(1)

# Lag all explanatory variables by one quarter
for var in [
    "CPI_Inflation_YoY",
    "IIP_Growth_YoY",
    "India_VIX_Avg",
    "GNPA_Ratio",
    "Credit_Deposit_Ratio"
]:
    individual_policy_df[f"{var}_L1"] = individual_policy_df[var].shift(1)

individual_policy_df = individual_policy_df.dropna().reset_index(drop=True)

individual_policy_df.head()

,Quarter_End,Repo_Rate,CPI_Inflation_YoY,IIP_Growth_YoY,India_VIX_Avg,GNPA_Ratio,Credit_Deposit_Ratio,d_Repo,d_Repo_L1,CPI_Inflation_YoY_L1,IIP_Growth_YoY_L1,India_VIX_Avg_L1,GNPA_Ratio_L1,Credit_Deposit_Ratio_L1
0,2013-09-30,7.50,10.078605,0.538598,23.530681,4.218281,78.35,0.25,-0.25,9.217051,0.836745,17.152349,4.004487,76.42
1,2013-12-31,7.75,10.574060,4.379646,20.370393,4.397677,76.81,0.25,0.25,10.078605,0.538598,23.530681,4.218281,78.35
2,2014-03-31,8.00,8.312830,5.405573,16.557519,4.115068,77.79,0.25,0.25,10.574060,4.379646,20.370393,4.397677,76.81
3,2014-06-30,8.00,7.859486,1.977220,24.068730,4.319126,77.16,0.00,0.25,8.312830,5.405573,16.557519,4.115068,77.79
4,2014-09-30,8.00,6.681568,-0.234349,14.214290,4.539415,75.97,0.00,0.00,7.859486,1.977220,24.068730,4.319126,77.16


In [202]:
# ── Repo reaction function using individual variables ────────────────────────

repo_individual_vars = [
    "d_Repo_L1",
    "CPI_Inflation_YoY_L1",
    "IIP_Growth_YoY_L1",
    "India_VIX_Avg_L1",
    "GNPA_Ratio_L1",
    "Credit_Deposit_Ratio_L1"
]

y_repo_individual = individual_policy_df["d_Repo"]

X_repo_individual = sm.add_constant(
    individual_policy_df[repo_individual_vars]
)

repo_individual_model = sm.OLS(
    y_repo_individual,
    X_repo_individual
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(repo_individual_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.492
Model:                            OLS   Adj. R-squared:                  0.414
Method:                 Least Squares   F-statistic:                     24.87
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           6.65e-12
Time:                        15:48:50   Log-Likelihood:                 5.0066
No. Observations:                  46   AIC:                             3.987
Df Residuals:                      39   BIC:                             16.79
Df Model:                           6                                         
Covariance Type:                  HAC                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [203]:
repo_individual_results = pd.DataFrame({
    "Variable": repo_individual_model.params.index,
    "Coefficient": repo_individual_model.params.values,
    "HAC Std. Error": repo_individual_model.bse.values,
    "z-statistic": repo_individual_model.tvalues.values,
    "P-value": repo_individual_model.pvalues.values,
    "95% CI Lower": repo_individual_model.conf_int().iloc[:, 0].values,
    "95% CI Upper": repo_individual_model.conf_int().iloc[:, 1].values
})

repo_individual_results["Significance"] = repo_individual_results["P-value"].apply(
    lambda p: "***" if p < 0.01
    else "**" if p < 0.05
    else "*" if p < 0.10
    else ""
)

repo_individual_results.round(4)

,Variable,Coefficient,HAC Std. Error,z-statistic,P-value,95% CI Lower,95% CI Upper,Significance
0,const,2.1583,0.8890,2.4278,0.0152,0.4159,3.9008,**
1,d_Repo_L1,0.5197,0.0992,5.2373,0.0000,0.3252,0.7142,***
2,CPI_Inflation_YoY_L1,0.0309,0.0237,1.3032,0.1925,-0.0156,0.0775,
3,IIP_Growth_YoY_L1,-0.0017,0.0016,-1.0375,0.2995,-0.0049,0.0015,
4,India_VIX_Avg_L1,0.0072,0.0075,0.9501,0.3421,-0.0076,0.0219,
5,GNPA_Ratio_L1,-0.0127,0.0208,-0.6084,0.5429,-0.0535,0.0281,
6,Credit_Deposit_Ratio_L1,-0.0313,0.0115,-2.7222,0.0065,-0.0539,-0.0088,***


In [204]:
from statsmodels.stats.outliers_influence import variance_inflation_factor

vif_df = pd.DataFrame({
    "Variable": repo_individual_vars,
    "VIF": [
        variance_inflation_factor(
            individual_policy_df[repo_individual_vars].values,
            i
        )
        for i in range(len(repo_individual_vars))
    ]
})

vif_df.sort_values("VIF", ascending=False).round(3)

,Variable,VIF
5,Credit_Deposit_Ratio_L1,30.928
3,India_VIX_Avg_L1,21.991
1,CPI_Inflation_YoY_L1,21.429
4,GNPA_Ratio_L1,10.207
0,d_Repo_L1,1.294
2,IIP_Growth_YoY_L1,1.249


In [205]:
inflation_vars = [
    "d_Repo_L1",
    "CPI_Inflation_YoY_L1"
]

inflation_model = sm.OLS(
    individual_policy_df["d_Repo"],
    sm.add_constant(individual_policy_df[inflation_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(inflation_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.419
Model:                            OLS   Adj. R-squared:                  0.392
Method:                 Least Squares   F-statistic:                     20.82
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           4.76e-07
Time:                        15:50:32   Log-Likelihood:                 1.9273
No. Observations:                  46   AIC:                             2.145
Df Residuals:                      43   BIC:                             7.631
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -0.2411 

In [206]:
activity_vars = [
    "d_Repo_L1",
    "IIP_Growth_YoY_L1"
]

activity_model = sm.OLS(
    individual_policy_df["d_Repo"],
    sm.add_constant(individual_policy_df[activity_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(activity_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.359
Model:                            OLS   Adj. R-squared:                  0.329
Method:                 Least Squares   F-statistic:                     24.02
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           9.92e-08
Time:                        15:50:47   Log-Likelihood:               -0.33463
No. Observations:                  46   AIC:                             6.669
Df Residuals:                      43   BIC:                             12.16
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                        coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                -0.0007      0.03

In [207]:
vix_vars = [
    "d_Repo_L1",
    "India_VIX_Avg_L1"
]

vix_model = sm.OLS(
    individual_policy_df["d_Repo"],
    sm.add_constant(individual_policy_df[vix_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(vix_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.420
Model:                            OLS   Adj. R-squared:                  0.393
Method:                 Least Squares   F-statistic:                     22.04
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           2.58e-07
Time:                        15:51:00   Log-Likelihood:                 1.9678
No. Observations:                  46   AIC:                             2.064
Df Residuals:                      43   BIC:                             7.550
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                       coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------
const               -0.2822      0.097  

In [208]:
gnpa_vars = [
    "d_Repo_L1",
    "GNPA_Ratio_L1"
]

gnpa_model = sm.OLS(
    individual_policy_df["d_Repo"],
    sm.add_constant(individual_policy_df[gnpa_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(gnpa_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.367
Model:                            OLS   Adj. R-squared:                  0.338
Method:                 Least Squares   F-statistic:                     23.45
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           1.30e-07
Time:                        15:51:10   Log-Likelihood:              -0.055185
No. Observations:                  46   AIC:                             6.110
Df Residuals:                      43   BIC:                             11.60
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                    coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.0681      0.084      0.813

In [209]:
cdr_vars = [
    "d_Repo_L1",
    "Credit_Deposit_Ratio_L1"
]

cdr_model = sm.OLS(
    individual_policy_df["d_Repo"],
    sm.add_constant(individual_policy_df[cdr_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

print(cdr_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.398
Model:                            OLS   Adj. R-squared:                  0.370
Method:                 Least Squares   F-statistic:                     32.44
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           2.58e-09
Time:                        15:51:21   Log-Likelihood:                 1.0829
No. Observations:                  46   AIC:                             3.834
Df Residuals:                      43   BIC:                             9.320
Df Model:                           2                                         
Covariance Type:                  HAC                                         
                              coef    std err          z      P>|z|      [0.025      0.975]
-------------------------------------------------------------------------------------------
const                     

In [210]:
reaction_models = {
    "Inflation": inflation_model,
    "Activity / IIP": activity_model,
    "Uncertainty / VIX": vix_model,
    "Asset Quality / GNPA": gnpa_model,
    "Funding Pressure / CDR": cdr_model
}

reaction_comparison = []

for model_name, model in reaction_models.items():
    focal_variable = [
        var for var in model.params.index
        if var not in ["const", "d_Repo_L1"]
    ][0]

    reaction_comparison.append({
        "Channel": model_name,
        "Variable": focal_variable,
        "Coefficient": model.params[focal_variable],
        "P-value": model.pvalues[focal_variable],
        "Adjusted R-squared": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic
    })

reaction_comparison = pd.DataFrame(reaction_comparison)

reaction_comparison["Significance"] = reaction_comparison["P-value"].apply(
    lambda p: "***" if p < 0.01
    else "**" if p < 0.05
    else "*" if p < 0.10
    else ""
)

reaction_comparison.sort_values("P-value").round(4)

,Channel,Variable,Coefficient,P-value,Adjusted R-squared,AIC,BIC,Significance
0,Inflation,CPI_Inflation_YoY_L1,0.0432,0.0003,0.3923,2.1454,7.6313,***
2,Uncertainty / VIX,India_VIX_Avg_L1,0.0162,0.0064,0.3934,2.0644,7.5503,***
4,Funding Pressure / CDR,Credit_Deposit_Ratio_L1,-0.0233,0.0590,0.3696,3.8342,9.3201,*
3,Asset Quality / GNPA,GNPA_Ratio_L1,-0.0107,0.3407,0.3376,6.1104,11.5963,
1,Activity / IIP,IIP_Growth_YoY_L1,-0.0006,0.7504,0.3295,6.6693,12.1552,


## ok Time for the FInalyzing

In [211]:
# ── Final compact RBI reaction model ─────────────────────────────────────────

final_policy_df = pressure_df[
    [
        "Quarter_End",
        "Repo_Rate",
        "Inflation_Uncertainty_Pressure_z",
        "Credit_Deposit_Ratio"
    ]
].copy()

final_policy_df = (
    final_policy_df
    .sort_values("Quarter_End")
    .reset_index(drop=True)
)

# Repo-rate change and policy persistence
final_policy_df["d_Repo"] = final_policy_df["Repo_Rate"].diff()
final_policy_df["d_Repo_L1"] = final_policy_df["d_Repo"].shift(1)

# Lagged inflation-uncertainty environment
final_policy_df["Inflation_Uncertainty_L1"] = (
    final_policy_df["Inflation_Uncertainty_Pressure_z"].shift(1)
)

# Standardise CDR before creating lags
cdr_mean = final_policy_df["Credit_Deposit_Ratio"].mean()
cdr_std = final_policy_df["Credit_Deposit_Ratio"].std()

final_policy_df["CDR_z"] = (
    (final_policy_df["Credit_Deposit_Ratio"] - cdr_mean) / cdr_std
)

final_policy_df["CDR_L1"] = final_policy_df["CDR_z"].shift(1)
final_policy_df["CDR_L2"] = final_policy_df["CDR_z"].shift(2)

final_policy_df = final_policy_df.dropna().reset_index(drop=True)

final_policy_df.head()

,Quarter_End,Repo_Rate,Inflation_Uncertainty_Pressure_z,Credit_Deposit_Ratio,d_Repo,d_Repo_L1,Inflation_Uncertainty_L1,CDR_z,CDR_L1,CDR_L2
0,2013-09-30,7.50,2.249405,78.35,0.25,-0.25,1.149481,1.041957,0.320153,0.884880
1,2013-12-31,7.75,1.996275,76.81,0.25,0.25,2.249405,0.466009,1.041957,0.320153
2,2014-03-31,8.00,0.786207,77.79,0.25,0.25,1.996275,0.832521,0.466009,1.041957
3,2014-06-30,8.00,1.617000,77.16,0.00,0.25,0.786207,0.596906,0.832521,0.466009
4,2014-09-30,8.00,-0.033900,75.97,0.00,0.00,1.617000,0.151856,0.596906,0.832521


In [212]:
# ── Compare one-lag versus two-lag CDR models ────────────────────────────────

base_vars = [
    "d_Repo_L1",
    "Inflation_Uncertainty_L1"
]

cdr_lag1_vars = base_vars + ["CDR_L1"]
cdr_lag2_vars = base_vars + ["CDR_L1", "CDR_L2"]

policy_model_cdr_l1 = sm.OLS(
    final_policy_df["d_Repo"],
    sm.add_constant(final_policy_df[cdr_lag1_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

policy_model_cdr_l2 = sm.OLS(
    final_policy_df["d_Repo"],
    sm.add_constant(final_policy_df[cdr_lag2_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 4}
)

policy_lag_comparison = pd.DataFrame({
    "Model": [
        "Inflation-Uncertainty + CDR Lag 1",
        "Inflation-Uncertainty + CDR Lags 1 and 2"
    ],
    "Observations": [
        int(policy_model_cdr_l1.nobs),
        int(policy_model_cdr_l2.nobs)
    ],
    "Adjusted R-squared": [
        policy_model_cdr_l1.rsquared_adj,
        policy_model_cdr_l2.rsquared_adj
    ],
    "AIC": [
        policy_model_cdr_l1.aic,
        policy_model_cdr_l2.aic
    ],
    "BIC": [
        policy_model_cdr_l1.bic,
        policy_model_cdr_l2.bic
    ]
})

policy_lag_comparison.round(4)

,Model,Observations,Adjusted R-squared,AIC,BIC
0,Inflation-Uncertainty + CDR Lag 1,46,0.4411,-0.7908,6.5238
1,Inflation-Uncertainty + CDR Lags 1 and 2,46,0.4313,0.8999,10.0431


In [213]:
final_repo_reaction_model = policy_model_cdr_l1

print(final_repo_reaction_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.478
Model:                            OLS   Adj. R-squared:                  0.441
Method:                 Least Squares   F-statistic:                     21.63
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           1.26e-08
Time:                        15:56:50   Log-Likelihood:                 4.3954
No. Observations:                  46   AIC:                           -0.7908
Df Residuals:                      42   BIC:                             6.524
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                   

### Inflation and uncertainty push RBI toward caution or tightening, while higher banking-system funding pressure, proxied by the Credit–Deposit Ratio, creates a weaker but meaningful counter-pressure toward easing or liquidity support.

In [214]:
# ── Monthly RBI policy-pressure panel: Jan-2021 onward ───────────────────────

START_MONTH = pd.Timestamp("2021-01-01")

def ensure_month_column(df, date_candidates=("Date", "Month", "Datetime", "index")):
    out = df.copy()

    if not any(col in out.columns for col in date_candidates):
        out = out.reset_index()

    date_col = next((col for col in date_candidates if col in out.columns), None)

    if date_col is None:
        raise ValueError(
            f"Could not find a date column. Available columns: {out.columns.tolist()}"
        )

    out["Month"] = pd.to_datetime(out[date_col]).dt.to_period("M")

    return out


# Macro / external data already created earlier
macro_policy_m = ensure_month_column(macro_monthly)

# Repo and Bank Rate source data
repo_policy_m = ensure_month_column(repo)
bank_policy_m = ensure_month_column(bank_rate)

repo_policy_m = (
    repo_policy_m
    .groupby("Month", as_index=False)["Repo_rate"]
    .last()
)

bank_policy_m = (
    bank_policy_m
    .groupby("Month", as_index=False)["Bank_rate"]
    .last()
)

monthly_policy_df = (
    macro_policy_m[
        [
            "Month",
            "CPI_Inflation_YoY",
            "India_VIX_Avg",
            "USDINR",
            "Brent_USD"
        ]
    ]
    .merge(repo_policy_m, on="Month", how="left")
    .merge(bank_policy_m, on="Month", how="left")
    .sort_values("Month")
    .reset_index(drop=True)
)

monthly_policy_df["Date"] = monthly_policy_df["Month"].dt.to_timestamp("M")

monthly_policy_df = (
    monthly_policy_df
    .loc[monthly_policy_df["Date"] >= START_MONTH]
    .sort_values("Date")
    .reset_index(drop=True)
)

print(
    "Monthly sample:",
    monthly_policy_df["Date"].min().date(),
    "to",
    monthly_policy_df["Date"].max().date()
)

monthly_policy_df.head()

Monthly sample: 2021-01-31 to 2024-10-31


,Month,CPI_Inflation_YoY,India_VIX_Avg,USDINR,Brent_USD,Repo_rate,Bank_rate,Date
0,2021-01,4.061252,22.589444,73.155171,55.351579,4.0,4.25,2021-01-31
1,2021-02,5.030181,23.650000,72.766445,62.227894,4.0,4.25,2021-02-28
2,2021-03,5.518170,21.925714,72.815147,65.702174,4.0,4.25,2021-03-31
3,2021-04,4.227213,21.752105,74.469327,65.328572,4.0,4.25,2021-04-30
4,2021-05,6.295560,20.313333,73.268552,68.258500,4.0,4.25,2021-05-31


In [215]:
# ── Monthly policy-pressure transformations ──────────────────────────────────

monthly_policy_df["USDINR_3M_Change"] = (
    monthly_policy_df["USDINR"]
    .pct_change(3)
    .mul(100)
)

monthly_policy_df["Brent_3M_Change"] = (
    monthly_policy_df["Brent_USD"]
    .pct_change(3)
    .mul(100)
)

# Smooth VIX slightly instead of using one volatile monthly reading
monthly_policy_df["India_VIX_3M_Avg"] = (
    monthly_policy_df["India_VIX_Avg"]
    .rolling(3)
    .mean()
)

# Policy-rate movements
monthly_policy_df["d_Repo"] = monthly_policy_df["Repo_rate"].diff()
monthly_policy_df["d_Bank_Rate"] = monthly_policy_df["Bank_rate"].diff()

monthly_policy_df[
    [
        "Date",
        "CPI_Inflation_YoY",
        "India_VIX_3M_Avg",
        "USDINR_3M_Change",
        "Brent_3M_Change",
        "Repo_rate",
        "Bank_rate",
        "d_Repo"
    ]
].tail(12)

,Date,CPI_Inflation_YoY,India_VIX_3M_Avg,USDINR_3M_Change,Brent_3M_Change,Repo_rate,Bank_rate,d_Repo
34,2023-11-30,5.552408,11.298833,0.657803,-3.615115,6.5,6.75,0.0
35,2023-12-31,5.691520,12.070500,0.213321,-16.646634,6.5,6.75,0.0
36,2024-01-31,5.099150,13.058667,-0.112600,-10.717598,6.5,6.75,0.0
37,2024-02-29,5.090498,14.371833,-0.457981,-0.492185,6.5,6.75,0.0
38,2024-03-31,4.853273,14.430111,-0.226441,9.494465,6.5,6.75,0.0
39,2024-04-30,4.828748,13.651611,0.337071,12.377790,6.5,6.75,0.0
40,2024-05-31,4.860335,15.190881,0.474231,1.675756,6.5,6.75,0.0
41,2024-06-30,5.082873,15.815998,0.554709,-2.097697,6.5,6.75,0.0
42,2024-07-31,3.596350,16.416589,0.207772,-5.935137,6.5,6.75,0.0
43,2024-08-31,3.651987,14.669128,0.618732,-4.960510,6.5,6.75,0.0


In [216]:
# ── Rebuild monthly current-policy panel WITHOUT IIP ─────────────────────────

START_MONTH = pd.Timestamp("2021-01-01")

# CPI already downloaded separately
cpi_current = cpi_monthly.copy()
cpi_current["Month"] = pd.to_datetime(cpi_current["Date"]).dt.to_period("M")

# India VIX
vix_current = india_vix_monthly.copy()
vix_current["Month"] = pd.to_datetime(vix_current["Date"]).dt.to_period("M")

# USD/INR + Brent
external_current = external_monthly.copy()

if "Date" not in external_current.columns:
    external_current = external_current.reset_index()

date_col_ext = next(
    (c for c in ["Date", "Datetime", "date", "index"] if c in external_current.columns),
    None
)

external_current["Month"] = pd.to_datetime(
    external_current[date_col_ext]
).dt.to_period("M")

# Repo
repo_current = repo.copy()
repo_current["Month"] = pd.to_datetime(repo_current["Month"]).dt.to_period("M")

repo_current = (
    repo_current
    .groupby("Month", as_index=False)["Repo_rate"]
    .last()
)

# Bank Rate
bank_current = bank_rate.copy()
bank_current["Month"] = pd.to_datetime(bank_current["Month"]).dt.to_period("M")

bank_current = (
    bank_current
    .groupby("Month", as_index=False)["Bank_rate"]
    .last()
)

# Final current-policy panel: intentionally NO IIP merge
monthly_policy_df = (
    cpi_current[["Month", "CPI_Inflation_YoY"]]
    .merge(
        vix_current[["Month", "India_VIX_Avg"]],
        on="Month",
        how="inner"
    )
    .merge(
        external_current[["Month", "USDINR", "Brent_USD"]],
        on="Month",
        how="inner"
    )
    .merge(repo_current, on="Month", how="left")
    .merge(bank_current, on="Month", how="left")
    .sort_values("Month")
    .reset_index(drop=True)
)

monthly_policy_df["Date"] = monthly_policy_df["Month"].dt.to_timestamp("M")

monthly_policy_df = (
    monthly_policy_df[
        monthly_policy_df["Date"] >= START_MONTH
    ]
    .sort_values("Date")
    .reset_index(drop=True)
)

print(
    "Monthly current-policy sample:",
    monthly_policy_df["Date"].min().date(),
    "to",
    monthly_policy_df["Date"].max().date()
)

monthly_policy_df.tail(12)

Monthly current-policy sample: 2021-01-31 to 2025-06-30


,Month,CPI_Inflation_YoY,India_VIX_Avg,USDINR,Brent_USD,Repo_rate,Bank_rate,Date
42,2024-07,3.596350,13.542273,83.582601,83.717727,6.50,6.75,2024-07-31
43,2024-08,3.651987,14.901429,83.888191,78.875001,6.50,6.75,2024-08-31
44,2024-09,5.486149,13.286667,83.809434,72.638499,6.50,6.75,2024-09-30
45,2024-10,6.206152,13.933182,84.023826,75.376957,6.50,6.75,2024-10-31
46,2024-11,5.475040,15.256316,84.348214,73.407000,6.50,6.75,2024-11-30
47,2024-12,5.223479,14.021429,84.981977,73.128572,6.50,6.75,2024-12-31
48,2025-01,4.258760,15.995455,86.261955,78.263334,6.50,6.75,2025-01-31
49,2025-02,3.606028,14.522105,86.964145,74.940526,6.25,6.50,2025-02-28
50,2025-03,3.336921,13.443684,86.641128,71.467142,6.25,6.50,2025-03-31
51,2025-04,3.160150,16.828948,85.646155,66.456667,6.00,6.25,2025-04-30


## CPI aaaya

In [217]:
# ── Load updated CPI index and calculate YoY inflation ───────────────────────

cpi_updated = pd.read_excel("CPI.xlsx")

cpi_updated["Month"] = pd.to_datetime(cpi_updated["Month"])
cpi_updated = cpi_updated.sort_values("Month").reset_index(drop=True)

# Headline CPI inflation, year-on-year
cpi_updated["CPI_Inflation_YoY"] = (
    cpi_updated["CPI"]
    .pct_change(12)
    .mul(100)
)

cpi_updated = cpi_updated.rename(columns={"Month": "Date"})

cpi_updated = cpi_updated[
    ["Date", "CPI", "CPI_Inflation_YoY"]
].dropna().reset_index(drop=True)

print(
    "Updated CPI inflation sample:",
    cpi_updated["Date"].min().date(),
    "to",
    cpi_updated["Date"].max().date()
)

cpi_updated.tail(15)

Updated CPI inflation sample: 2014-01-01 to 2026-05-01


,Date,CPI,CPI_Inflation_YoY
134,2025-03-01,101.39,3.564862
135,2025-04-01,101.58,3.336724
136,2025-05-01,101.90,3.033367
137,2025-06-01,102.51,2.305389
138,2025-07-01,103.35,1.622419
139,2025-08-01,103.74,2.005900
140,2025-09-01,103.74,1.407625
141,2025-10-01,103.74,0.038573
142,2025-11-01,104.01,0.492754
143,2025-12-01,104.10,1.166181


In [218]:
# ── Rebuild current RBI monthly panel using updated CPI ──────────────────────

cpi_current = cpi_updated.copy()
cpi_current["Month"] = cpi_current["Date"].dt.to_period("M")

vix_current = india_vix_monthly.copy()

if "Date" not in vix_current.columns:
    vix_current = vix_current.reset_index()

vix_date_col = next(
    c for c in ["Date", "Datetime", "date", "index"]
    if c in vix_current.columns
)

vix_current["Month"] = pd.to_datetime(
    vix_current[vix_date_col]
).dt.to_period("M")

external_current = external_monthly.copy()

if "Date" not in external_current.columns:
    external_current = external_current.reset_index()

external_date_col = next(
    c for c in ["Date", "Datetime", "date", "index"]
    if c in external_current.columns
)

external_current["Month"] = pd.to_datetime(
    external_current[external_date_col]
).dt.to_period("M")

repo_current = repo.copy()
repo_current["Month"] = pd.to_datetime(repo_current["Month"]).dt.to_period("M")

repo_current = (
    repo_current
    .groupby("Month", as_index=False)["Repo_rate"]
    .last()
)

bank_current = bank_rate.copy()
bank_current["Month"] = pd.to_datetime(bank_current["Month"]).dt.to_period("M")

bank_current = (
    bank_current
    .groupby("Month", as_index=False)["Bank_rate"]
    .last()
)

monthly_policy_df = (
    cpi_current[["Month", "CPI_Inflation_YoY"]]
    .merge(
        vix_current[["Month", "India_VIX_Avg"]],
        on="Month",
        how="inner"
    )
    .merge(
        external_current[["Month", "USDINR", "Brent_USD"]],
        on="Month",
        how="inner"
    )
    .merge(repo_current, on="Month", how="left")
    .merge(bank_current, on="Month", how="left")
    .sort_values("Month")
    .reset_index(drop=True)
)

monthly_policy_df["Date"] = monthly_policy_df["Month"].dt.to_timestamp("M")

monthly_policy_df = monthly_policy_df[
    monthly_policy_df["Date"] >= pd.Timestamp("2021-01-01")
].copy()

print(
    "Updated monthly sample:",
    monthly_policy_df["Date"].min().date(),
    "to",
    monthly_policy_df["Date"].max().date()
)

monthly_policy_df.tail(15)

Updated monthly sample: 2021-01-31 to 2026-05-31


,Month,CPI_Inflation_YoY,India_VIX_Avg,USDINR,Brent_USD,Repo_rate,Bank_rate,Date
134,2025-03,3.564862,13.443684,86.641128,71.467142,6.25,6.50,2025-03-31
135,2025-04,3.336724,16.828948,85.646155,66.456667,6.00,6.25,2025-04-30
136,2025-05,3.033367,18.022857,85.203787,63.970476,6.00,6.25,2025-05-31
137,2025-06,2.305389,14.310952,85.928105,69.350000,5.50,5.75,2025-06-30
138,2025-07,1.622419,11.653043,86.095682,69.553044,5.50,5.75,2025-07-31
139,2025-08,2.005900,11.957368,87.496467,67.261428,5.50,5.75,2025-08-31
140,2025-09,1.407625,10.679091,88.258800,67.548571,5.50,5.75,2025-09-30
141,2025-10,0.038573,11.061905,88.380448,63.953913,5.50,5.75,2025-10-31
142,2025-11,0.492754,12.304211,88.810835,63.679474,5.50,5.75,2025-11-30
143,2025-12,1.166181,10.198636,90.008018,61.628636,5.25,5.50,2025-12-31


In [219]:
print(monthly_policy_df["Date"].min())
print(monthly_policy_df["Date"].max())
print("Observations:", len(monthly_policy_df))

2021-01-31 00:00:00
2026-05-31 00:00:00
Observations: 65


In [220]:
monthly_policy_df = monthly_policy_df.sort_values("Date").reset_index(drop=True)

monthly_policy_df["USDINR_3M_Change"] = (
    monthly_policy_df["USDINR"].pct_change(3) * 100
)

monthly_policy_df["Brent_3M_Change"] = (
    monthly_policy_df["Brent_USD"].pct_change(3) * 100
)

monthly_policy_df["India_VIX_3M_Avg"] = (
    monthly_policy_df["India_VIX_Avg"]
    .rolling(3)
    .mean()
)

monthly_policy_df["d_Repo"] = monthly_policy_df["Repo_rate"].diff()
monthly_policy_df["d_Bank_Rate"] = monthly_policy_df["Bank_rate"].diff()

monthly_model_df = monthly_policy_df.copy()

monthly_model_df["d_Repo_L1"] = monthly_model_df["d_Repo"].shift(1)
monthly_model_df["d_Bank_Rate_L1"] = monthly_model_df["d_Bank_Rate"].shift(1)

for var in [
    "CPI_Inflation_YoY",
    "India_VIX_3M_Avg",
    "USDINR_3M_Change",
    "Brent_3M_Change"
]:
    monthly_model_df[f"{var}_L1"] = monthly_model_df[var].shift(1)

monthly_model_df = monthly_model_df.dropna().reset_index(drop=True)

print("Usable monthly observations:", len(monthly_model_df))
monthly_model_df.tail()

Usable monthly observations: 61


,Month,CPI_Inflation_YoY,India_VIX_Avg,USDINR,Brent_USD,Repo_rate,Bank_rate,Date,USDINR_3M_Change,Brent_3M_Change,India_VIX_3M_Avg,d_Repo,d_Bank_Rate,d_Repo_L1,d_Bank_Rate_L1,CPI_Inflation_YoY_L1,India_VIX_3M_Avg_L1,USDINR_3M_Change_L1,Brent_3M_Change_L1
56,2026-01,2.734337,11.952105,90.748358,64.765499,5.25,5.5,2026-01-31,2.679224,1.269017,11.484984,0.0,0.0,-0.25,-0.25,1.166181,11.188251,1.981919,-8.763967
57,2026-02,3.207659,12.908500,90.726594,69.406843,5.25,5.5,2026-02-28,2.157124,8.994058,11.686414,0.0,0.0,0.00,0.00,2.734337,11.484984,2.679224,1.269017
58,2026-03,3.402702,22.106842,92.802195,99.599546,5.25,5.5,2026-03-31,3.104365,61.612446,15.655816,0.0,0.0,0.00,0.00,3.207659,11.686414,2.157124,8.994058
59,2026-04,3.484938,19.970000,93.494664,102.464286,5.25,5.5,2026-04-30,3.026287,58.208132,18.328447,0.0,0.0,0.00,0.00,3.402702,15.655816,3.104365,61.612446
60,2026-05,3.935231,17.762632,95.538276,104.092000,5.25,5.5,2026-05-31,5.303497,49.973685,19.946491,0.0,0.0,0.00,0.00,3.484938,18.328447,3.026287,58.208132


In [221]:
monthly_repo_vars = [
    "d_Repo_L1",
    "CPI_Inflation_YoY_L1",
    "India_VIX_3M_Avg_L1",
    "USDINR_3M_Change_L1",
    "Brent_3M_Change_L1"
]

from statsmodels.stats.outliers_influence import variance_inflation_factor

monthly_vif = pd.DataFrame({
    "Variable": monthly_repo_vars,
    "VIF": [
        variance_inflation_factor(
            monthly_model_df[monthly_repo_vars].values,
            i
        )
        for i in range(len(monthly_repo_vars))
    ]
})

monthly_vif.sort_values("VIF", ascending=False).round(3)

,Variable,VIF
2,India_VIX_3M_Avg_L1,16.370
1,CPI_Inflation_YoY_L1,13.979
3,USDINR_3M_Change_L1,2.366
0,d_Repo_L1,1.375
4,Brent_3M_Change_L1,1.127


In [222]:
# ── Build monthly Inflation–Uncertainty measure ──────────────────────────────

from scipy.stats import zscore

monthly_model_df = monthly_model_df.copy()

monthly_model_df["CPI_z_L1"] = zscore(
    monthly_model_df["CPI_Inflation_YoY_L1"],
    nan_policy="omit"
)

monthly_model_df["VIX_z_L1"] = zscore(
    monthly_model_df["India_VIX_3M_Avg_L1"],
    nan_policy="omit"
)

monthly_model_df["Inflation_Uncertainty_L1"] = (
    monthly_model_df["CPI_z_L1"]
    + monthly_model_df["VIX_z_L1"]
)

monthly_final_vars = [
    "d_Repo_L1",
    "Inflation_Uncertainty_L1",
    "USDINR_3M_Change_L1",
    "Brent_3M_Change_L1"
]

monthly_final_vif = pd.DataFrame({
    "Variable": monthly_final_vars,
    "VIF": [
        variance_inflation_factor(
            monthly_model_df[monthly_final_vars].values,
            i
        )
        for i in range(len(monthly_final_vars))
    ]
})

monthly_final_vif.sort_values("VIF", ascending=False).round(3)

,Variable,VIF
0,d_Repo_L1,1.520
1,Inflation_Uncertainty_L1,1.369
2,USDINR_3M_Change_L1,1.287
3,Brent_3M_Change_L1,1.208


In [223]:
# ── Final monthly Repo reaction model: Jan-2021 onward ───────────────────────

monthly_repo_final_model = sm.OLS(
    monthly_model_df["d_Repo"],
    sm.add_constant(monthly_model_df[monthly_final_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 3}
)

print(monthly_repo_final_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.359
Model:                            OLS   Adj. R-squared:                  0.314
Method:                 Least Squares   F-statistic:                     4.924
Date:                Tue, 23 Jun 2026   Prob (F-statistic):            0.00179
Time:                        16:41:47   Log-Likelihood:                 39.901
No. Observations:                  61   AIC:                            -69.80
Df Residuals:                      56   BIC:                            -59.25
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                   

In [224]:
# ── Final monthly Bank Rate reaction model ───────────────────────────────────

monthly_bank_final_vars = [
    "d_Bank_Rate_L1",
    "Inflation_Uncertainty_L1",
    "USDINR_3M_Change_L1",
    "Brent_3M_Change_L1"
]

monthly_bank_final_model = sm.OLS(
    monthly_model_df["d_Bank_Rate"],
    sm.add_constant(monthly_model_df[monthly_bank_final_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 3}
)

print(monthly_bank_final_model.summary())

                            OLS Regression Results                            
Dep. Variable:            d_Bank_Rate   R-squared:                       0.359
Model:                            OLS   Adj. R-squared:                  0.314
Method:                 Least Squares   F-statistic:                     4.924
Date:                Tue, 23 Jun 2026   Prob (F-statistic):            0.00179
Time:                        16:41:54   Log-Likelihood:                 39.901
No. Observations:                  61   AIC:                            -69.80
Df Residuals:                      56   BIC:                            -59.25
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                   

In [225]:
# ── Check whether Repo and Bank Rate series are actually distinct ────────────

rate_check = monthly_policy_df[
    ["Date", "Repo_rate", "Bank_rate", "d_Repo", "d_Bank_Rate"]
].copy()

print("Repo vs Bank Rate level correlation:")
print(rate_check[["Repo_rate", "Bank_rate"]].corr())

print("\nRepo vs Bank Rate change correlation:")
print(rate_check[["d_Repo", "d_Bank_Rate"]].corr())

print("\nMonths where Repo and Bank Rate changes differ:")
display(
    rate_check[
        rate_check["d_Repo"].fillna(0) != rate_check["d_Bank_Rate"].fillna(0)
    ]
)

print("\nUnique Repo changes:")
print(rate_check["d_Repo"].value_counts(dropna=False).sort_index())

print("\nUnique Bank Rate changes:")
print(rate_check["d_Bank_Rate"].value_counts(dropna=False).sort_index())

Repo vs Bank Rate level correlation:
           Repo_rate  Bank_rate
Repo_rate        1.0        1.0
Bank_rate        1.0        1.0

Repo vs Bank Rate change correlation:
             d_Repo  d_Bank_Rate
d_Repo          1.0          1.0
d_Bank_Rate     1.0          1.0

Months where Repo and Bank Rate changes differ:


,Date,Repo_rate,Bank_rate,d_Repo,d_Bank_Rate



Unique Repo changes:
d_Repo
-0.50     1
-0.25     3
 0.00    54
 0.25     1
 0.35     1
 0.40     1
 0.50     3
 NaN      1
Name: count, dtype: int64

Unique Bank Rate changes:
d_Bank_Rate
-0.50     1
-0.25     3
 0.00    54
 0.25     1
 0.35     1
 0.40     1
 0.50     3
 NaN      1
Name: count, dtype: int64


## Important

In [228]:
# ── Full monthly RBI pressure model: Jan-2013 onward ─────────────────────────

full_monthly_policy_df = monthly_policy_df.copy()

full_monthly_policy_df = (
    full_monthly_policy_df[
        full_monthly_policy_df["Date"] >= pd.Timestamp("2013-01-01")
    ]
    .sort_values("Date")
    .reset_index(drop=True)
)

print(
    "Raw sample:",
    full_monthly_policy_df["Date"].min().date(),
    "to",
    full_monthly_policy_df["Date"].max().date()
)

print("Raw observations:", len(full_monthly_policy_df))

Raw sample: 2021-01-31 to 2026-05-31
Raw observations: 65


In [229]:
# ── Build full monthly RBI policy panel: Jan-2013 onward ─────────────────────

START_MONTH = pd.Timestamp("2013-01-01")

# Updated CPI inflation
cpi_full = cpi_updated.copy()
cpi_full["Month"] = pd.to_datetime(cpi_full["Date"]).dt.to_period("M")

# India VIX
vix_full = india_vix_monthly.copy()

if "Date" not in vix_full.columns:
    vix_full = vix_full.reset_index()

vix_date_col = next(
    c for c in ["Date", "Datetime", "date", "index"]
    if c in vix_full.columns
)

vix_full["Month"] = pd.to_datetime(
    vix_full[vix_date_col]
).dt.to_period("M")

# USD/INR and Brent
external_full = external_monthly.copy()

if "Date" not in external_full.columns:
    external_full = external_full.reset_index()

external_date_col = next(
    c for c in ["Date", "Datetime", "date", "index"]
    if c in external_full.columns
)

external_full["Month"] = pd.to_datetime(
    external_full[external_date_col]
).dt.to_period("M")

# RBI Repo Rate
repo_full = repo.copy()
repo_full["Month"] = pd.to_datetime(repo_full["Month"]).dt.to_period("M")

repo_full = (
    repo_full
    .groupby("Month", as_index=False)["Repo_rate"]
    .last()
)

# RBI Bank Rate
bank_full = bank_rate.copy()
bank_full["Month"] = pd.to_datetime(bank_full["Month"]).dt.to_period("M")

bank_full = (
    bank_full
    .groupby("Month", as_index=False)["Bank_rate"]
    .last()
)

# Full common monthly panel
monthly_policy_df = (
    cpi_full[["Month", "CPI_Inflation_YoY"]]
    .merge(
        vix_full[["Month", "India_VIX_Avg"]],
        on="Month",
        how="inner"
    )
    .merge(
        external_full[["Month", "USDINR", "Brent_USD"]],
        on="Month",
        how="inner"
    )
    .merge(repo_full, on="Month", how="left")
    .merge(bank_full, on="Month", how="left")
    .sort_values("Month")
    .reset_index(drop=True)
)

monthly_policy_df["Date"] = monthly_policy_df["Month"].dt.to_timestamp("M")

monthly_policy_df = (
    monthly_policy_df[
        monthly_policy_df["Date"] >= START_MONTH
    ]
    .sort_values("Date")
    .reset_index(drop=True)
)

print(
    "Full monthly sample:",
    monthly_policy_df["Date"].min().date(),
    "to",
    monthly_policy_df["Date"].max().date()
)

print("Raw observations:", len(monthly_policy_df))

monthly_policy_df.head(), monthly_policy_df.tail()

Full monthly sample: 2014-01-31 to 2026-05-31
Raw observations: 149


(     Month  CPI_Inflation_YoY  India_VIX_Avg     USDINR   Brent_USD  \
 0  2014-01           8.529946      16.244348  61.998652  107.199500   
 1  2014-02           7.747748      16.574210  62.140200  108.706111   
 2  2014-03           8.093525      16.854000  60.996524  107.857619   
 3  2014-04           8.407871      28.496666  60.315227  108.051905   
 4  2014-05           8.348135      26.267143  59.264227  109.219524   
 
    Repo_rate  Bank_rate       Date  
 0        8.0        9.0 2014-01-31  
 1        8.0        9.0 2014-02-28  
 2        8.0        9.0 2014-03-31  
 3        8.0        9.0 2014-04-30  
 4        8.0        9.0 2014-05-31  ,
        Month  CPI_Inflation_YoY  India_VIX_Avg     USDINR   Brent_USD  \
 144  2026-01           2.734337      11.952105  90.748358   64.765499   
 145  2026-02           3.207659      12.908500  90.726594   69.406843   
 146  2026-03           3.402702      22.106842  92.802195   99.599546   
 147  2026-04           3.484938      19.

In [230]:
# ── Full-sample monthly transformations ──────────────────────────────────────

monthly_policy_df = monthly_policy_df.sort_values("Date").reset_index(drop=True)

monthly_policy_df["USDINR_3M_Change"] = (
    monthly_policy_df["USDINR"].pct_change(3) * 100
)

monthly_policy_df["Brent_3M_Change"] = (
    monthly_policy_df["Brent_USD"].pct_change(3) * 100
)

monthly_policy_df["India_VIX_3M_Avg"] = (
    monthly_policy_df["India_VIX_Avg"]
    .rolling(3)
    .mean()
)

monthly_policy_df["d_Repo"] = monthly_policy_df["Repo_rate"].diff()

monthly_model_df = monthly_policy_df.copy()

monthly_model_df["d_Repo_L1"] = monthly_model_df["d_Repo"].shift(1)

for var in [
    "CPI_Inflation_YoY",
    "India_VIX_3M_Avg",
    "USDINR_3M_Change",
    "Brent_3M_Change"
]:
    monthly_model_df[f"{var}_L1"] = monthly_model_df[var].shift(1)

monthly_model_df = monthly_model_df.dropna().reset_index(drop=True)

print(
    "Usable model sample:",
    monthly_model_df["Date"].min().date(),
    "to",
    monthly_model_df["Date"].max().date()
)

print("Usable observations:", len(monthly_model_df))

Usable model sample: 2014-05-31 to 2026-05-31
Usable observations: 145


In [231]:
from scipy.stats import zscore
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import pandas as pd

# ── Build lagged monthly model dataset ───────────────────────────────────────

monthly_policy_df = monthly_policy_df.sort_values("Date").reset_index(drop=True)

monthly_policy_df["USDINR_3M_Change"] = (
    monthly_policy_df["USDINR"].pct_change(3) * 100
)

monthly_policy_df["Brent_3M_Change"] = (
    monthly_policy_df["Brent_USD"].pct_change(3) * 100
)

monthly_policy_df["India_VIX_3M_Avg"] = (
    monthly_policy_df["India_VIX_Avg"].rolling(3).mean()
)

monthly_policy_df["d_Repo"] = monthly_policy_df["Repo_rate"].diff()

monthly_model_df = monthly_policy_df.copy()

monthly_model_df["d_Repo_L1"] = monthly_model_df["d_Repo"].shift(1)

for var in [
    "CPI_Inflation_YoY",
    "India_VIX_3M_Avg",
    "USDINR_3M_Change",
    "Brent_3M_Change"
]:
    monthly_model_df[f"{var}_L1"] = monthly_model_df[var].shift(1)

monthly_model_df = monthly_model_df.dropna().reset_index(drop=True)

# ── Inflation–uncertainty index ──────────────────────────────────────────────

monthly_model_df["CPI_z_L1"] = zscore(
    monthly_model_df["CPI_Inflation_YoY_L1"]
)

monthly_model_df["VIX_z_L1"] = zscore(
    monthly_model_df["India_VIX_3M_Avg_L1"]
)

monthly_model_df["Inflation_Uncertainty_L1"] = (
    monthly_model_df["CPI_z_L1"]
    + monthly_model_df["VIX_z_L1"]
)

monthly_repo_vars = [
    "d_Repo_L1",
    "Inflation_Uncertainty_L1",
    "USDINR_3M_Change_L1",
    "Brent_3M_Change_L1"
]

# ── VIF ──────────────────────────────────────────────────────────────────────

monthly_vif = pd.DataFrame({
    "Variable": monthly_repo_vars,
    "VIF": [
        variance_inflation_factor(
            monthly_model_df[monthly_repo_vars].values,
            i
        )
        for i in range(len(monthly_repo_vars))
    ]
}).sort_values("VIF", ascending=False)

display(monthly_vif.round(3))

# ── Full-sample Repo reaction model ──────────────────────────────────────────

full_monthly_repo_model = sm.OLS(
    monthly_model_df["d_Repo"],
    sm.add_constant(monthly_model_df[monthly_repo_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 3}
)

print("Usable model sample:",
      monthly_model_df["Date"].min().date(),
      "to",
      monthly_model_df["Date"].max().date())

print("Usable observations:", len(monthly_model_df))

print(full_monthly_repo_model.summary())

,Variable,VIF
3,Brent_3M_Change_L1,1.054
2,USDINR_3M_Change_L1,1.035
0,d_Repo_L1,1.027
1,Inflation_Uncertainty_L1,1.010


Usable model sample: 2014-05-31 to 2026-05-31
Usable observations: 145
                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.110
Model:                            OLS   Adj. R-squared:                  0.085
Method:                 Least Squares   F-statistic:                     3.162
Date:                Tue, 23 Jun 2026   Prob (F-statistic):             0.0160
Time:                        16:58:30   Log-Likelihood:                 76.408
No. Observations:                 145   AIC:                            -142.8
Df Residuals:                     140   BIC:                            -127.9
Df Model:                           4                                         
Covariance Type:                  HAC                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------

In [232]:
# ── Parsimonious full-sample Repo model ──────────────────────────────────────

parsimonious_repo_vars = [
    "d_Repo_L1",
    "USDINR_3M_Change_L1",
    "Brent_3M_Change_L1"
]

parsimonious_monthly_repo_model = sm.OLS(
    monthly_model_df["d_Repo"],
    sm.add_constant(monthly_model_df[parsimonious_repo_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 3}
)

model_comparison = pd.DataFrame({
    "Model": [
        "Full: Inflation-Uncertainty + INR + Brent",
        "Parsimonious: INR + Brent"
    ],
    "Adjusted_R2": [
        full_monthly_repo_model.rsquared_adj,
        parsimonious_monthly_repo_model.rsquared_adj
    ],
    "AIC": [
        full_monthly_repo_model.aic,
        parsimonious_monthly_repo_model.aic
    ],
    "BIC": [
        full_monthly_repo_model.bic,
        parsimonious_monthly_repo_model.bic
    ]
})

display(model_comparison.round(4))

print(parsimonious_monthly_repo_model.summary())

,Model,Adjusted_R2,AIC,BIC
0,Full: Inflation-Uncertainty + INR + Brent,0.0847,-142.8160,-127.9323
1,Parsimonious: INR + Brent,0.0719,-141.7705,-129.8636


                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.091
Model:                            OLS   Adj. R-squared:                  0.072
Method:                 Least Squares   F-statistic:                     3.727
Date:                Tue, 23 Jun 2026   Prob (F-statistic):             0.0129
Time:                        17:01:07   Log-Likelihood:                 74.885
No. Observations:                 145   AIC:                            -141.8
Df Residuals:                     141   BIC:                            -129.9
Df Model:                           3                                         
Covariance Type:                  HAC                                         
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.0325    

## Improvements

In [234]:
monthly_model_df["Repo_Hike"] = (
    monthly_model_df["d_Repo"] > 0
).astype(int)

monthly_model_df["Repo_Cut"] = (
    monthly_model_df["d_Repo"] < 0
).astype(int)

monthly_model_df["Repo_Change_Any"] = (
    monthly_model_df["d_Repo"] != 0
).astype(int)

print(monthly_model_df["Repo_Change_Any"].value_counts())

Repo_Change_Any
0    119
1     26
Name: count, dtype: int64


In [235]:
import statsmodels.api as sm

policy_action_vars = [
    "Inflation_Uncertainty_L1",
    "USDINR_3M_Change_L1",
    "Brent_3M_Change_L1"
]

repo_action_logit = sm.Logit(
    monthly_model_df["Repo_Change_Any"],
    sm.add_constant(monthly_model_df[policy_action_vars])
).fit(disp=False)

print(repo_action_logit.summary())

                           Logit Regression Results                           
Dep. Variable:        Repo_Change_Any   No. Observations:                  145
Model:                          Logit   Df Residuals:                      141
Method:                           MLE   Df Model:                            3
Date:                Tue, 23 Jun 2026   Pseudo R-squ.:                 0.04643
Time:                        17:05:15   Log-Likelihood:                -65.034
converged:                       True   LL-Null:                       -68.200
Covariance Type:            nonrobust   LLR p-value:                   0.09647
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                       -1.6874      0.261     -6.463      0.000      -2.199      -1.176
Inflation_Uncertainty_L1    -0.0519      0.131     -0.396      0.692      -0.309       0.

In [237]:
# ── Logit: individual policy-pressure channels ───────────────────────────────

logit_vars_full = [
    "d_Repo_L1",
    "CPI_Inflation_YoY_L1",
    "India_VIX_3M_Avg_L1",
    "USDINR_3M_Change_L1",
    "Brent_3M_Change_L1"
]

logit_vif = pd.DataFrame({
    "Variable": logit_vars_full,
    "VIF": [
        variance_inflation_factor(
            monthly_model_df[logit_vars_full].values,
            i
        )
        for i in range(len(logit_vars_full))
    ]
}).sort_values("VIF", ascending=False)

display(logit_vif.round(3))

,Variable,VIF
2,India_VIX_3M_Avg_L1,11.514
1,CPI_Inflation_YoY_L1,10.890
3,USDINR_3M_Change_L1,1.243
0,d_Repo_L1,1.180
4,Brent_3M_Change_L1,1.088


In [238]:
# ── Full logit: probability of any Repo-rate action ──────────────────────────

repo_action_logit_full = sm.Logit(
    monthly_model_df["Repo_Change_Any"],
    sm.add_constant(monthly_model_df[logit_vars_full])
).fit(
    disp=False,
    cov_type="HC1"
)

print(repo_action_logit_full.summary())

                           Logit Regression Results                           
Dep. Variable:        Repo_Change_Any   No. Observations:                  145
Model:                          Logit   Df Residuals:                      139
Method:                           MLE   Df Model:                            5
Date:                Tue, 23 Jun 2026   Pseudo R-squ.:                  0.1049
Time:                        17:08:22   Log-Likelihood:                -61.046
converged:                       True   LL-Null:                       -68.200
Covariance Type:                  HC1   LLR p-value:                   0.01376
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -1.3719      0.731     -1.877      0.060      -2.804       0.060
d_Repo_L1                4.1305      1.508      2.740      0.006       1.175       7.086
CPI_Inflatio

In [239]:
# ── Odds ratios ──────────────────────────────────────────────────────────────

logit_odds = pd.DataFrame({
    "Variable": repo_action_logit_full.params.index,
    "Coefficient": repo_action_logit_full.params.values,
    "Odds_Ratio": np.exp(repo_action_logit_full.params.values),
    "P_value": repo_action_logit_full.pvalues.values
})

display(logit_odds.round(4))

,Variable,Coefficient,Odds_Ratio,P_value
0,const,-1.3719,0.2536,0.0605
1,d_Repo_L1,4.1305,62.2086,0.0062
2,CPI_Inflation_YoY_L1,-0.2682,0.7648,0.1144
3,India_VIX_3M_Avg_L1,0.0591,1.0609,0.1884
4,USDINR_3M_Change_L1,0.0835,1.0871,0.4680
5,Brent_3M_Change_L1,-0.0280,0.9724,0.0338


In [240]:
# ── Comparable channel-logit models ─────────────────────────────────────────

logit_specs = {
    "CPI + INR + Brent": [
        "d_Repo_L1",
        "CPI_Inflation_YoY_L1",
        "USDINR_3M_Change_L1",
        "Brent_3M_Change_L1"
    ],
    "VIX + INR + Brent": [
        "d_Repo_L1",
        "India_VIX_3M_Avg_L1",
        "USDINR_3M_Change_L1",
        "Brent_3M_Change_L1"
    ],
    "CPI + VIX + INR + Brent": logit_vars_full
}

logit_comparison = []

for name, vars_ in logit_specs.items():
    model = sm.Logit(
        monthly_model_df["Repo_Change_Any"],
        sm.add_constant(monthly_model_df[vars_])
    ).fit(disp=False, cov_type="HC1")

    logit_comparison.append({
        "Model": name,
        "Pseudo_R2": model.prsquared,
        "AIC": model.aic,
        "BIC": model.bic,
        "LLR_p_value": model.llr_pvalue
    })

logit_comparison = pd.DataFrame(logit_comparison)

display(
    logit_comparison
    .sort_values(["BIC", "AIC"])
    .round(4)
)

,Model,Pseudo_R2,AIC,BIC,LLR_p_value
0,CPI + INR + Brent,0.0957,133.3406,148.2242,0.0110
1,VIX + INR + Brent,0.0843,134.8987,149.7823,0.0215
2,CPI + VIX + INR + Brent,0.1049,134.0914,151.9518,0.0138


In [241]:
# ── Test whether Brent adds information beyond CPI ───────────────────────────

logit_specs_brent_test = {
    "CPI + INR": [
        "d_Repo_L1",
        "CPI_Inflation_YoY_L1",
        "USDINR_3M_Change_L1"
    ],
    "CPI + INR + Brent": [
        "d_Repo_L1",
        "CPI_Inflation_YoY_L1",
        "USDINR_3M_Change_L1",
        "Brent_3M_Change_L1"
    ],
    "INR + Brent only": [
        "d_Repo_L1",
        "USDINR_3M_Change_L1",
        "Brent_3M_Change_L1"
    ]
}

brent_test_results = []

for name, vars_ in logit_specs_brent_test.items():
    model = sm.Logit(
        monthly_model_df["Repo_Change_Any"],
        sm.add_constant(monthly_model_df[vars_])
    ).fit(disp=False, cov_type="HC1")

    brent_test_results.append({
        "Model": name,
        "Pseudo_R2": model.prsquared,
        "AIC": model.aic,
        "BIC": model.bic,
        "LLR_p_value": model.llr_pvalue
    })

brent_test_results = pd.DataFrame(brent_test_results)

display(
    brent_test_results
    .sort_values(["BIC", "AIC"])
    .round(4)
)

,Model,Pseudo_R2,AIC,BIC,LLR_p_value
2,INR + Brent only,0.0833,133.0327,144.9396,0.0099
1,CPI + INR + Brent,0.0957,133.3406,148.2242,0.0110
0,CPI + INR,0.0575,136.5597,148.4666,0.0494


In [242]:
preferred_logit_vars = [
    "d_Repo_L1",
    "CPI_Inflation_YoY_L1",
    "USDINR_3M_Change_L1",
    "Brent_3M_Change_L1"
]

preferred_repo_action_logit = sm.Logit(
    monthly_model_df["Repo_Change_Any"],
    sm.add_constant(monthly_model_df[preferred_logit_vars])
).fit(
    disp=False,
    cov_type="HC1"
)

logit_results_table = pd.DataFrame({
    "Variable": preferred_repo_action_logit.params.index,
    "Coefficient": preferred_repo_action_logit.params.values,
    "Odds_Ratio": np.exp(preferred_repo_action_logit.params.values),
    "P_value": preferred_repo_action_logit.pvalues.values
})

display(logit_results_table.round(4))
print(preferred_repo_action_logit.summary())

,Variable,Coefficient,Odds_Ratio,P_value
0,const,-0.8241,0.4386,0.2548
1,d_Repo_L1,3.6870,39.9262,0.0083
2,CPI_Inflation_YoY_L1,-0.1824,0.8333,0.2052
3,USDINR_3M_Change_L1,0.1075,1.1135,0.3398
4,Brent_3M_Change_L1,-0.0303,0.9702,0.0312


                           Logit Regression Results                           
Dep. Variable:        Repo_Change_Any   No. Observations:                  145
Model:                          Logit   Df Residuals:                      140
Method:                           MLE   Df Model:                            4
Date:                Tue, 23 Jun 2026   Pseudo R-squ.:                 0.09575
Time:                        17:12:02   Log-Likelihood:                -61.670
converged:                       True   LL-Null:                       -68.200
Covariance Type:                  HC1   LLR p-value:                   0.01099
                           coef    std err          z      P>|z|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -0.8241      0.724     -1.139      0.255      -2.243       0.594
d_Repo_L1                3.6870      1.397      2.639      0.008       0.949       6.425
CPI_Inflatio

In [243]:
# ── Create CPI lags up to 3 months ──────────────────────────────────────────

monthly_model_df = monthly_model_df.copy()

monthly_model_df["CPI_L1"] = monthly_model_df["CPI_Inflation_YoY"].shift(1)
monthly_model_df["CPI_L2"] = monthly_model_df["CPI_Inflation_YoY"].shift(2)
monthly_model_df["CPI_L3"] = monthly_model_df["CPI_Inflation_YoY"].shift(3)

# Rebuild clean dataset after adding new lags
logit_lag_df = monthly_model_df.dropna(
    subset=[
        "Repo_Change_Any",
        "d_Repo_L1",
        "CPI_L1",
        "CPI_L2",
        "CPI_L3",
        "USDINR_3M_Change_L1",
        "Brent_3M_Change_L1"
    ]
).copy()

print(
    "Lagged-logit sample:",
    logit_lag_df["Date"].min().date(),
    "to",
    logit_lag_df["Date"].max().date()
)
print("Observations:", len(logit_lag_df))

Lagged-logit sample: 2014-08-31 to 2026-05-31
Observations: 142


In [244]:
# ── Distributed-lag CPI logit model ─────────────────────────────────────────

cpi_lag_vars = [
    "d_Repo_L1",
    "CPI_L1",
    "CPI_L2",
    "CPI_L3",
    "USDINR_3M_Change_L1",
    "Brent_3M_Change_L1"
]

repo_action_logit_cpi_lags = sm.Logit(
    logit_lag_df["Repo_Change_Any"],
    sm.add_constant(logit_lag_df[cpi_lag_vars])
).fit(
    disp=False,
    cov_type="HC1"
)

print(repo_action_logit_cpi_lags.summary())

                           Logit Regression Results                           
Dep. Variable:        Repo_Change_Any   No. Observations:                  142
Model:                          Logit   Df Residuals:                      135
Method:                           MLE   Df Model:                            6
Date:                Tue, 23 Jun 2026   Pseudo R-squ.:                 0.09747
Time:                        17:14:17   Log-Likelihood:                -61.012
converged:                       True   LL-Null:                       -67.600
Covariance Type:                  HC1   LLR p-value:                   0.04030
                          coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------
const                  -0.9272      0.795     -1.166      0.244      -2.486       0.632
d_Repo_L1               3.5781      1.466      2.441      0.015       0.705       6.452
CPI_L1          

In [245]:
id="qtr_final_01"
# ── Quarterly continuous RBI Repo reaction dataset: 2014 onward ─────────────

quarterly_policy_df = monthly_policy_df.copy()

quarterly_policy_df["Date"] = pd.to_datetime(quarterly_policy_df["Date"])
quarterly_policy_df = quarterly_policy_df[
    quarterly_policy_df["Date"] >= pd.Timestamp("2014-01-01")
].copy()

quarterly_policy_df["Quarter"] = quarterly_policy_df["Date"].dt.to_period("Q")

# CPI and VIX: quarterly averages
# Repo: quarter-end policy rate
# INR and Brent: quarter-end levels, then QoQ percentage changes
quarterly_policy_df = (
    quarterly_policy_df
    .groupby("Quarter", as_index=False)
    .agg(
        CPI_Inflation_YoY=("CPI_Inflation_YoY", "mean"),
        India_VIX_Avg=("India_VIX_Avg", "mean"),
        USDINR=("USDINR", "last"),
        Brent_USD=("Brent_USD", "last"),
        Repo_rate=("Repo_rate", "last")
    )
)

quarterly_policy_df["Date"] = (
    quarterly_policy_df["Quarter"]
    .dt.to_timestamp(how="end")
)

quarterly_policy_df["USDINR_QoQ_Change"] = (
    quarterly_policy_df["USDINR"].pct_change() * 100
)

quarterly_policy_df["Brent_QoQ_Change"] = (
    quarterly_policy_df["Brent_USD"].pct_change() * 100
)

quarterly_policy_df["d_Repo"] = (
    quarterly_policy_df["Repo_rate"].diff()
)

# One-quarter lag: information available before RBI acts in the current quarter
quarterly_policy_df["d_Repo_L1"] = quarterly_policy_df["d_Repo"].shift(1)

for var in [
    "CPI_Inflation_YoY",
    "India_VIX_Avg",
    "USDINR_QoQ_Change",
    "Brent_QoQ_Change"
]:
    quarterly_policy_df[f"{var}_L1"] = quarterly_policy_df[var].shift(1)

quarterly_model_df = (
    quarterly_policy_df
    .dropna()
    .reset_index(drop=True)
)

print(
    "Quarterly usable sample:",
    quarterly_model_df["Quarter"].iloc[0],
    "to",
    quarterly_model_df["Quarter"].iloc[-1]
)

print("Quarterly observations:", len(quarterly_model_df))

quarterly_model_df.tail()

Quarterly usable sample: 2014Q3 to 2026Q2
Quarterly observations: 48


,Quarter,CPI_Inflation_YoY,India_VIX_Avg,USDINR,Brent_USD,Repo_rate,Date,USDINR_QoQ_Change,Brent_QoQ_Change,d_Repo,d_Repo_L1,CPI_Inflation_YoY_L1,India_VIX_Avg_L1,USDINR_QoQ_Change_L1,Brent_QoQ_Change_L1
43,2025Q2,2.891827,16.387586,85.928105,69.350000,5.50,2025-06-30 23:59:59.999999999,-0.822961,-2.962400,-0.75,-0.25,3.707227,14.653748,1.952356,-2.271929
44,2025Q3,1.678648,11.429834,88.258800,67.548571,5.50,2025-09-30 23:59:59.999999999,2.712377,-2.597591,0.00,-0.75,2.891827,16.387586,-0.822961,-2.962400
45,2025Q4,0.565836,11.188251,90.008018,61.628636,5.25,2025-12-31 23:59:59.999999999,1.981919,-8.763967,-0.25,0.00,1.678648,11.429834,2.712377,-2.597591
46,2026Q1,3.114899,15.655816,92.802195,99.599546,5.25,2026-03-31 23:59:59.999999999,3.104365,61.612446,0.00,-0.25,0.565836,11.188251,1.981919,-8.763967
47,2026Q2,3.710084,18.866316,95.538276,104.092000,5.25,2026-06-30 23:59:59.999999999,2.948293,4.510516,0.00,0.00,3.114899,15.655816,3.104365,61.612446


In [246]:
id="qtr_final_02"
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import pandas as pd

quarterly_full_vars = [
    "d_Repo_L1",
    "CPI_Inflation_YoY_L1",
    "India_VIX_Avg_L1",
    "USDINR_QoQ_Change_L1",
    "Brent_QoQ_Change_L1"
]

quarterly_vif = pd.DataFrame({
    "Variable": quarterly_full_vars,
    "VIF": [
        variance_inflation_factor(
            quarterly_model_df[quarterly_full_vars].values,
            i
        )
        for i in range(len(quarterly_full_vars))
    ]
}).sort_values("VIF", ascending=False)

display(quarterly_vif.round(3))

,Variable,VIF
2,India_VIX_Avg_L1,16.597
1,CPI_Inflation_YoY_L1,15.539
0,d_Repo_L1,1.424
3,USDINR_QoQ_Change_L1,1.225
4,Brent_QoQ_Change_L1,1.032


In [248]:
id="qtr_final_03"
# ── Quarterly continuous Repo reaction models ────────────────────────────────

quarterly_specs = {
    "A_CPI_INR_VIX": [
        "d_Repo_L1",
        "CPI_Inflation_YoY_L1",
        "USDINR_QoQ_Change_L1",
        "India_VIX_Avg_L1"
    ],
    
    "B_Brent_INR_VIX": [
        "d_Repo_L1",
        "Brent_QoQ_Change_L1",
        "USDINR_QoQ_Change_L1",
        "India_VIX_Avg_L1"
    ],
    
    "C_CPI_Brent_INR_VIX": [
        "d_Repo_L1",
        "CPI_Inflation_YoY_L1",
        "USDINR_QoQ_Change_L1",
        "India_VIX_Avg_L1"
    ]
}

quarterly_models = {}
quarterly_model_comparison = []

for model_name, vars_ in quarterly_specs.items():
    
    model = sm.OLS(
        quarterly_model_df["d_Repo"],
        sm.add_constant(quarterly_model_df[vars_])
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 2}
    )
    
    quarterly_models[model_name] = model
    
    quarterly_model_comparison.append({
        "Model": model_name,
        "Observations": int(model.nobs),
        "Adjusted_R2": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic,
        "DW": sm.stats.stattools.durbin_watson(model.resid)
    })

quarterly_model_comparison = pd.DataFrame(
    quarterly_model_comparison
).sort_values(["BIC", "AIC"])

display(quarterly_model_comparison.round(4))

,Model,Observations,Adjusted_R2,AIC,BIC,DW
1,B_Brent_INR_VIX,48,0.3771,6.868,16.224,2.2562
0,A_CPI_INR_VIX,48,0.3078,11.934,21.290,2.0111
2,C_CPI_Brent_INR_VIX,48,0.3078,11.934,21.290,2.0111


In [250]:
# ── US Effective Federal Funds Rate (DFF) ────────────────────────────────────

FEDFUNDS_URL = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFF"

fedfunds = pd.read_csv(FEDFUNDS_URL)

print("Downloaded columns:", fedfunds.columns.tolist())

# FRED usually returns: observation_date, DFF
date_col = next(
    c for c in ["observation_date", "DATE", "Date", "date"]
    if c in fedfunds.columns
)

rate_col = next(
    c for c in ["DFF", "Fed_Funds_Rate"]
    if c in fedfunds.columns
)

fedfunds = (
    fedfunds
    .rename(columns={
        date_col: "Date",
        rate_col: "Fed_Funds_Rate"
    })
    .copy()
)

fedfunds["Date"] = pd.to_datetime(fedfunds["Date"], errors="coerce")
fedfunds["Fed_Funds_Rate"] = pd.to_numeric(
    fedfunds["Fed_Funds_Rate"],
    errors="coerce"
)

fedfunds = (
    fedfunds
    .dropna(subset=["Date", "Fed_Funds_Rate"])
    .sort_values("Date")
    .reset_index(drop=True)
)

fedfunds["Quarter"] = fedfunds["Date"].dt.to_period("Q")

fedfunds_quarterly = (
    fedfunds
    .groupby("Quarter", as_index=False)
    .agg(
        Fed_Funds_Rate=("Fed_Funds_Rate", "last")
    )
)

fedfunds_quarterly["Fed_Funds_Change"] = (
    fedfunds_quarterly["Fed_Funds_Rate"]
    .diff()
)

fedfunds_quarterly.tail()

Downloaded columns: ['observation_date', 'DFF']


,Quarter,Fed_Funds_Rate,Fed_Funds_Change
283,2025Q2,4.33,0.00
284,2025Q3,4.09,-0.24
285,2025Q4,3.64,-0.45
286,2026Q1,3.64,0.00
287,2026Q2,3.63,-0.01


In [251]:
# ── Add Fed Funds Rate to quarterly RBI policy dataset ───────────────────────

quarterly_policy_df = (
    quarterly_policy_df
    .merge(
        fedfunds_quarterly[
            ["Quarter", "Fed_Funds_Rate", "Fed_Funds_Change"]
        ],
        on="Quarter",
        how="left"
    )
    .sort_values("Quarter")
    .reset_index(drop=True)
)

quarterly_policy_df["Fed_Funds_Change_L1"] = (
    quarterly_policy_df["Fed_Funds_Change"].shift(1)
)

quarterly_model_df = (
    quarterly_policy_df
    .dropna()
    .reset_index(drop=True)
)

print(
    "Quarterly sample with Fed Funds:",
    quarterly_model_df["Quarter"].iloc[0],
    "to",
    quarterly_model_df["Quarter"].iloc[-1]
)

quarterly_model_df.tail()

Quarterly sample with Fed Funds: 2014Q3 to 2026Q2


,Quarter,CPI_Inflation_YoY,India_VIX_Avg,USDINR,Brent_USD,Repo_rate,Date,USDINR_QoQ_Change,Brent_QoQ_Change,d_Repo,d_Repo_L1,CPI_Inflation_YoY_L1,India_VIX_Avg_L1,USDINR_QoQ_Change_L1,Brent_QoQ_Change_L1,Fed_Funds_Rate,Fed_Funds_Change,Fed_Funds_Change_L1
43,2025Q2,2.891827,16.387586,85.928105,69.350000,5.50,2025-06-30 23:59:59.999999999,-0.822961,-2.962400,-0.75,-0.25,3.707227,14.653748,1.952356,-2.271929,4.33,0.00,0.00
44,2025Q3,1.678648,11.429834,88.258800,67.548571,5.50,2025-09-30 23:59:59.999999999,2.712377,-2.597591,0.00,-0.75,2.891827,16.387586,-0.822961,-2.962400,4.09,-0.24,0.00
45,2025Q4,0.565836,11.188251,90.008018,61.628636,5.25,2025-12-31 23:59:59.999999999,1.981919,-8.763967,-0.25,0.00,1.678648,11.429834,2.712377,-2.597591,3.64,-0.45,-0.24
46,2026Q1,3.114899,15.655816,92.802195,99.599546,5.25,2026-03-31 23:59:59.999999999,3.104365,61.612446,0.00,-0.25,0.565836,11.188251,1.981919,-8.763967,3.64,0.00,-0.45
47,2026Q2,3.710084,18.866316,95.538276,104.092000,5.25,2026-06-30 23:59:59.999999999,2.948293,4.510516,0.00,0.00,3.114899,15.655816,3.104365,61.612446,3.63,-0.01,0.00


In [252]:
# ── Fed Funds robustness models ─────────────────────────────────────────────

quarterly_specs_fed = {
    "B_Brent_INR_VIX": [
        "d_Repo_L1",
        "Brent_QoQ_Change_L1",
        "USDINR_QoQ_Change_L1",
        "India_VIX_Avg_L1"
    ],

    "D_Brent_INR_VIX_FedFunds": [
        "d_Repo_L1",
        "Brent_QoQ_Change_L1",
        "USDINR_QoQ_Change_L1",
        "India_VIX_Avg_L1",
        "Fed_Funds_Change_L1"
    ],

    "E_CPI_Brent_INR_VIX_FedFunds": [
        "d_Repo_L1",
        "CPI_Inflation_YoY_L1",
        "Brent_QoQ_Change_L1",
        "USDINR_QoQ_Change_L1",
        "India_VIX_Avg_L1",
        "Fed_Funds_Change_L1"
    ]
}

quarterly_models_fed = {}
quarterly_fed_comparison = []

for model_name, vars_ in quarterly_specs_fed.items():

    model = sm.OLS(
        quarterly_model_df["d_Repo"],
        sm.add_constant(quarterly_model_df[vars_])
    ).fit(
        cov_type="HAC",
        cov_kwds={"maxlags": 2}
    )

    quarterly_models_fed[model_name] = model

    quarterly_fed_comparison.append({
        "Model": model_name,
        "Observations": int(model.nobs),
        "Adjusted_R2": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic,
        "DW": sm.stats.stattools.durbin_watson(model.resid)
    })

quarterly_fed_comparison = pd.DataFrame(
    quarterly_fed_comparison
).sort_values(["BIC", "AIC"])

display(quarterly_fed_comparison.round(4))

,Model,Observations,Adjusted_R2,AIC,BIC,DW
0,B_Brent_INR_VIX,48,0.3771,6.8680,16.2240,2.2562
1,D_Brent_INR_VIX_FedFunds,48,0.4028,5.7163,16.9435,2.2381
2,E_CPI_Brent_INR_VIX_FedFunds,48,0.3920,7.4174,20.5158,2.2230


In [253]:
fed_full_vars = quarterly_specs_fed["D_Brent_INR_VIX_FedFunds"]

fed_vif = pd.DataFrame({
    "Variable": fed_full_vars,
    "VIF": [
        variance_inflation_factor(
            quarterly_model_df[fed_full_vars].values,
            i
        )
        for i in range(len(fed_full_vars))
    ]
}).sort_values("VIF", ascending=False)

display(fed_vif.round(3))

,Variable,VIF
0,d_Repo_L1,2.501
4,Fed_Funds_Change_L1,2.412
3,India_VIX_Avg_L1,1.580
2,USDINR_QoQ_Change_L1,1.252
1,Brent_QoQ_Change_L1,1.042


## ghar pe aa gya phir se

In [254]:
# ── Quarterly Inflation–Uncertainty Pressure Index ───────────────────────────

from scipy.stats import zscore

quarterly_model_df = quarterly_model_df.copy()

quarterly_model_df["CPI_z_L1"] = zscore(
    quarterly_model_df["CPI_Inflation_YoY_L1"],
    nan_policy="omit"
)

quarterly_model_df["VIX_z_L1"] = zscore(
    quarterly_model_df["India_VIX_Avg_L1"],
    nan_policy="omit"
)

quarterly_model_df["Inflation_Uncertainty_L1"] = (
    quarterly_model_df["CPI_z_L1"]
    + quarterly_model_df["VIX_z_L1"]
)

In [255]:
# ── Quarterly Repo reaction model with Inflation–Uncertainty index ───────────

inflation_uncertainty_vars = [
    "d_Repo_L1",
    "Inflation_Uncertainty_L1",
    "Brent_QoQ_Change_L1",
    "USDINR_QoQ_Change_L1",
    "Fed_Funds_Change_L1"
]

inflation_uncertainty_model = sm.OLS(
    quarterly_model_df["d_Repo"],
    sm.add_constant(quarterly_model_df[inflation_uncertainty_vars])
).fit(
    cov_type="HAC",
    cov_kwds={"maxlags": 2}
)

print(inflation_uncertainty_model.summary())

                            OLS Regression Results                            
Dep. Variable:                 d_Repo   R-squared:                       0.468
Model:                            OLS   Adj. R-squared:                  0.405
Method:                 Least Squares   F-statistic:                     11.50
Date:                Tue, 23 Jun 2026   Prob (F-statistic):           4.95e-07
Time:                        17:28:43   Log-Likelihood:                 3.2252
No. Observations:                  48   AIC:                             5.550
Df Residuals:                      42   BIC:                             16.78
Df Model:                           5                                         
Covariance Type:                  HAC                                         
                               coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------
const                   

In [256]:
# ── Compare final quarterly alternatives ─────────────────────────────────────

final_model_comparison = pd.DataFrame({
    "Model": [
        "External pressure: Brent + INR + VIX + Fed Funds",
        "CPI included separately: CPI + Brent + INR + VIX + Fed Funds",
        "Inflation-Uncertainty index + Brent + INR + Fed Funds"
    ],
    "Adjusted_R2": [
        quarterly_models_fed["D_Brent_INR_VIX_FedFunds"].rsquared_adj,
        quarterly_models_fed["E_CPI_Brent_INR_VIX_FedFunds"].rsquared_adj,
        inflation_uncertainty_model.rsquared_adj
    ],
    "AIC": [
        quarterly_models_fed["D_Brent_INR_VIX_FedFunds"].aic,
        quarterly_models_fed["E_CPI_Brent_INR_VIX_FedFunds"].aic,
        inflation_uncertainty_model.aic
    ],
    "BIC": [
        quarterly_models_fed["D_Brent_INR_VIX_FedFunds"].bic,
        quarterly_models_fed["E_CPI_Brent_INR_VIX_FedFunds"].bic,
        inflation_uncertainty_model.bic
    ],
    "DW": [
        sm.stats.stattools.durbin_watson(
            quarterly_models_fed["D_Brent_INR_VIX_FedFunds"].resid
        ),
        sm.stats.stattools.durbin_watson(
            quarterly_models_fed["E_CPI_Brent_INR_VIX_FedFunds"].resid
        ),
        sm.stats.stattools.durbin_watson(
            inflation_uncertainty_model.resid
        )
    ]
})

display(
    final_model_comparison
    .sort_values(["BIC", "AIC"])
    .round(4)
)

,Model,Adjusted_R2,AIC,BIC,DW
2,Inflation-Uncertainty index + Brent + INR + Fe...,0.4049,5.5496,16.7768,2.2111
0,External pressure: Brent + INR + VIX + Fed Funds,0.4028,5.7163,16.9435,2.2381
1,CPI included separately: CPI + Brent + INR + V...,0.3920,7.4174,20.5158,2.2230


## RBI’s quarterly Repo-rate adjustments appear to reflect a combination of domestic inflation and uncertainty pressure, global monetary-policy spillovers from the United States, and imported-inflation risk from oil markets. The Fed Funds Rate and the inflation–uncertainty index are the strongest statistically supported channels. Brent adds weaker but correctly signed pressure, while the INR channel becomes less precise once these broader external conditions are controlled for.